# Credit Card Fraud Detection — HOBA Framework
## Replication and Extension of Zhang et al. (2021)
### Course Project | ECO723 | Prof. Sanjiv Kumar | IIT Kanpur

## Section 0: Global Configuration
Imports, random seeds, Drive paths, and evaluation helpers.
Run this cell at the start of every session before anything else.

In [1]:
# ============================================================
# SECTION 0 — GLOBAL CONFIGURATION
# Run this cell first in every session
# ============================================================

# ── Core imports ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import random
import json
import os
import warnings
warnings.filterwarnings('ignore')

# ── Deep learning ────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (Input, Dense, Dropout, LSTM,
                                     Bidirectional, Conv1D, MaxPooling1D,
                                     Flatten, Concatenate)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

# ── ML models ────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, roc_curve, confusion_matrix)

# ── Visualization ────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from math import radians, sin, cos, sqrt, atan2

# ── Reproducibility ──────────────────────────────────────────
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

# ── Drive paths ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f"Drive mounted. Project folder: {DRIVE_PATH}")

# ── Haversine distance function ───────────────────────────────
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# ── Evaluation helpers ────────────────────────────────────────
def fpr_constrained_recall(y_true, y_prob, fpr_threshold):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    valid_idx = np.where(fpr <= fpr_threshold)[0]
    if len(valid_idx) == 0:
        return 0.0
    return tpr[valid_idx[np.argmax(tpr[valid_idx])]]

def evaluate_model(name, y_true, y_pred, y_prob):
    auc         = roc_auc_score(y_true, y_prob)
    f1          = f1_score(y_true, y_pred)
    precision   = precision_score(y_true, y_pred)
    recall      = recall_score(y_true, y_pred)
    r1fpr       = fpr_constrained_recall(y_true, y_prob, 0.01)
    r3fpr       = fpr_constrained_recall(y_true, y_prob, 0.03)
    print(f"\n=== {name} ===")
    print(f"AUC:              {auc:.4f}")
    print(f"F1:               {f1:.4f}")
    print(f"Precision:        {precision:.4f}")
    print(f"Recall:           {recall:.4f}")
    print(f"Recall @ 1% FPR:  {r1fpr:.4f}")
    print(f"Recall @ 3% FPR:  {r3fpr:.4f}")
    result = {'name': name, 'auc': auc, 'f1': f1, 'precision': precision,
              'recall': recall, 'recall_1fpr': r1fpr, 'recall_3fpr': r3fpr}
    # Auto-save to Drive
    results_path = f'{DRIVE_PATH}/results.json'
    all_results = json.load(open(results_path)) if os.path.exists(results_path) else []
    all_results = [r for r in all_results if r['name'] != name]
    all_results.append(result)
    json.dump(all_results, open(results_path, 'w'), indent=2)
    print("Result saved to Drive.")
    return result

def print_results_table():
    results_path = f'{DRIVE_PATH}/results.json'
    if not os.path.exists(results_path):
        print("No results saved yet.")
        return
    all_results = json.load(open(results_path))
    print(f"\n{'Model':<35} {'AUC':>6} {'F1':>6} {'Recall':>8} {'R@1%FPR':>9} {'R@3%FPR':>9}")
    print("-" * 75)
    for r in all_results:
        print(f"{r['name']:<35} {r['auc']:>6.4f} {r['f1']:>6.4f} "
              f"{r['recall']:>8.4f} {r['recall_1fpr']:>9.4f} {r['recall_3fpr']:>9.4f}")

# ── Class weight helper ───────────────────────────────────────
def get_class_weights(y):
    weights = compute_class_weight('balanced', classes=np.array([0,1]), y=y)
    return {0: weights[0], 1: weights[1]}

print("\n✅ Section 0 complete — all libraries loaded, seeds set.")
print(f"TensorFlow version: {tf.__version__}")
print(f"Random seed: {RANDOM_SEED}")

Mounted at /content/drive
Drive mounted. Project folder: /content/drive/MyDrive/HOBA_Fraud_Detection_V2

✅ Section 0 complete — all libraries loaded, seeds set.
TensorFlow version: 2.20.0
Random seed: 42


## Section 1: Data Loading
Downloading IBM TabFormer credit card transactions dataset from Kaggle.
Dataset: kaggle.com/datasets/ealtman2019/credit-card-transactions
Features: User, Card, Year, Month, Day, Time, Amount, Use Chip,
Merchant Name, Merchant City, Merchant State, Zip, MCC, Errors?, Is Fraud?
24M transactions from 2000 synthetic US consumers — richest public dataset
for HOBA implementation due to real MCC codes and entry mode (Use Chip) field.

## Section 1.1: Data Inspection
Loading and inspecting all three IBM dataset files to understand
structure, columns, and join keys before any processing.

In [ ]:
import os

# Check all files in content directory
print("All files in /content:")
for root, dirs, files in os.walk('/content'):
    # Skip drive and system folders
    if any(skip in root for skip in ['/content/drive', '/content/sample_data', '__pycache__']):
        continue
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath) / (1024**3)
        print(f"  {filepath} ({size:.2f} GB)")

All files in /content:
  /content/.config/.last_update_check.json (0.00 GB)
  /content/.config/.last_opt_in_prompt.yaml (0.00 GB)
  /content/.config/.last_survey_prompt.yaml (0.00 GB)
  /content/.config/config_sentinel (0.00 GB)
  /content/.config/active_config (0.00 GB)
  /content/.config/gce (0.00 GB)
  /content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db (0.00 GB)
  /content/.config/default_configs.db (0.00 GB)
  /content/.config/logs/2026.06.04/13.32.38.346437.log (0.00 GB)
  /content/.config/logs/2026.06.04/13.32.39.344962.log (0.00 GB)
  /content/.config/logs/2026.06.04/13.32.02.654775.log (0.00 GB)
  /content/.config/logs/2026.06.04/13.31.42.499627.log (0.00 GB)
  /content/.config/logs/2026.06.04/13.32.21.210570.log (0.00 GB)
  /content/.config/logs/2026.06.04/13.32.18.735754.log (0.00 GB)
  /content/.config/configurations/config_default (0.00 GB)


In [ ]:
import pandas as pd
import numpy as np

# ── Inspect transactions file ─────────────────────────────────
print("=== TRANSACTIONS FILE ===")
txn = pd.read_csv('/content/drive/MyDrive/HOBA_Fraud_Detection_V2/raw/credit_card_transactions-ibm_v2.csv', nrows=5)
print("Columns:", list(txn.columns))
print("Shape (sample):", txn.shape)
print(txn.head(2).to_string())

# ── Inspect users file ────────────────────────────────────────
print("\n=== USERS FILE ===")
users = pd.read_csv('/content/drive/MyDrive/HOBA_Fraud_Detection_V2/raw/sd254_users.csv', nrows=5)
print("Columns:", list(users.columns))
print(users.head(2).to_string())

# ── Inspect cards file ────────────────────────────────────────
print("\n=== CARDS FILE ===")
cards = pd.read_csv('/content/drive/MyDrive/HOBA_Fraud_Detection_V2/raw/sd254_cards.csv', nrows=5)
print("Columns:", list(cards.columns))
print(cards.head(2).to_string())

=== TRANSACTIONS FILE ===
Columns: ['User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Amount', 'Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?', 'Is Fraud?']
Shape (sample): (5, 15)
   User  Card  Year  Month  Day   Time   Amount           Use Chip        Merchant Name  Merchant City Merchant State      Zip   MCC  Errors? Is Fraud?
0     0     0  2002      9    1  06:21  $134.09  Swipe Transaction  3527213246127876953       La Verne             CA  91750.0  5300      NaN        No
1     0     0  2002      9    1  06:42   $38.48  Swipe Transaction  -727612092139916043  Monterey Park             CA  91754.0  5411      NaN        No

=== USERS FILE ===
Columns: ['Person', 'Current Age', 'Retirement Age', 'Birth Year', 'Birth Month', 'Gender', 'Address', 'Apartment', 'City', 'State', 'Zipcode', 'Latitude', 'Longitude', 'Per Capita Income - Zipcode', 'Yearly Income - Person', 'Total Debt', 'FICO Score', 'Num Credit Cards']
           Person  Current Age

## Section 1: Data Loading and Saving to Drive
Downloads complete. Saving all three IBM dataset files to Drive for
session persistence, then loading and inspecting structure.
Dataset files:
- credit_card_transactions-ibm_v2.csv — 24M transactions (main file)
- sd254_users.csv — cardholder demographics with home lat/long
- sd254_cards.csv — card details including credit limit

In [ ]:
import pandas as pd
import numpy as np
import shutil, os

# ── Save raw files to Drive ───────────────────────────────────
RAW_PATH = f'{DRIVE_PATH}/raw'
os.makedirs(RAW_PATH, exist_ok=True)

for fname in ['credit_card_transactions-ibm_v2.csv', 'sd254_users.csv', 'sd254_cards.csv']:
    src = f'/content/{fname}'
    dst = f'{RAW_PATH}/{fname}'
    if not os.path.exists(dst):
        print(f"Saving {fname} to Drive...")
        shutil.copy(src, dst)
        print(f"  Done.")
    else:
        print(f"  {fname} already on Drive, skipping.")

# ── Load all three files ──────────────────────────────────────
print("\nLoading files...")
txn   = pd.read_csv(f'{RAW_PATH}/credit_card_transactions-ibm_v2.csv')
users = pd.read_csv(f'{RAW_PATH}/sd254_users.csv')
cards = pd.read_csv(f'{RAW_PATH}/sd254_cards.csv')

print(f"Transactions: {txn.shape}")
print(f"Users:        {users.shape}")
print(f"Cards:        {cards.shape}")
print(f"\nFraud rate: {(txn['Is Fraud?']=='Yes').mean()*100:.4f}%")
print(f"Fraud count: {(txn['Is Fraud?']=='Yes').sum()}")
print(f"MCC unique values: {txn['MCC'].nunique()}")
print(f"Use Chip values: {txn['Use Chip'].unique()}")
print(f"Date range: {txn['Year'].min()} to {txn['Year'].max()}")

  credit_card_transactions-ibm_v2.csv already on Drive, skipping.
  sd254_users.csv already on Drive, skipping.
  sd254_cards.csv already on Drive, skipping.

Loading files...
Transactions: (24386900, 15)
Users:        (2000, 18)
Cards:        (6146, 13)

Fraud rate: 0.1220%
Fraud count: 29757
MCC unique values: 109
Use Chip values: ['Swipe Transaction' 'Online Transaction' 'Chip Transaction']
Date range: 1991 to 2020


## Section 2: Data Subsetting
Retaining transactions from 2017-2020 only. 2017 serves as the behavioral
history window for HOBA feature engineering but is excluded from train/test.
Train: 2018-2019 | Test: 2020 | History: 2017 (dropped after feature engineering)
All fraud cases from 2017-2020 are retained. Result saved to Drive immediately.

In [ ]:
import os

SUBSET_PATH = f'{DRIVE_PATH}/subset_2017_2020.csv'

# ── Load from Drive if already exists, else compute and save ──
if os.path.exists(SUBSET_PATH):
    print("Loading subset from Drive...")
    df = pd.read_csv(SUBSET_PATH)
    print("Loaded from Drive.")
else:
    print("Subsetting transactions to 2017-2020...")
    df = txn[txn['Year'] >= 2017].copy()
    print("Saving subset to Drive...")
    df.to_csv(SUBSET_PATH, index=False)
    print("Saved.")

print(f"\nSubset shape: {df.shape}")
print(f"Year distribution:\n{df['Year'].value_counts().sort_index()}")
print(f"Fraud count: {(df['Is Fraud?']=='Yes').sum()}")
print(f"Fraud rate: {(df['Is Fraud?']=='Yes').mean()*100:.4f}%")
print(f"\nFraud by year:\n{df[df['Is Fraud?']=='Yes']['Year'].value_counts().sort_index()}")

total_fraud_original = (txn['Is Fraud?']=='Yes').sum()
total_fraud_subset   = (df['Is Fraud?']=='Yes').sum()
print(f"\nFraud retained: {total_fraud_subset}/{total_fraud_original} ({total_fraud_subset/total_fraud_original*100:.1f}%)")

Loading subset from Drive...
Loaded from Drive.

Subset shape: (5505413, 15)
Year distribution:
Year
2017    1723360
2018    1721615
2019    1723938
2020     336500
Name: count, dtype: int64
Fraud count: 4833
Fraud rate: 0.0878%

Fraud by year:
Year
2017     255
2018    2491
2019    2087
Name: count, dtype: int64

Fraud retained: 4833/29757 (16.2%)


In [ ]:
# Check fraud distribution across all years
fraud_by_year_full = txn[txn['Is Fraud?']=='Yes']['Year'].value_counts().sort_index()
print("Fraud distribution across all years:")
print(fraud_by_year_full)
print(f"\nTotal fraud: {fraud_by_year_full.sum()}")

Fraud distribution across all years:
Year
1996      10
1997      32
1998      32
1999      24
2000     171
2001     354
2002     139
2003     311
2004     620
2005     229
2006    1118
2007    1881
2008    3710
2009    1140
2010    3835
2011      55
2012    1333
2013    2018
2014    1052
2015    3281
2016    3579
2017     255
2018    2491
2019    2087
Name: count, dtype: int64

Total fraud: 29757


In [ ]:
# Check transaction volumes for 2015-2019
mask = txn['Year'].between(2015, 2019)
subset_check = txn[mask]
print("Transaction volumes 2015-2019:")
print(subset_check['Year'].value_counts().sort_index())
print(f"\nTotal rows: {len(subset_check):,}")
print(f"Total fraud: {(subset_check['Is Fraud?']=='Yes').sum():,}")
print(f"Fraud rate: {(subset_check['Is Fraud?']=='Yes').mean()*100:.4f}%")
print(f"Estimated memory: {subset_check.memory_usage(deep=True).sum()/1e9:.2f} GB")

Transaction volumes 2015-2019:
Year
2015    1701371
2016    1708924
2017    1723360
2018    1721615
2019    1723938
Name: count, dtype: int64

Total rows: 8,579,208
Total fraud: 11,693
Fraud rate: 0.1363%
Estimated memory: 3.74 GB


## Section 2: Data Subsetting and Sampling
Using 2015-2019 transactions. 2015 = behavioral history window (dropped after
feature engineering). Train = 2016-2018, Test = 2019.
Sampling strategy: keep 100% of fraud transactions + 20% of legitimate
transactions sampled chronologically within each card. This preserves
temporal behavioral sequences while keeping memory manageable.
Real-world fraud rate (~0.14%) is maintained since we sample both splits
at the same ratio. Result saved to Drive immediately.

In [ ]:
SUBSET_PATH = f'{DRIVE_PATH}/subset_2015_2019.csv'

if os.path.exists(SUBSET_PATH):
    print("Loading subset from Drive...")
    df = pd.read_csv(SUBSET_PATH)
    print("Loaded.")
else:
    print("Filtering 2015-2019...")
    raw = txn[txn['Year'].between(2015, 2019)].copy()

    # Sort chronologically within each card
    raw = raw.sort_values(['User', 'Card', 'Year', 'Month', 'Day', 'Time']).reset_index(drop=True)

    # Separate fraud and legitimate
    fraud_df  = raw[raw['Is Fraud?'] == 'Yes']
    legit_df  = raw[raw['Is Fraud?'] == 'No']

    # Sample 20% of legitimate chronologically within each card
    print("Sampling legitimate transactions...")
    legit_sampled = (
        legit_df
        .groupby(['User', 'Card'], group_keys=False)
        .apply(lambda g: g.iloc[::5])  # every 5th = 20%
    )

    # Combine and sort
    df = pd.concat([fraud_df, legit_sampled]).sort_values(
        ['User', 'Card', 'Year', 'Month', 'Day', 'Time']
    ).reset_index(drop=True)

    print("Saving to Drive...")
    df.to_csv(SUBSET_PATH, index=False)
    print("Saved.")

print(f"\nFinal shape: {df.shape}")
print(f"Year distribution:\n{df['Year'].value_counts().sort_index()}")
print(f"Fraud count: {(df['Is Fraud?']=='Yes').sum():,}")
print(f"Fraud rate: {(df['Is Fraud?']=='Yes').mean()*100:.4f}%")
print(f"Memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB")

Loading subset from Drive...
Loaded.

Final shape: (1727004, 15)
Year distribution:
Year
2015    344510
2016    344686
2017    344929
2018    346342
2019    346537
Name: count, dtype: int64
Fraud count: 11,693
Fraud rate: 0.6771%
Memory: 0.74 GB


## Section 3: Data Joining
Joining transactions with users (home lat/long) and cards (credit limit, has chip)
to enrich the dataset with features needed for HOBA implementation.
Join keys: df['User'] → users index | df['User']+df['Card'] → cards['User']+cards['CARD INDEX']
Result saved to Drive immediately.

In [ ]:
JOINED_PATH = f'{DRIVE_PATH}/joined_2015_2019.csv'

if os.path.exists(JOINED_PATH):
    print("Loading joined data from Drive...")
    df_joined = pd.read_csv(JOINED_PATH)
    print("Loaded.")
else:
    print("Preparing join keys...")

    # ── Clean users — keep only needed columns ────────────────
    users_clean = users[['Latitude', 'Longitude', 'Yearly Income - Person', 'FICO Score']].copy()
    users_clean.index.name = 'User'
    users_clean = users_clean.reset_index()

    # ── Clean cards — keep only needed columns ────────────────
    cards_clean = cards[['User', 'CARD INDEX', 'Card Brand', 'Card Type',
                          'Has Chip', 'Credit Limit', 'Card on Dark Web']].copy()
    cards_clean = cards_clean.rename(columns={'CARD INDEX': 'Card'})

    # ── Join users onto transactions ──────────────────────────
    print("Joining users...")
    df_joined = df.merge(users_clean, on='User', how='left')

    # ── Join cards onto transactions ──────────────────────────
    print("Joining cards...")
    df_joined = df_joined.merge(cards_clean, on=['User', 'Card'], how='left')

    print("Saving to Drive...")
    df_joined.to_csv(JOINED_PATH, index=False)
    print("Saved.")

print(f"\nFinal shape: {df_joined.shape}")
print(f"Columns: {list(df_joined.columns)}")
print(f"Missing values:\n{df_joined.isnull().sum()[df_joined.isnull().sum() > 0]}")
print(f"Memory: {df_joined.memory_usage(deep=True).sum()/1e9:.2f} GB")

Loading joined data from Drive...
Loaded.

Final shape: (1727004, 24)
Columns: ['User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Amount', 'Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?', 'Is Fraud?', 'Latitude', 'Longitude', 'Yearly Income - Person', 'FICO Score', 'Card Brand', 'Card Type', 'Has Chip', 'Credit Limit', 'Card on Dark Web']
Missing values:
Merchant State     219953
Zip                235312
Errors?           1699420
dtype: int64
Memory: 1.34 GB


## Section 4: Data Cleaning
Cleaning all columns for HOBA feature engineering:
- Amount, Credit Limit, Yearly Income: strip $ and commas → float
- Is Fraud?: Yes/No → 1/0 binary target
- Errors?: NaN means no error → binary flag (1=error, 0=clean transaction)
- Merchant State: NaN = online transaction → fill with 'ONLINE'
- Zip: NaN = online transaction → fill with 0
- Datetime: combine Year/Month/Day/Time into single datetime column
- Card identifier: User_Card combined key for groupby operations
- Unix timestamp: for time-aware rolling window computations
- Online flag: binary indicator from Use Chip field
Result saved to Drive immediately.

In [ ]:
CLEAN_PATH = f'{DRIVE_PATH}/clean_2015_2019.csv'

if os.path.exists(CLEAN_PATH):
    print("Loading clean data from Drive...")
    df_clean = pd.read_csv(CLEAN_PATH)
    df_clean['datetime'] = pd.to_datetime(df_clean['datetime'])
    print("Loaded.")
else:
    print("Cleaning data...")
    df_clean = df_joined.copy()

    # ── Numeric: strip $ and commas ──────────────────────────
    for col in ['Amount', 'Credit Limit', 'Yearly Income - Person']:
        df_clean[col] = (df_clean[col].astype(str)
                         .str.replace('[\$,]', '', regex=True)
                         .astype(float))

    # ── Target variable ───────────────────────────────────────
    df_clean['is_fraud'] = (df_clean['Is Fraud?'] == 'Yes').astype(int)

    # ── Errors: binary flag ───────────────────────────────────
    df_clean['has_error'] = df_clean['Errors?'].notna().astype(int)

    # ── Fill missing geographic fields ────────────────────────
    df_clean['Merchant State'] = df_clean['Merchant State'].fillna('ONLINE')
    df_clean['Zip'] = df_clean['Zip'].fillna(0)

    # ── Datetime column ───────────────────────────────────────
    df_clean['datetime'] = pd.to_datetime(
        df_clean['Year'].astype(str) + '-' +
        df_clean['Month'].astype(str) + '-' +
        df_clean['Day'].astype(str) + ' ' +
        df_clean['Time']
    )

    # ── Card identifier ───────────────────────────────────────
    df_clean['card_id'] = (df_clean['User'].astype(str) + '_' +
                           df_clean['Card'].astype(str))

    # ── Unix timestamp for rolling computations ───────────────
    df_clean['unix_time'] = df_clean['datetime'].astype(np.int64) // 10**9

    # ── Online transaction flag ───────────────────────────────
    df_clean['is_online'] = (df_clean['Use Chip'] == 'Online Transaction').astype(int)

    # ── Drop redundant original columns ──────────────────────
    df_clean = df_clean.drop(columns=['Is Fraud?', 'Errors?', 'Year',
                                       'Month', 'Day', 'Time'])

    # ── Sort chronologically within each card ─────────────────
    df_clean = df_clean.sort_values(
        ['card_id', 'datetime']).reset_index(drop=True)

    print("Saving to Drive...")
    df_clean.to_csv(CLEAN_PATH, index=False)
    print("Saved.")

print(f"\nFinal shape: {df_clean.shape}")
print(f"Columns: {list(df_clean.columns)}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")
print(f"Fraud rate: {df_clean['is_fraud'].mean()*100:.4f}%")
print(f"Date range: {df_clean['datetime'].min()} to {df_clean['datetime'].max()}")
print(f"\nSample:")
print(df_clean[['card_id','datetime','Amount','MCC','Use Chip',
                'is_fraud','Credit Limit','Latitude','Longitude']].head(3))

Cleaning data...
Saving to Drive...
Saved.

Final shape: (1727004, 24)
Columns: ['User', 'Card', 'Amount', 'Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Latitude', 'Longitude', 'Yearly Income - Person', 'FICO Score', 'Card Brand', 'Card Type', 'Has Chip', 'Credit Limit', 'Card on Dark Web', 'is_fraud', 'has_error', 'datetime', 'card_id', 'unix_time', 'is_online']
Missing values: 0
Fraud rate: 0.6771%
Date range: 2015-01-01 00:01:00 to 2019-12-31 23:59:00

Sample:
  card_id            datetime  Amount   MCC          Use Chip  is_fraud  \
0     0_0 2015-01-02 06:58:00   34.99  5411  Chip Transaction         0   
1     0_0 2015-01-07 06:04:00   75.42  5912  Chip Transaction         0   
2     0_0 2015-01-13 06:27:00  101.89  5411  Chip Transaction         0   

   Credit Limit  Latitude  Longitude  
0       24295.0     34.15    -117.76  
1       24295.0     34.15    -117.76  
2       24295.0     34.15    -117.76  


## Section 5: Exploratory Data Analysis
Understanding fraud patterns across key dimensions before feature engineering.
EDA findings directly inform HOBA characteristic definitions and rule-based flags.

In [ ]:
# ── Basic fraud statistics ────────────────────────────────────
print("=== BASIC FRAUD STATISTICS ===")
print(f"Total transactions: {len(df_clean):,}")
print(f"Fraud transactions: {df_clean['is_fraud'].sum():,}")
print(f"Fraud rate: {df_clean['is_fraud'].mean()*100:.4f}%")
print(f"Unique cards: {df_clean['card_id'].nunique():,}")
print(f"Unique MCCs: {df_clean['MCC'].nunique()}")
print(f"Date range: {df_clean['datetime'].min()} to {df_clean['datetime'].max()}")

# ── Fraud by entry mode ───────────────────────────────────────
print("\n=== FRAUD RATE BY ENTRY MODE ===")
entry_fraud = df_clean.groupby('Use Chip')['is_fraud'].agg(['mean','sum','count'])
entry_fraud['fraud_rate_%'] = (entry_fraud['mean']*100).round(4)
print(entry_fraud[['fraud_rate_%','sum','count']].sort_values('fraud_rate_%', ascending=False))

# ── Fraud by hour ─────────────────────────────────────────────
print("\n=== FRAUD RATE BY HOUR ===")
df_clean['hour'] = df_clean['datetime'].dt.hour
hour_fraud = df_clean.groupby('hour')['is_fraud'].mean()*100
print(hour_fraud.round(4).to_string())

# ── Amount statistics ─────────────────────────────────────────
print("\n=== AMOUNT: FRAUD VS LEGIT ===")
print("Legitimate:")
print(df_clean[df_clean['is_fraud']==0]['Amount'].describe().round(2))
print("\nFraud:")
print(df_clean[df_clean['is_fraud']==1]['Amount'].describe().round(2))

# ── Utilization ratio ─────────────────────────────────────────
print("\n=== UTILIZATION RATIO (Amount/Credit Limit) ===")
df_clean['utilization'] = df_clean['Amount'] / df_clean['Credit Limit']
print("Legitimate:")
print(df_clean[df_clean['is_fraud']==0]['utilization'].describe().round(4))
print("\nFraud:")
print(df_clean[df_clean['is_fraud']==1]['utilization'].describe().round(4))

=== BASIC FRAUD STATISTICS ===
Total transactions: 1,727,004
Fraud transactions: 11,693
Fraud rate: 0.6771%
Unique cards: 4,435
Unique MCCs: 109
Date range: 2015-01-01 00:01:00 to 2019-12-31 23:59:00

=== FRAUD RATE BY ENTRY MODE ===
                    fraud_rate_%   sum    count
Use Chip                                       
Online Transaction        2.7369  5978   218420
Chip Transaction          0.3978  4836  1215663
Swipe Transaction         0.3001   879   292921

=== FRAUD RATE BY HOUR ===
hour
0     0.0057
1     0.0133
2     0.0399
3     0.1398
4     0.2228
5     0.1558
6     0.0400
7     0.0687
8     0.0663
9     0.4486
10    1.2341
11    1.2962
12    1.1324
13    1.1368
14    1.0090
15    0.9676
16    0.9006
17    1.0469
18    0.9932
19    0.8552
20    0.0725
21    0.0689
22    0.0037
23    0.0047

=== AMOUNT: FRAUD VS LEGIT ===
Legitimate:
count    1715311.00
mean          42.79
std           80.49
min         -500.00
25%            9.04
50%           29.20
75%           63.

## Section 5.1: Utilization Ratio Diagnostic and Hour × Entry Mode Analysis
Diagnosing infinity/NaN values in utilization ratio before using it as a feature.
Also checking whether hour-of-day fraud signal is independent of entry mode
or driven purely by online transaction timing patterns.

In [ ]:
# ── Utilization diagnostic ────────────────────────────────────
print("Credit Limit distribution:")
print(df_clean['Credit Limit'].describe())
print(f"\nZero credit limits: {(df_clean['Credit Limit']==0).sum()}")
print(f"Negative credit limits: {(df_clean['Credit Limit']<0).sum()}")
print(f"Negative amounts: {(df_clean['Amount']<0).sum()}")
print(f"\nInf utilization count: {np.isinf(df_clean['utilization']).sum()}")
print(f"NaN utilization count: {np.isnan(df_clean['utilization']).sum()}")

# ── Hour × entry mode interaction ─────────────────────────────
print("\n=== FRAUD RATE BY HOUR — ONLINE ONLY ===")
online = df_clean[df_clean['is_online']==1]
print(online.groupby('hour')['is_fraud'].mean().mul(100).round(4).to_string())

print("\n=== FRAUD RATE BY HOUR — CHIP + SWIPE ONLY ===")
physical = df_clean[df_clean['is_online']==0]
print(physical.groupby('hour')['is_fraud'].mean().mul(100).round(4).to_string())

Credit Limit distribution:
count    1.727004e+06
mean     1.540618e+04
std      1.201326e+04
min      0.000000e+00
25%      7.784000e+03
50%      1.343800e+04
75%      2.075900e+04
max      1.413910e+05
Name: Credit Limit, dtype: float64

Zero credit limits: 8078
Negative credit limits: 0
Negative amounts: 83426

Inf utilization count: 8065
NaN utilization count: 13

=== FRAUD RATE BY HOUR — ONLINE ONLY ===
hour
0     0.0249
1     0.0666
2     0.2370
3     1.0411
4     1.0313
5     0.9552
6     0.2813
7     0.5768
8     0.5558
9     3.2898
10    6.9930
11    6.0546
12    5.8459
13    4.6299
14    3.0755
15    3.0845
16    2.2967
17    2.8265
18    1.5174
19    0.7277
20    0.5252
21    0.4980
22    0.0186
23    0.0273

=== FRAUD RATE BY HOUR — CHIP + SWIPE ONLY ===
hour
0     0.0000
1     0.0000
2     0.0000
3     0.0084
4     0.0268
5     0.0202
6     0.0104
7     0.0049
8     0.0126
9     0.0703
10    0.4183
11    0.6310
12    0.5873
13    0.6480
14    0.6218
15    0.6062
16    0.627

## Section 5.2: MCC Fraud Rate Analysis
Identifying high-risk MCCs to define the high-risk MCC HOBA characteristic.
Top fraud-rate MCCs will be flagged as high-risk for characteristic segmentation.

In [ ]:
# ── Fraud rate by MCC ─────────────────────────────────────────
mcc_fraud = df_clean.groupby('MCC')['is_fraud'].agg(['mean','sum','count'])
mcc_fraud.columns = ['fraud_rate', 'fraud_count', 'total']
mcc_fraud['fraud_rate_%'] = (mcc_fraud['fraud_rate']*100).round(4)
mcc_fraud = mcc_fraud[mcc_fraud['total'] >= 100]  # min 100 transactions for reliability
mcc_fraud = mcc_fraud.sort_values('fraud_rate_%', ascending=False)

print(f"=== TOP 20 HIGH-RISK MCCs ===")
print(mcc_fraud[['fraud_rate_%','fraud_count','total']].head(20).to_string())

print(f"\n=== BOTTOM 10 LOW-RISK MCCs ===")
print(mcc_fraud[['fraud_rate_%','fraud_count','total']].tail(10).to_string())

print(f"\nOverall MCC fraud rate: {df_clean['is_fraud'].mean()*100:.4f}%")
print(f"MCCs with fraud rate > 1%: {(mcc_fraud['fraud_rate_%'] > 1).sum()}")
print(f"MCCs with zero fraud: {(mcc_fraud['fraud_rate_%'] == 0).sum()}")

=== TOP 20 HIGH-RISK MCCs ===
      fraud_rate_%  fraud_count  total
MCC                                   
4411       86.0606          142    165
5045       42.2414          196    464
5094       29.0023          250    862
5712       27.9483          173    619
5732       24.3968          273   1119
4112       20.5128           72    351
5816       17.3127           67    387
5932       12.6643          106    837
3389        9.5583          132   1381
4131        8.6379           26    301
7996        8.0082          117   1461
5947        7.5540           21    278
5193        7.4165           91   1227
3359        7.2085          102   1415
4511        6.5421           28    428
3504        6.4548          132   2045
3722        5.9730          124   2076
3000        5.8524           23    393
3509        5.4904          117   2131
5977        5.1125           50    978

=== BOTTOM 10 LOW-RISK MCCs ===
      fraud_rate_%  fraud_count  total
MCC                                   
5

In [ ]:
print(f"MCCs with fraud rate > 5%: {(mcc_fraud['fraud_rate_%'] > 5).sum()}")
print(f"Transactions in these MCCs: {mcc_fraud[mcc_fraud['fraud_rate_%'] > 5]['total'].sum():,}")
print(f"Fraud in these MCCs: {mcc_fraud[mcc_fraud['fraud_rate_%'] > 5]['fraud_count'].sum():,}")
print(f"% of all fraud captured: {mcc_fraud[mcc_fraud['fraud_rate_%'] > 5]['fraud_count'].sum()/df_clean['is_fraud'].sum()*100:.1f}%")

MCCs with fraud rate > 5%: 20
Transactions in these MCCs: 18,918
Fraud in these MCCs: 2,242
% of all fraud captured: 19.2%


In [ ]:
# Create MCC fraud rate mapping from training data only (2015-2018)
# Never use test year (2019) to compute this — leakage prevention
train_mask = df_clean['datetime'].dt.year < 2019
mcc_fraud_rate = (df_clean[train_mask]
                  .groupby('MCC')['is_fraud']
                  .mean()
                  .rename('mcc_fraud_rate'))

# Check coverage
print(f"MCCs in training data: {len(mcc_fraud_rate)}")
print(f"MCCs in full dataset: {df_clean['MCC'].nunique()}")
print(f"\nMCC fraud rate distribution:")
print(mcc_fraud_rate.describe().round(4))

# How many transactions get a non-zero MCC fraud rate
df_clean['mcc_fraud_rate'] = df_clean['MCC'].map(mcc_fraud_rate).fillna(0)
print(f"\nTransactions with mcc_fraud_rate > 0: {(df_clean['mcc_fraud_rate']>0).sum():,}")
print(f"Transactions with mcc_fraud_rate = 0: {(df_clean['mcc_fraud_rate']==0).sum():,}")

MCCs in training data: 109
MCCs in full dataset: 109

MCC fraud rate distribution:
count    109.0000
mean       0.0718
std        0.1391
min        0.0000
25%        0.0030
50%        0.0235
75%        0.0509
max        0.8712
Name: mcc_fraud_rate, dtype: float64

Transactions with mcc_fraud_rate > 0: 1,573,243
Transactions with mcc_fraud_rate = 0: 153,761


## Section 5.3: Error Flag and Refund Analysis
Checking whether transaction errors and negative amounts (refunds)
carry independent fraud signal beyond entry mode effects.

In [ ]:
# ── Error flag fraud signal ───────────────────────────────────
print("=== ERROR FLAG FRAUD SIGNAL ===")
error_fraud = df_clean.groupby('has_error')['is_fraud'].agg(['mean','sum','count'])
error_fraud['fraud_rate_%'] = (error_fraud['mean']*100).round(4)
print(error_fraud[['fraud_rate_%','sum','count']])

# ── Error types breakdown ─────────────────────────────────────
print("\n=== FRAUD RATE BY ERROR TYPE ===")
df_joined_check = df_joined.copy()
df_joined_check['is_fraud'] = (df_joined_check['Is Fraud?']=='Yes').astype(int)
error_types = df_joined_check[df_joined_check['Errors?'].notna()]
print(error_types.groupby('Errors?')['is_fraud'].agg(['mean','sum','count'])
      .assign(fraud_rate=lambda x: (x['mean']*100).round(4))
      [['fraud_rate','sum','count']]
      .sort_values('fraud_rate', ascending=False))

# ── Refund/negative amount fraud signal ───────────────────────
print("\n=== NEGATIVE AMOUNT (REFUND) FRAUD SIGNAL ===")
df_clean['is_refund'] = (df_clean['Amount'] < 0).astype(int)
refund_fraud = df_clean.groupby('is_refund')['is_fraud'].agg(['mean','sum','count'])
refund_fraud['fraud_rate_%'] = (refund_fraud['mean']*100).round(4)
print(refund_fraud[['fraud_rate_%','sum','count']])

# ── Dark web card fraud signal ────────────────────────────────
print("\n=== DARK WEB CARD FRAUD SIGNAL ===")
dark_fraud = df_clean.groupby('Card on Dark Web')['is_fraud'].agg(['mean','sum','count'])
dark_fraud['fraud_rate_%'] = (dark_fraud['mean']*100).round(4)
print(dark_fraud[['fraud_rate_%','sum','count']])

=== ERROR FLAG FRAUD SIGNAL ===
           fraud_rate_%    sum    count
has_error                              
0                0.6595  11208  1699420
1                1.7583    485    27584

=== FRAUD RATE BY ERROR TYPE ===
                                       fraud_rate  sum  count
Errors?                                                      
Bad CVV,Insufficient Balance              27.2727    3     11
Bad CVV                                    9.9129   91    918
Bad PIN,Insufficient Balance               7.4074    2     27
Bad Expiration                             5.1370   45    876
Bad Card Number                            3.4181   39   1141
Bad PIN                                    3.3027  140   4239
Insufficient Balance                       0.8483  142  16739
Technical Glitch                           0.6765   23   3400
Bad CVV,Technical Glitch                   0.0000    0      1
Bad Card Number,Bad Expiration             0.0000    0      5
Bad Card Number,Bad CVV       

## Section 6: Characteristic Flag Engineering
Computing 9 binary characteristic flags per transaction.
Characteristics are overlapping — a single transaction can belong to
multiple characteristics simultaneously, producing dense HOBA features.
These flags determine which historical transactions are aggregated
within each HOBA feature computation.

Characteristics:
1. is_purchase: amount > 0 (positive transaction, not refund)
2. is_online: Use Chip == 'Online Transaction'
3. is_chip: Use Chip == 'Chip Transaction'  
4. is_swipe: Use Chip == 'Swipe Transaction'
5. is_abnormal_hour: hour in [9,10,11,12,13,14,15,16,17,18,19] (EDA-validated high fraud hours)
6. is_foreign_merchant: merchant state ≠ cardholder state AND not online
7. is_high_risk_mcc: mcc_fraud_rate > 0.05 (top fraud-rate MCCs from EDA)
8. is_error: has_error == 1
9. is_bad_cvv: Errors? contains 'Bad CVV'
All flags saved to Drive immediately.

"char_bad_cvv dropped — 0.1% activation rate and fraud rate (0.74%) barely above overall rate (0.68%), no meaningful signal. Replaced with char_high_amount — amount > 2× card's rolling 30-day mean, computed during HOBA feature engineering phase."

In [ ]:
FLAGS_PATH = f'{DRIVE_PATH}/df_with_flags.csv'

if os.path.exists(FLAGS_PATH):
    print("Loading flags from Drive...")
    df_flags = pd.read_csv(FLAGS_PATH)
    df_flags['datetime'] = pd.to_datetime(df_flags['datetime'])
    print("Loaded.")
else:
    print("Computing characteristic flags...")
    df_flags = df_clean.copy()

    # ── Reload errors column from joined data for Bad CVV ─────
    errors_col = df_joined['Errors?'].copy()
    errors_col.index = df_joined.index

    # ── Hour of day ───────────────────────────────────────────
    df_flags['hour'] = df_flags['datetime'].dt.hour
    df_flags['day_of_week'] = df_flags['datetime'].dt.dayofweek

    # ── Cardholder state from users file ──────────────────────
    user_state = users[['State']].copy()
    user_state.index.name = 'User'
    user_state = user_state.reset_index()
    df_flags = df_flags.merge(user_state, on='User', how='left')
    df_flags = df_flags.rename(columns={'State': 'cardholder_state'})

    # ── Characteristic 1: Purchase ────────────────────────────
    df_flags['char_purchase'] = (df_flags['Amount'] > 0).astype(int)

    # ── Characteristic 2: Online ──────────────────────────────
    df_flags['char_online'] = (df_flags['Use Chip'] == 'Online Transaction').astype(int)

    # ── Characteristic 3: Chip ────────────────────────────────
    df_flags['char_chip'] = (df_flags['Use Chip'] == 'Chip Transaction').astype(int)

    # ── Characteristic 4: Swipe ───────────────────────────────
    df_flags['char_swipe'] = (df_flags['Use Chip'] == 'Swipe Transaction').astype(int)

    # ── Characteristic 5: Abnormal hour (EDA-validated) ──────
    abnormal_hours = list(range(9, 20))  # 9am-7pm high fraud window
    df_flags['char_abnormal_hour'] = df_flags['hour'].isin(abnormal_hours).astype(int)

    # ── Characteristic 6: Foreign merchant ───────────────────
    df_flags['char_foreign_merchant'] = (
        (df_flags['Merchant State'] != df_flags['cardholder_state']) &
        (df_flags['Merchant State'] != 'ONLINE') &
        (df_flags['char_online'] == 0)
    ).astype(int)

    # ── Characteristic 7: High risk MCC ──────────────────────
    df_flags['char_high_risk_mcc'] = (df_flags['mcc_fraud_rate'] > 0.05).astype(int)

    # ── Characteristic 8: Error transaction ──────────────────
    df_flags['char_error'] = df_flags['has_error'].astype(int)

    # ── Characteristic 9: Bad CVV ────────────────────────────
    # Reload from original joined file
    bad_cvv_mask = df_joined['Errors?'].fillna('').str.contains('Bad CVV')
    bad_cvv_series = bad_cvv_mask.values[:len(df_flags)]
    df_flags['char_bad_cvv'] = bad_cvv_series.astype(int)

    print("Saving to Drive...")
    df_flags.to_csv(FLAGS_PATH, index=False)
    print("Saved.")

# ── Verify flags ──────────────────────────────────────────────
char_cols = [c for c in df_flags.columns if c.startswith('char_')]
print(f"\nCharacteristic flags: {char_cols}")
print(f"\nFlag activation rates:")
for col in char_cols:
    rate = df_flags[col].mean()*100
    fraud_rate = df_flags[df_flags[col]==1]['is_fraud'].mean()*100
    print(f"  {col}: {rate:.1f}% of transactions | fraud rate when active: {fraud_rate:.4f}%")

Loading flags from Drive...
Loaded.

Characteristic flags: ['char_purchase', 'char_online', 'char_chip', 'char_swipe', 'char_abnormal_hour', 'char_foreign_merchant', 'char_high_risk_mcc', 'char_error', 'char_bad_cvv']

Flag activation rates:
  char_purchase: 95.1% of transactions | fraud rate when active: 0.6900%
  char_online: 12.6% of transactions | fraud rate when active: 2.7369%
  char_chip: 70.4% of transactions | fraud rate when active: 0.3978%
  char_swipe: 17.0% of transactions | fraud rate when active: 0.3001%
  char_abnormal_hour: 64.7% of transactions | fraud rate when active: 1.0135%
  char_foreign_merchant: 8.9% of transactions | fraud rate when active: 3.0836%
  char_high_risk_mcc: 1.0% of transactions | fraud rate when active: 13.3739%
  char_error: 1.6% of transactions | fraud rate when active: 1.7583%
  char_bad_cvv: 0.1% of transactions | fraud rate when active: 0.7447%


## Section 7: HOBA Feature Engineering
Computing 768 aggregated HOBA features across 8 characteristics × 8 periods
× 3 measures × 4 statistics, plus 24 current transaction snapshot features.

Aggregation periods:
- Time-based: past 1 day, 3 days, 5 days
- Count-based: past 5, 7, 15 transactions within characteristic
- Lag windows: past 6th-10th, 8th-15th transactions within characteristic
- Current snapshot: current transaction's own values

Behavior measures (3):
- Transaction amount
- Time interval between sequential transactions within characteristic (seconds)
- Utilization ratio (amount / credit_limit, zero-limit cards set to limit=1)

Aggregation statistics (4): mean, std, max, sum

Key implementation rules:
1. Features computed using ONLY past transactions (shift to exclude current)
2. Zero-fill when no history exists in characteristic — meaningful signal
3. Characteristics are overlapping binary flags — dense features result
4. Time intervals computed within characteristic sequence, not overall
All features saved to Drive immediately after computation.

## Section 7.1: HOBA Features — char_purchase
Computing HOBA aggregation features for the purchase characteristic.
purchase = Amount > 0 (95.1% of transactions).
Vectorized pandas rolling operations — no Python loops over rows.
Time intervals computed as unix_time.diff() within card's purchase sequence.
Saved to Drive immediately after computation.

In [ ]:
import gc
import os
import numpy as np
import pandas as pd

HOBA_DIR = f'{DRIVE_PATH}/hoba_chars'
os.makedirs(HOBA_DIR, exist_ok=True)

char      = 'char_purchase'
char_name = 'purchase'
char_path = f'{HOBA_DIR}/{char_name}.csv'

if os.path.exists(char_path):
    print(f"✓ {char_name} already saved on Drive. Loading to verify...")
    saved = pd.read_csv(char_path)
    print(f"  Shape: {saved.shape}")
    del saved
else:
    print(f"Processing: {char_name}")

    # ── Filter and prepare ────────────────────────────────────
    cols_needed = ['card_id', 'datetime', 'unix_time',
                   'Amount', 'utilization', 'is_fraud']
    char_df = df_flags[df_flags[char] == 1][cols_needed].copy()
    char_df = char_df.sort_values(['card_id', 'datetime'])
    char_df['orig_index'] = char_df.index
    char_df = char_df.set_index('datetime')

    print(f"  Rows in characteristic: {len(char_df):,}")

    # ── Time interval within characteristic sequence ──────────
    char_df['time_interval'] = (char_df.groupby('card_id')['unix_time']
                                 .diff().fillna(0))

    # ── Helper: vectorized rolling stats ─────────────────────
    def rolling_stats(series_shifted, window, is_count=False):
        if is_count:
            r = series_shifted.rolling(window, min_periods=1)
        else:
            r = series_shifted.rolling(window, min_periods=1)
        return {
            'mean': r.mean(),
            'std':  r.std().fillna(0),
            'max':  r.max(),
            'sum':  r.sum()
        }

    # ── Build result dataframe ────────────────────────────────
    result = pd.DataFrame(index=char_df.index)
    result['orig_index'] = char_df['orig_index']

    measures = [('Amount', 'amt'), ('time_interval', 'timeint'), ('utilization', 'util')]

    # ── Time-based windows ────────────────────────────────────
    for period_name, window in [('1d','1D'), ('3d','3D'), ('5d','5D')]:
        print(f"  Computing {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp     = shifted.groupby(char_df['card_id'])
            r       = grp.rolling(window, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    # ── Count-based windows ───────────────────────────────────
    for period_name, cnt in [('cnt5',5), ('cnt7',7), ('cnt15',15)]:
        print(f"  Computing {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            r       = shifted.groupby(char_df['card_id']).rolling(cnt, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    # ── Lag windows ───────────────────────────────────────────
    for period_name, start, end in [('lag6to10',5,10), ('lag8to15',7,15)]:
        print(f"  Computing {period_name}...")
        for col, cm in measures:
            # lag window = rolling(end) - rolling(start)
            shifted  = char_df.groupby('card_id')[col].shift(1)
            grp      = shifted.groupby(char_df['card_id'])
            r_end    = grp.rolling(end,   min_periods=1)
            r_start  = grp.rolling(start, min_periods=1)
            # Sum of lag window = sum(end) - sum(start)
            lag_sum  = r_end.sum().values  - r_start.sum().values
            lag_cnt  = r_end.count().values - r_start.count().values
            lag_cnt  = np.where(lag_cnt <= 0, 1, lag_cnt)
            lag_mean = lag_sum / lag_cnt
            # For max and std we use shift-based approach
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = lag_mean
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = 0.0
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = lag_sum

    # ── Current snapshot ──────────────────────────────────────
    print("  Computing current snapshot...")
    result[f'hoba_{char_name}_current_amt']     = char_df['Amount'].values
    result[f'hoba_{char_name}_current_util']    = char_df['utilization'].values
    result[f'hoba_{char_name}_current_timeint'] = char_df['time_interval'].values

    # ── Fill NaN with 0 ───────────────────────────────────────
    result = result.fillna(0)

    # ── Save to Drive ─────────────────────────────────────────
    print("  Saving to Drive...")
    result.to_csv(char_path, index=False)
    print(f"  ✓ Saved. Shape: {result.shape}")

    # ── Free memory ───────────────────────────────────────────
    del char_df, result
    gc.collect()
    print("  Memory freed.")

✓ purchase already saved on Drive. Loading to verify...
  Shape: (1642252, 100)


## Section 7.2-7.9: HOBA Features — Remaining Characteristics
Running identical vectorized pipeline for each remaining characteristic.
Each cell processes one characteristic, saves to Drive, frees memory.
Run cells sequentially — confirm each saves before running next.

In [ ]:
gc.collect()
char, char_name = 'char_online', 'online'
char_path = f'{HOBA_DIR}/{char_name}.csv'

if os.path.exists(char_path):
    print(f"✓ {char_name} already on Drive.")
else:
    print(f"Processing: {char_name}")
    cols_needed = ['card_id','datetime','unix_time','Amount','utilization','is_fraud']
    char_df = df_flags[df_flags[char]==1][cols_needed].copy()
    char_df = char_df.sort_values(['card_id','datetime'])
    char_df['orig_index'] = char_df.index
    char_df = char_df.set_index('datetime')
    print(f"  Rows: {len(char_df):,}")

    char_df['time_interval'] = char_df.groupby('card_id')['unix_time'].diff().fillna(0)
    result = pd.DataFrame(index=char_df.index)
    result['orig_index'] = char_df['orig_index']
    measures = [('Amount','amt'),('time_interval','timeint'),('utilization','util')]

    for period_name, window in [('1d','1D'),('3d','3D'),('5d','5D')]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(char_df['card_id'])
            r = grp.rolling(window, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    for period_name, cnt in [('cnt5',5),('cnt7',7),('cnt15',15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            r = shifted.groupby(char_df['card_id']).rolling(cnt, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    for period_name, start, end in [('lag6to10',5,10),('lag8to15',7,15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(char_df['card_id'])
            r_end   = grp.rolling(end,   min_periods=1)
            r_start = grp.rolling(start, min_periods=1)
            lag_sum = r_end.sum().values - r_start.sum().values
            lag_cnt = np.where((r_end.count().values - r_start.count().values) <= 0, 1,
                               r_end.count().values - r_start.count().values)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = 0.0
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = lag_sum

    result[f'hoba_{char_name}_current_amt']     = char_df['Amount'].values
    result[f'hoba_{char_name}_current_util']    = char_df['utilization'].values
    result[f'hoba_{char_name}_current_timeint'] = char_df['time_interval'].values
    result = result.fillna(0)
    result.to_csv(char_path, index=False)
    print(f"  ✓ Saved. Shape: {result.shape}")
    del char_df, result; gc.collect()

Processing: online
  Rows: 218,420
  1d...
  3d...
  5d...
  cnt5...
  cnt7...
  cnt15...
  lag6to10...
  lag8to15...
  ✓ Saved. Shape: (218420, 100)


In [ ]:
gc.collect()
char, char_name = 'char_chip', 'chip'
char_path = f'{HOBA_DIR}/{char_name}.csv'

if os.path.exists(char_path):
    print(f"✓ {char_name} already on Drive.")
else:
    print(f"Processing: {char_name}")
    cols_needed = ['card_id','datetime','unix_time','Amount','utilization','is_fraud']
    char_df = df_flags[df_flags[char]==1][cols_needed].copy()
    char_df = char_df.sort_values(['card_id','datetime'])
    char_df['orig_index'] = char_df.index
    char_df = char_df.set_index('datetime')
    print(f"  Rows: {len(char_df):,}")

    char_df['time_interval'] = char_df.groupby('card_id')['unix_time'].diff().fillna(0)
    result = pd.DataFrame(index=char_df.index)
    result['orig_index'] = char_df['orig_index']
    measures = [('Amount','amt'),('time_interval','timeint'),('utilization','util')]

    for period_name, window in [('1d','1D'),('3d','3D'),('5d','5D')]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(char_df['card_id'])
            r = grp.rolling(window, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    for period_name, cnt in [('cnt5',5),('cnt7',7),('cnt15',15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            r = shifted.groupby(char_df['card_id']).rolling(cnt, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    for period_name, start, end in [('lag6to10',5,10),('lag8to15',7,15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(char_df['card_id'])
            r_end   = grp.rolling(end,   min_periods=1)
            r_start = grp.rolling(start, min_periods=1)
            lag_sum = r_end.sum().values - r_start.sum().values
            lag_cnt = np.where((r_end.count().values - r_start.count().values) <= 0, 1,
                               r_end.count().values - r_start.count().values)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = 0.0
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = lag_sum

    result[f'hoba_{char_name}_current_amt']     = char_df['Amount'].values
    result[f'hoba_{char_name}_current_util']    = char_df['utilization'].values
    result[f'hoba_{char_name}_current_timeint'] = char_df['time_interval'].values
    result = result.fillna(0)
    result.to_csv(char_path, index=False)
    print(f"  ✓ Saved. Shape: {result.shape}")
    del char_df, result; gc.collect()

Processing: chip
  Rows: 1,215,663
  1d...
  3d...
  5d...
  cnt5...
  cnt7...
  cnt15...
  lag6to10...
  lag8to15...
  ✓ Saved. Shape: (1215663, 100)


In [ ]:
gc.collect()
char, char_name = 'char_swipe', 'swipe'
char_path = f'{HOBA_DIR}/{char_name}.csv'

if os.path.exists(char_path):
    print(f"✓ {char_name} already on Drive.")
else:
    print(f"Processing: {char_name}")
    cols_needed = ['card_id','datetime','unix_time','Amount','utilization','is_fraud']
    char_df = df_flags[df_flags[char]==1][cols_needed].copy()
    char_df = char_df.sort_values(['card_id','datetime'])
    char_df['orig_index'] = char_df.index
    char_df = char_df.set_index('datetime')
    print(f"  Rows: {len(char_df):,}")

    char_df['time_interval'] = char_df.groupby('card_id')['unix_time'].diff().fillna(0)
    result = pd.DataFrame(index=char_df.index)
    result['orig_index'] = char_df['orig_index']
    measures = [('Amount','amt'),('time_interval','timeint'),('utilization','util')]

    for period_name, window in [('1d','1D'),('3d','3D'),('5d','5D')]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(char_df['card_id'])
            r = grp.rolling(window, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    for period_name, cnt in [('cnt5',5),('cnt7',7),('cnt15',15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            r = shifted.groupby(char_df['card_id']).rolling(cnt, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    for period_name, start, end in [('lag6to10',5,10),('lag8to15',7,15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(char_df['card_id'])
            r_end   = grp.rolling(end,   min_periods=1)
            r_start = grp.rolling(start, min_periods=1)
            lag_sum = r_end.sum().values - r_start.sum().values
            lag_cnt = np.where((r_end.count().values - r_start.count().values) <= 0, 1,
                               r_end.count().values - r_start.count().values)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = 0.0
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = lag_sum

    result[f'hoba_{char_name}_current_amt']     = char_df['Amount'].values
    result[f'hoba_{char_name}_current_util']    = char_df['utilization'].values
    result[f'hoba_{char_name}_current_timeint'] = char_df['time_interval'].values
    result = result.fillna(0)
    result.to_csv(char_path, index=False)
    print(f"  ✓ Saved. Shape: {result.shape}")
    del char_df, result; gc.collect()

Processing: swipe
  Rows: 292,921
  1d...
  3d...
  5d...
  cnt5...
  cnt7...
  cnt15...
  lag6to10...
  lag8to15...
  ✓ Saved. Shape: (292921, 100)


In [ ]:
gc.collect()
char, char_name = 'char_abnormal_hour', 'abnormal_hour'
char_path = f'{HOBA_DIR}/{char_name}.csv'

if os.path.exists(char_path):
    print(f"✓ {char_name} already on Drive.")
else:
    print(f"Processing: {char_name}")
    cols_needed = ['card_id','datetime','unix_time','Amount','utilization','is_fraud']
    char_df = df_flags[df_flags[char]==1][cols_needed].copy()
    char_df = char_df.sort_values(['card_id','datetime'])
    char_df['orig_index'] = char_df.index
    char_df = char_df.set_index('datetime')
    print(f"  Rows: {len(char_df):,}")

    char_df['time_interval'] = char_df.groupby('card_id')['unix_time'].diff().fillna(0)
    result = pd.DataFrame(index=char_df.index)
    result['orig_index'] = char_df['orig_index']
    measures = [('Amount','amt'),('time_interval','timeint'),('utilization','util')]

    for period_name, window in [('1d','1D'),('3d','3D'),('5d','5D')]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(char_df['card_id'])
            r = grp.rolling(window, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    for period_name, cnt in [('cnt5',5),('cnt7',7),('cnt15',15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            r = shifted.groupby(char_df['card_id']).rolling(cnt, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    for period_name, start, end in [('lag6to10',5,10),('lag8to15',7,15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(char_df['card_id'])
            r_end   = grp.rolling(end,   min_periods=1)
            r_start = grp.rolling(start, min_periods=1)
            lag_sum = r_end.sum().values - r_start.sum().values
            lag_cnt = np.where((r_end.count().values - r_start.count().values) <= 0, 1,
                               r_end.count().values - r_start.count().values)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = 0.0
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = lag_sum

    result[f'hoba_{char_name}_current_amt']     = char_df['Amount'].values
    result[f'hoba_{char_name}_current_util']    = char_df['utilization'].values
    result[f'hoba_{char_name}_current_timeint'] = char_df['time_interval'].values
    result = result.fillna(0)
    result.to_csv(char_path, index=False)
    print(f"  ✓ Saved. Shape: {result.shape}")
    del char_df, result; gc.collect()

Processing: abnormal_hour
  Rows: 1,117,318
  1d...
  3d...
  5d...
  cnt5...
  cnt7...
  cnt15...
  lag6to10...
  lag8to15...
  ✓ Saved. Shape: (1117318, 100)


In [ ]:
gc.collect()
char, char_name = 'char_foreign_merchant', 'foreign_merchant'
char_path = f'{HOBA_DIR}/{char_name}.csv'

if os.path.exists(char_path):
    print(f"✓ {char_name} already on Drive.")
else:
    print(f"Processing: {char_name}")
    cols_needed = ['card_id','datetime','unix_time','Amount','utilization','is_fraud']
    char_df = df_flags[df_flags[char]==1][cols_needed].copy()
    char_df = char_df.sort_values(['card_id','datetime'])
    char_df['orig_index'] = char_df.index
    char_df = char_df.set_index('datetime')
    print(f"  Rows: {len(char_df):,}")

    char_df['time_interval'] = char_df.groupby('card_id')['unix_time'].diff().fillna(0)
    result = pd.DataFrame(index=char_df.index)
    result['orig_index'] = char_df['orig_index']
    measures = [('Amount','amt'),('time_interval','timeint'),('utilization','util')]

    for period_name, window in [('1d','1D'),('3d','3D'),('5d','5D')]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(char_df['card_id'])
            r = grp.rolling(window, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    for period_name, cnt in [('cnt5',5),('cnt7',7),('cnt15',15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            r = shifted.groupby(char_df['card_id']).rolling(cnt, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    for period_name, start, end in [('lag6to10',5,10),('lag8to15',7,15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(char_df['card_id'])
            r_end   = grp.rolling(end,   min_periods=1)
            r_start = grp.rolling(start, min_periods=1)
            lag_sum = r_end.sum().values - r_start.sum().values
            lag_cnt = np.where((r_end.count().values - r_start.count().values) <= 0, 1,
                               r_end.count().values - r_start.count().values)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = 0.0
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = lag_sum

    result[f'hoba_{char_name}_current_amt']     = char_df['Amount'].values
    result[f'hoba_{char_name}_current_util']    = char_df['utilization'].values
    result[f'hoba_{char_name}_current_timeint'] = char_df['time_interval'].values
    result = result.fillna(0)
    result.to_csv(char_path, index=False)
    print(f"  ✓ Saved. Shape: {result.shape}")
    del char_df, result; gc.collect()

Processing: foreign_merchant
  Rows: 153,523
  1d...
  3d...
  5d...
  cnt5...
  cnt7...
  cnt15...
  lag6to10...
  lag8to15...
  ✓ Saved. Shape: (153523, 100)


In [ ]:
gc.collect()
char, char_name = 'char_high_risk_mcc', 'high_risk_mcc'
char_path = f'{HOBA_DIR}/{char_name}.csv'

if os.path.exists(char_path):
    print(f"✓ {char_name} already on Drive.")
else:
    print(f"Processing: {char_name}")
    cols_needed = ['card_id','datetime','unix_time','Amount','utilization','is_fraud']
    char_df = df_flags[df_flags[char]==1][cols_needed].copy()
    char_df = char_df.sort_values(['card_id','datetime'])
    char_df['orig_index'] = char_df.index
    char_df = char_df.set_index('datetime')
    print(f"  Rows: {len(char_df):,}")

    char_df['time_interval'] = char_df.groupby('card_id')['unix_time'].diff().fillna(0)
    result = pd.DataFrame(index=char_df.index)
    result['orig_index'] = char_df['orig_index']
    measures = [('Amount','amt'),('time_interval','timeint'),('utilization','util')]

    for period_name, window in [('1d','1D'),('3d','3D'),('5d','5D')]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(char_df['card_id'])
            r = grp.rolling(window, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    for period_name, cnt in [('cnt5',5),('cnt7',7),('cnt15',15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            r = shifted.groupby(char_df['card_id']).rolling(cnt, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    for period_name, start, end in [('lag6to10',5,10),('lag8to15',7,15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(char_df['card_id'])
            r_end   = grp.rolling(end,   min_periods=1)
            r_start = grp.rolling(start, min_periods=1)
            lag_sum = r_end.sum().values - r_start.sum().values
            lag_cnt = np.where((r_end.count().values - r_start.count().values) <= 0, 1,
                               r_end.count().values - r_start.count().values)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = 0.0
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = lag_sum

    result[f'hoba_{char_name}_current_amt']     = char_df['Amount'].values
    result[f'hoba_{char_name}_current_util']    = char_df['utilization'].values
    result[f'hoba_{char_name}_current_timeint'] = char_df['time_interval'].values
    result = result.fillna(0)
    result.to_csv(char_path, index=False)
    print(f"  ✓ Saved. Shape: {result.shape}")
    del char_df, result; gc.collect()

Processing: high_risk_mcc
  Rows: 17,235
  1d...
  3d...
  5d...
  cnt5...
  cnt7...
  cnt15...
  lag6to10...
  lag8to15...
  ✓ Saved. Shape: (17235, 100)


In [ ]:
gc.collect()
char, char_name = 'char_error', 'error'
char_path = f'{HOBA_DIR}/{char_name}.csv'

if os.path.exists(char_path):
    print(f"✓ {char_name} already on Drive.")
else:
    print(f"Processing: {char_name}")
    cols_needed = ['card_id','datetime','unix_time','Amount','utilization','is_fraud']
    char_df = df_flags[df_flags[char]==1][cols_needed].copy()
    char_df = char_df.sort_values(['card_id','datetime'])
    char_df['orig_index'] = char_df.index
    char_df = char_df.set_index('datetime')
    print(f"  Rows: {len(char_df):,}")

    char_df['time_interval'] = char_df.groupby('card_id')['unix_time'].diff().fillna(0)
    result = pd.DataFrame(index=char_df.index)
    result['orig_index'] = char_df['orig_index']
    measures = [('Amount','amt'),('time_interval','timeint'),('utilization','util')]

    for period_name, window in [('1d','1D'),('3d','3D'),('5d','5D')]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(char_df['card_id'])
            r = grp.rolling(window, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    for period_name, cnt in [('cnt5',5),('cnt7',7),('cnt15',15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            r = shifted.groupby(char_df['card_id']).rolling(cnt, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    for period_name, start, end in [('lag6to10',5,10),('lag8to15',7,15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(char_df['card_id'])
            r_end   = grp.rolling(end,   min_periods=1)
            r_start = grp.rolling(start, min_periods=1)
            lag_sum = r_end.sum().values - r_start.sum().values
            lag_cnt = np.where((r_end.count().values - r_start.count().values) <= 0, 1,
                               r_end.count().values - r_start.count().values)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = 0.0
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = lag_sum

    result[f'hoba_{char_name}_current_amt']     = char_df['Amount'].values
    result[f'hoba_{char_name}_current_util']    = char_df['utilization'].values
    result[f'hoba_{char_name}_current_timeint'] = char_df['time_interval'].values
    result = result.fillna(0)
    result.to_csv(char_path, index=False)
    print(f"  ✓ Saved. Shape: {result.shape}")
    del char_df, result; gc.collect()

Processing: error
  Rows: 27,584
  1d...
  3d...
  5d...
  cnt5...
  cnt7...
  cnt15...
  lag6to10...
  lag8to15...
  ✓ Saved. Shape: (27584, 100)


In [ ]:
import gc
import numpy as np

# ── Recompute char_high_amount ────────────────────────────────
print("Recomputing char_high_amount...")

df_flags['utilization'] = (df_flags['Amount'] /
                            df_flags['Credit Limit'].clip(lower=1)).clip(-1, 5)

df_flags = df_flags.sort_values(['card_id', 'datetime']).reset_index(drop=True)
df_flags = df_flags.set_index('datetime')

rolling_mean = (df_flags.groupby('card_id')['Amount']
                .apply(lambda x: x.shift(1).rolling('30D').mean()))
df_flags['rolling_mean_30d'] = rolling_mean.values
df_flags['char_high_amount'] = (
    df_flags['Amount'] > 2 * df_flags['rolling_mean_30d']
).fillna(False).astype(int)

df_flags = df_flags.reset_index()

print(f"char_high_amount activation: {df_flags['char_high_amount'].mean()*100:.1f}%")
print(f"char_high_amount column confirmed in df_flags: {'char_high_amount' in df_flags.columns}")
gc.collect()

Recomputing char_high_amount...
char_high_amount activation: 21.4%
char_high_amount column confirmed in df_flags: True


14

In [ ]:
gc.collect()
char, char_name = 'char_high_amount', 'high_amount'
char_path = f'{HOBA_DIR}/{char_name}.csv'

if os.path.exists(char_path):
    print(f"✓ {char_name} already on Drive.")
else:
    print(f"Processing: {char_name}")
    cols_needed = ['card_id','datetime','unix_time','Amount','utilization','is_fraud']
    char_df = df_flags[df_flags[char]==1][cols_needed].copy()
    char_df = char_df.sort_values(['card_id','datetime'])
    char_df['orig_index'] = char_df.index
    char_df = char_df.set_index('datetime')
    print(f"  Rows: {len(char_df):,}")

    char_df['time_interval'] = char_df.groupby('card_id')['unix_time'].diff().fillna(0)
    result = pd.DataFrame(index=char_df.index)
    result['orig_index'] = char_df['orig_index']
    measures = [('Amount','amt'),('time_interval','timeint'),('utilization','util')]

    for period_name, window in [('1d','1D'),('3d','3D'),('5d','5D')]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(char_df['card_id'])
            r = grp.rolling(window, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    for period_name, cnt in [('cnt5',5),('cnt7',7),('cnt15',15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            r = shifted.groupby(char_df['card_id']).rolling(cnt, min_periods=1)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = r.mean().values
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = r.max().values
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = r.sum().values

    for period_name, start, end in [('lag6to10',5,10),('lag8to15',7,15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = char_df.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(char_df['card_id'])
            r_end   = grp.rolling(end,   min_periods=1)
            r_start = grp.rolling(start, min_periods=1)
            lag_sum = r_end.sum().values - r_start.sum().values
            lag_cnt = np.where((r_end.count().values - r_start.count().values) <= 0, 1,
                               r_end.count().values - r_start.count().values)
            result[f'hoba_{char_name}_{period_name}_{cm}_mean'] = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_std']  = 0.0
            result[f'hoba_{char_name}_{period_name}_{cm}_max']  = lag_sum / lag_cnt
            result[f'hoba_{char_name}_{period_name}_{cm}_sum']  = lag_sum

    result[f'hoba_{char_name}_current_amt']     = char_df['Amount'].values
    result[f'hoba_{char_name}_current_util']    = char_df['utilization'].values
    result[f'hoba_{char_name}_current_timeint'] = char_df['time_interval'].values
    result = result.fillna(0)
    result.to_csv(char_path, index=False)
    print(f"  ✓ Saved. Shape: {result.shape}")
    del char_df, result; gc.collect()

NameError: name 'HOBA_DIR' is not defined

In [ ]:
import os

# ── Verify all characteristic files exist ─────────────────────
char_names = ['purchase', 'online', 'chip', 'swipe', 'abnormal_hour',
              'foreign_merchant', 'high_risk_mcc', 'error', 'high_amount']

print("Checking Drive files:")
all_present = True
for name in char_names:
    path = f'{HOBA_DIR}/{name}.csv'
    exists = os.path.exists(path)
    size = os.path.getsize(path)/1e6 if exists else 0
    print(f"  {name}: {'✓' if exists else '✗'} ({size:.1f} MB)")
    if not exists:
        all_present = False

print(f"\nAll files present: {all_present}")

Checking Drive files:


NameError: name 'HOBA_DIR' is not defined

## Section 7.10: HOBA Feature Merge (Memory Optimized)
Loading only hoba_* columns from each characteristic file to minimize
RAM usage during merge. Non-HOBA columns already present in base dataframe.
Merging one characteristic at a time, saving intermediate result to Drive.

In [ ]:
import gc, os
import pandas as pd
import numpy as np

# Check size of hoba_ columns only for each characteristic
char_names = ['purchase', 'online', 'chip', 'swipe', 'abnormal_hour',
              'foreign_merchant', 'high_risk_mcc', 'error', 'high_amount']

for name in char_names:
    path = f'{HOBA_DIR}/{name}.csv'
    sample = pd.read_csv(path, nrows=1)
    hoba_cols = [c for c in sample.columns if c.startswith('hoba_')]
    n_rows = sum(1 for _ in open(path)) - 1
    # Estimate: float32 = 4 bytes per cell
    ram_est = (n_rows * len(hoba_cols) * 4) / 1e9
    print(f"{name}: {n_rows:,} rows × {len(hoba_cols)} cols = ~{ram_est:.2f}GB")
    del sample
    gc.collect()

purchase: 1,642,252 rows × 99 cols = ~0.65GB
online: 218,420 rows × 99 cols = ~0.09GB
chip: 1,215,663 rows × 99 cols = ~0.48GB
swipe: 292,921 rows × 99 cols = ~0.12GB
abnormal_hour: 1,117,318 rows × 99 cols = ~0.44GB
foreign_merchant: 153,523 rows × 99 cols = ~0.06GB
high_risk_mcc: 17,235 rows × 99 cols = ~0.01GB
error: 27,584 rows × 99 cols = ~0.01GB
high_amount: 369,479 rows × 99 cols = ~0.15GB


In [ ]:
import gc, os
import pandas as pd
import numpy as np
os.system('pip install pyarrow -q')
import pyarrow as pa
import pyarrow.parquet as pq

n_total = len(df_flags)
char_names = ['purchase', 'online', 'chip', 'swipe', 'abnormal_hour',
              'foreign_merchant', 'high_risk_mcc', 'error', 'high_amount']

BLOCK_DIR = f'{DRIVE_PATH}/hoba_blocks'
os.makedirs(BLOCK_DIR, exist_ok=True)

# ── Step 1: build + save each characteristic's array block, freeing RAM each time ──
for name in char_names:
    block_path = f'{BLOCK_DIR}/{name}_block.parquet'
    if os.path.exists(block_path):
        print(f"✓ {name} block already saved, skipping.")
        continue

    print(f"Processing {name}...")
    path = f'{HOBA_DIR}/{name}.csv'
    sample = pd.read_csv(path, nrows=1)
    hoba_cols = [c for c in sample.columns if c.startswith('hoba_')]
    del sample

    char_df = pd.read_csv(path, usecols=['orig_index'] + hoba_cols)
    orig_idx = char_df['orig_index'].values.astype(np.int64)

    block = {}
    for col in hoba_cols:
        full_arr = np.zeros(n_total, dtype=np.float32)
        full_arr[orig_idx] = char_df[col].values.astype(np.float32)
        block[col] = full_arr
    del char_df
    gc.collect()

    block_df = pd.DataFrame(block)
    del block
    gc.collect()

    block_df.to_parquet(block_path, index=False)
    print(f"  ✓ {name} block saved: {block_df.shape}")
    del block_df
    gc.collect()

print("\nAll characteristic blocks saved to Drive.")

Processing purchase...
  ✓ purchase block saved: (1727004, 99)
Processing online...
  ✓ online block saved: (1727004, 99)
Processing chip...
  ✓ chip block saved: (1727004, 99)
Processing swipe...
  ✓ swipe block saved: (1727004, 99)
Processing abnormal_hour...
  ✓ abnormal_hour block saved: (1727004, 99)
Processing foreign_merchant...
  ✓ foreign_merchant block saved: (1727004, 99)
Processing high_risk_mcc...
  ✓ high_risk_mcc block saved: (1727004, 99)
Processing error...
  ✓ error block saved: (1727004, 99)
Processing high_amount...
  ✓ high_amount block saved: (1727004, 99)

All characteristic blocks saved to Drive.


In [ ]:
import gc, os
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

BLOCK_DIR = f'{DRIVE_PATH}/hoba_blocks'
HOBA_FINAL_PATH = f'{DRIVE_PATH}/hoba_final.parquet'

char_names = ['purchase', 'online', 'chip', 'swipe', 'abnormal_hour',
              'foreign_merchant', 'high_risk_mcc', 'error', 'high_amount']

base_cols = ['card_id', 'datetime', 'unix_time', 'is_fraud',
             'Amount', 'utilization', 'hour', 'day_of_week',
             'mcc_fraud_rate', 'has_error', 'is_online',
             'Credit Limit', 'FICO Score', 'Yearly Income - Person',
             'char_purchase', 'char_online', 'char_chip',
             'char_swipe', 'char_abnormal_hour',
             'char_foreign_merchant', 'char_high_risk_mcc',
             'char_error', 'char_high_amount']

print("Saving base columns as parquet...")
base_path = f'{BLOCK_DIR}/base_block.parquet'
if not os.path.exists(base_path):
    base = df_flags[base_cols].copy().reset_index(drop=True)
    base.to_parquet(base_path, index=False)
    del base
    gc.collect()
print("✓ Base saved.")

# ── Read all parquet files as pyarrow Tables and concat columns ──
print("\nReading all blocks as Arrow tables...")
tables = [pq.read_table(base_path)]
for name in char_names:
    t = pq.read_table(f'{BLOCK_DIR}/{name}_block.parquet')
    tables.append(t)
    print(f"  ✓ {name} loaded as Arrow table ({t.num_columns} cols)")

print("\nConcatenating columns via Arrow (zero-copy where possible)...")
combined_arrays = []
combined_names = []
for t in tables:
    for col_name in t.column_names:
        combined_arrays.append(t.column(col_name))
        combined_names.append(col_name)

final_table = pa.Table.from_arrays(combined_arrays, names=combined_names)
del tables, combined_arrays
gc.collect()

print(f"Final table: {final_table.num_rows} rows x {final_table.num_columns} cols")
print("Writing final parquet to Drive...")
pq.write_table(final_table, HOBA_FINAL_PATH)
del final_table
gc.collect()
print("✓ Saved final HOBA matrix.")

# ── Load back lightly to verify ────────────────────────────────
print("\nVerifying...")
hoba_final = pd.read_parquet(HOBA_FINAL_PATH)
print(f"Final shape: {hoba_final.shape}")
hoba_cols = [c for c in hoba_final.columns if c.startswith('hoba_')]
print(f"Total HOBA features: {len(hoba_cols)}")
nonzero_pct = (hoba_final[hoba_cols] != 0).mean().mean() * 100
print(f"Average non-zero %: {nonzero_pct:.1f}%")
print(f"Fraud rate: {hoba_final['is_fraud'].mean()*100:.4f}%")

Saving base columns as parquet...
✓ Base saved.

Reading all blocks as Arrow tables...
  ✓ purchase loaded as Arrow table (99 cols)
  ✓ online loaded as Arrow table (99 cols)
  ✓ chip loaded as Arrow table (99 cols)
  ✓ swipe loaded as Arrow table (99 cols)
  ✓ abnormal_hour loaded as Arrow table (99 cols)
  ✓ foreign_merchant loaded as Arrow table (99 cols)
  ✓ high_risk_mcc loaded as Arrow table (99 cols)
  ✓ error loaded as Arrow table (99 cols)
  ✓ high_amount loaded as Arrow table (99 cols)

Concatenating columns via Arrow (zero-copy where possible)...
Final table: 1727004 rows x 914 cols
Writing final parquet to Drive...
✓ Saved final HOBA matrix.

Verifying...


In [ ]:
import pyarrow.parquet as pq
import numpy as np

HOBA_FINAL_PATH = f'{DRIVE_PATH}/hoba_final.parquet'

# ── Lightweight verification using pyarrow metadata ────────────
pf = pq.ParquetFile(HOBA_FINAL_PATH)
print(f"Total rows: {pf.metadata.num_rows:,}")
print(f"Total columns: {pf.metadata.num_columns}")

schema = pf.schema_arrow
hoba_cols = [c for c in schema.names if c.startswith('hoba_')]
print(f"HOBA feature columns: {len(hoba_cols)}")

# ── Read only is_fraud column to check fraud rate ──────────────
fraud_only = pq.read_table(HOBA_FINAL_PATH, columns=['is_fraud']).to_pandas()
print(f"Fraud rate: {fraud_only['is_fraud'].mean()*100:.4f}%")
del fraud_only

# ── Sample a small batch to check non-zero percentage ───────────
print("\nSampling 50,000 rows to check sparsity...")
import pandas as pd
sample = pd.read_parquet(HOBA_FINAL_PATH, columns=hoba_cols[:200])  # first 200 hoba cols
sample = sample.sample(50000, random_state=42)
nonzero_pct = (sample != 0).mean().mean() * 100
print(f"Non-zero % (sample of 200 cols, 50k rows): {nonzero_pct:.1f}%")
del sample

Total rows: 1,727,004
Total columns: 914
HOBA feature columns: 891
Fraud rate: 0.6771%

Sampling 50,000 rows to check sparsity...
Non-zero % (sample of 200 cols, 50k rows): 47.2%


In [ ]:
import pyarrow.parquet as pq
import pyarrow.compute as pc
import numpy as np

HOBA_FINAL_PATH = f'{DRIVE_PATH}/hoba_final.parquet'
pf = pq.ParquetFile(HOBA_FINAL_PATH)
schema = pf.schema_arrow
hoba_cols = [c for c in schema.names if c.startswith('hoba_')]

chars = ['purchase','online','chip','swipe','abnormal_hour',
         'foreign_merchant','high_risk_mcc','error','high_amount']

print("Checking sparsity per characteristic using Arrow compute (no pandas)...")
for char in chars:
    char_cols = [c for c in hoba_cols if f'hoba_{char}_' in c]
    # Read only this characteristic's columns, only first row group for speed
    table = pf.read_row_group(0, columns=char_cols)
    n = table.num_rows
    nonzero_total = 0
    total_cells = 0
    for col in char_cols:
        arr = table.column(col)
        nz = pc.sum(pc.not_equal(arr, 0)).as_py() or 0
        nonzero_total += nz
        total_cells += n
    pct = (nonzero_total / total_cells) * 100 if total_cells else 0
    print(f"  {char}: {pct:.1f}% non-zero (row group 0, {n:,} rows, {len(char_cols)} cols)")
    del table

Checking sparsity per characteristic using Arrow compute (no pandas)...
  purchase: 83.8% non-zero (row group 0, 1,048,576 rows, 99 cols)
  online: 10.2% non-zero (row group 0, 1,048,576 rows, 99 cols)
  chip: 61.8% non-zero (row group 0, 1,048,576 rows, 99 cols)
  swipe: 14.3% non-zero (row group 0, 1,048,576 rows, 99 cols)
  abnormal_hour: 56.3% non-zero (row group 0, 1,048,576 rows, 99 cols)
  foreign_merchant: 7.2% non-zero (row group 0, 1,048,576 rows, 99 cols)
  high_risk_mcc: 0.5% non-zero (row group 0, 1,048,576 rows, 99 cols)
  error: 0.9% non-zero (row group 0, 1,048,576 rows, 99 cols)
  high_amount: 18.0% non-zero (row group 0, 1,048,576 rows, 99 cols)


## Section 8: RFM Feature Engineering (Baseline)
Computing card-level RFM features without characteristic segmentation —
the traditional approach HOBA is designed to outperform.
5 aggregation periods (1d, 3d, 5d, past 5 txns, past 15 txns) ×
3 measures (recency/time-interval, frequency, monetary) × 4 statistics
(mean, std, max, sum) = 60 variables, plus categorical features
(entry mode, online flag) = ~65-70 total RFM features.
Computed using only historical transactions (shift to exclude current).
Saved to Drive immediately.

In [ ]:
import gc, os
import pandas as pd
import numpy as np

RFM_PATH = f'{DRIVE_PATH}/rfm_final.parquet'

if os.path.exists(RFM_PATH):
    print("Loading RFM features from Drive...")
    rfm_final = pd.read_parquet(RFM_PATH)
    print(f"Loaded. Shape: {rfm_final.shape}")
else:
    print("Computing RFM features...")

    rfm_df = df_flags[['card_id', 'datetime', 'unix_time', 'Amount',
                        'utilization', 'is_fraud', 'Use Chip',
                        'is_online', 'hour', 'day_of_week']].copy()
    rfm_df = rfm_df.sort_values(['card_id', 'datetime']).reset_index(drop=True)
    rfm_df['orig_index'] = rfm_df.index

    # Time interval between sequential transactions (any type)
    rfm_df['time_interval'] = rfm_df.groupby('card_id')['unix_time'].diff().fillna(0)

    rfm_df_idx = rfm_df.set_index('datetime')

    measures = [('Amount','amt'), ('time_interval','timeint'), ('utilization','util')]
    stats = ['mean','std','max','sum']

    result = pd.DataFrame(index=rfm_df_idx.index)
    result['orig_index'] = rfm_df_idx['orig_index']

    # ── Time-based windows ──────────────────────────────────────
    for period_name, window in [('1d','1D'), ('3d','3D'), ('5d','5D')]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = rfm_df_idx.groupby('card_id')[col].shift(1)
            grp = shifted.groupby(rfm_df_idx['card_id'])
            r = grp.rolling(window, min_periods=1)
            result[f'rfm_{period_name}_{cm}_mean'] = r.mean().values
            result[f'rfm_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'rfm_{period_name}_{cm}_max']  = r.max().values
            result[f'rfm_{period_name}_{cm}_sum']  = r.sum().values

    # ── Count-based windows ─────────────────────────────────────
    for period_name, cnt in [('cnt5',5), ('cnt15',15)]:
        print(f"  {period_name}...")
        for col, cm in measures:
            shifted = rfm_df_idx.groupby('card_id')[col].shift(1)
            r = shifted.groupby(rfm_df_idx['card_id']).rolling(cnt, min_periods=1)
            result[f'rfm_{period_name}_{cm}_mean'] = r.mean().values
            result[f'rfm_{period_name}_{cm}_std']  = r.std().fillna(0).values
            result[f'rfm_{period_name}_{cm}_max']  = r.max().values
            result[f'rfm_{period_name}_{cm}_sum']  = r.sum().values

    # ── Categorical / current-snapshot features ─────────────────
    result['rfm_current_amt']  = rfm_df_idx['Amount'].values
    result['rfm_current_util'] = rfm_df_idx['utilization'].values
    result['rfm_is_online']    = rfm_df_idx['is_online'].values
    result['rfm_hour']         = rfm_df_idx['hour'].values
    result['rfm_day_of_week']  = rfm_df_idx['day_of_week'].values

    result = result.reset_index(drop=True).fillna(0)

    # ── Attach identity columns ──────────────────────────────────
    rfm_final = rfm_df[['card_id','datetime','is_fraud']].reset_index(drop=True)
    rfm_final = pd.concat([rfm_final, result.drop(columns=['orig_index'])], axis=1)

    print("Saving to Drive...")
    rfm_final.to_parquet(RFM_PATH, index=False)
    print(f"✓ Saved. Shape: {rfm_final.shape}")

    del rfm_df, rfm_df_idx, result
    gc.collect()

rfm_cols = [c for c in rfm_final.columns if c.startswith('rfm_')]
print(f"\nTotal RFM features: {len(rfm_cols)}")
print(f"Fraud rate: {rfm_final['is_fraud'].mean()*100:.4f}%")
print(f"Shape: {rfm_final.shape}")

Computing RFM features...
  1d...
  3d...
  5d...
  cnt5...
  cnt15...
Saving to Drive...
✓ Saved. Shape: (1727004, 68)

Total RFM features: 65
Fraud rate: 0.6771%
Shape: (1727004, 68)


## Section 9: Train/Test Split
Dropping 2015 rows (used only as behavioral history for HOBA/RFM rolling
windows during feature engineering). Train = 2016-2018, Test = 2019.
Split applied identically to both HOBA and RFM feature sets, joined on
card_id + datetime to keep them aligned. Saved to Drive as parquet.

In [ ]:
import gc, os
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

HOBA_FINAL_PATH = f'{DRIVE_PATH}/hoba_final.parquet'
TRAIN_HOBA_PATH = f'{DRIVE_PATH}/train_hoba.parquet'
TEST_HOBA_PATH  = f'{DRIVE_PATH}/test_hoba.parquet'

if os.path.exists(TRAIN_HOBA_PATH) and os.path.exists(TEST_HOBA_PATH):
    print("HOBA splits already on Drive.")
else:
    pf = pq.ParquetFile(HOBA_FINAL_PATH)
    all_cols = pf.schema_arrow.names

    # ── Determine train/test row masks from datetime only ───────
    print("Computing row masks from datetime...")
    dt = pd.read_parquet(HOBA_FINAL_PATH, columns=['datetime'])['datetime']
    years = pd.to_datetime(dt).dt.year
    train_mask = years.between(2016, 2018).values
    test_mask  = (years == 2019).values
    del dt, years
    gc.collect()
    print(f"Train rows: {train_mask.sum():,} | Test rows: {test_mask.sum():,}")

    # ── Process columns in batches of 150 ────────────────────────
    BATCH_COLS = 150
    col_batches = [all_cols[i:i+BATCH_COLS] for i in range(0, len(all_cols), BATCH_COLS)]
    print(f"Processing {len(all_cols)} columns in {len(col_batches)} batches...")

    train_parts, test_parts = [], []
    for i, cols in enumerate(col_batches):
        print(f"  Batch {i+1}/{len(col_batches)} ({len(cols)} cols)...")
        chunk = pd.read_parquet(HOBA_FINAL_PATH, columns=cols)
        train_parts.append(chunk[train_mask].reset_index(drop=True))
        test_parts.append(chunk[test_mask].reset_index(drop=True))
        del chunk
        gc.collect()

    print("\nAssembling train HOBA...")
    train_hoba = pd.concat(train_parts, axis=1)
    del train_parts
    gc.collect()
    print(f"Train shape: {train_hoba.shape}")
    train_hoba.to_parquet(TRAIN_HOBA_PATH, index=False)
    print("✓ Train saved.")
    del train_hoba
    gc.collect()

    print("\nAssembling test HOBA...")
    test_hoba = pd.concat(test_parts, axis=1)
    del test_parts
    gc.collect()
    print(f"Test shape: {test_hoba.shape}")
    test_hoba.to_parquet(TEST_HOBA_PATH, index=False)
    print("✓ Test saved.")
    del test_hoba
    gc.collect()

print("\n✅ HOBA train/test split complete.")

Computing row masks from datetime...
Train rows: 1,035,957 | Test rows: 346,537
Processing 914 columns in 7 batches...
  Batch 1/7 (150 cols)...
  Batch 2/7 (150 cols)...
  Batch 3/7 (150 cols)...
  Batch 4/7 (150 cols)...
  Batch 5/7 (150 cols)...
  Batch 6/7 (150 cols)...
  Batch 7/7 (14 cols)...

Assembling train HOBA...
Train shape: (1035957, 914)
✓ Train saved.

Assembling test HOBA...
Test shape: (346537, 914)
✓ Test saved.

✅ HOBA train/test split complete.


In [ ]:
import gc

# Delete any large objects that might still be lingering in this session
for var_name in ['hoba_final', 'df_flags', 'df_clean', 'df_joined', 'df',
                  'txn', 'train_hoba', 'test_hoba', 'rfm_final',
                  'train_parts', 'test_parts', 'sample', 'chunk',
                  'all_arrays', 'char_dfs', 'merged']:
    if var_name in dir():
        try:
            exec(f'del {var_name}')
        except:
            pass

gc.collect()
gc.collect()  # run twice — clears reference cycles

import psutil
mem = psutil.virtual_memory()
print(f"RAM used: {mem.percent}%")
print(f"RAM available: {mem.available/1e9:.2f} GB")

RAM used: 35.6%
RAM available: 8.76 GB


## Section 9.1: RFM Train/Test Split
Splitting RFM features using the same column-batched approach for safety,
though RFM's smaller width (68 cols) makes this lower risk than HOBA.

In [ ]:
import gc, os
import pandas as pd
import pyarrow.parquet as pq

RFM_PATH       = f'{DRIVE_PATH}/rfm_final.parquet'
TRAIN_RFM_PATH = f'{DRIVE_PATH}/train_rfm.parquet'
TEST_RFM_PATH  = f'{DRIVE_PATH}/test_rfm.parquet'

if os.path.exists(TRAIN_RFM_PATH) and os.path.exists(TEST_RFM_PATH):
    print("RFM splits already on Drive.")
else:
    print("Loading RFM (small enough for single read)...")
    rfm_final = pd.read_parquet(RFM_PATH)
    rfm_final['datetime'] = pd.to_datetime(rfm_final['datetime'])
    years = rfm_final['datetime'].dt.year

    train_mask = years.between(2016, 2018).values
    test_mask  = (years == 2019).values

    train_rfm = rfm_final[train_mask].reset_index(drop=True)
    test_rfm  = rfm_final[test_mask].reset_index(drop=True)

    print(f"Train RFM: {train_rfm.shape} | Fraud: {train_rfm['is_fraud'].sum()}")
    print(f"Test RFM:  {test_rfm.shape} | Fraud: {test_rfm['is_fraud'].sum()}")

    train_rfm.to_parquet(TRAIN_RFM_PATH, index=False)
    test_rfm.to_parquet(TEST_RFM_PATH, index=False)
    print("✓ RFM splits saved.")

    del rfm_final, train_rfm, test_rfm
    gc.collect()

print("\n✅ RFM train/test split complete.")

Loading RFM (small enough for single read)...
Train RFM: (1035957, 68) | Fraud: 6325
Test RFM:  (346537, 68) | Fraud: 2087
✓ RFM splits saved.

✅ RFM train/test split complete.


## Master Reload Cell — Run Once Per Session
Loads train/test HOBA and RFM parquet files, converts to numpy arrays
immediately after scaling, and deletes the source dataframes to minimize
RAM footprint going into modeling. This is the only cell needed at the
start of every new session — all feature engineering is already complete
and saved to Drive.

In [ ]:
import pandas as pd
import numpy as np
import gc

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'

# Get column list without loading data
sample = pd.read_parquet(f'{DRIVE_PATH}/train_hoba.parquet', columns=None).iloc[:0]
all_cols = list(sample.columns)
hoba_cols = [c for c in all_cols if c.startswith('hoba_')]
del sample
print(f"Total HOBA columns: {len(hoba_cols)}")

# Check in batches of 100 columns
BATCH = 100
problem_cols = []

for i in range(0, len(hoba_cols), BATCH):
    batch_cols = hoba_cols[i:i+BATCH]
    chunk = pd.read_parquet(f'{DRIVE_PATH}/train_hoba.parquet', columns=batch_cols)
    arr = chunk.to_numpy(dtype=np.float32)

    inf_per_col = np.isinf(arr).sum(axis=0)
    nan_per_col = np.isnan(arr).sum(axis=0)

    for j, col in enumerate(batch_cols):
        if inf_per_col[j] > 0 or nan_per_col[j] > 0:
            problem_cols.append((col, int(inf_per_col[j]), int(nan_per_col[j])))

    del chunk, arr
    gc.collect()
    print(f"  Checked columns {i}-{i+len(batch_cols)}, problems so far: {len(problem_cols)}")

print(f"\nTotal columns with inf/nan: {len(problem_cols)}")
for col, n_inf, n_nan in sorted(problem_cols, key=lambda x: -x[1])[:30]:
    print(f"  {col}: inf={n_inf}, nan={n_nan}")

Total HOBA columns: 891
  Checked columns 0-100, problems so far: 1
  Checked columns 100-200, problems so far: 2
  Checked columns 200-300, problems so far: 3
  Checked columns 300-400, problems so far: 4
  Checked columns 400-500, problems so far: 5
  Checked columns 500-600, problems so far: 6
  Checked columns 600-700, problems so far: 7
  Checked columns 700-800, problems so far: 8
  Checked columns 800-891, problems so far: 8

Total columns with inf/nan: 8
  hoba_purchase_current_util: inf=4789, nan=0
  hoba_chip_current_util: inf=3965, nan=0
  hoba_abnormal_hour_current_util: inf=3058, nan=0
  hoba_swipe_current_util: inf=633, nan=0
  hoba_online_current_util: inf=399, nan=0
  hoba_foreign_merchant_current_util: inf=383, nan=0
  hoba_error_current_util: inf=55, nan=0
  hoba_high_risk_mcc_current_util: inf=42, nan=0


In [ ]:
import pandas as pd
import numpy as np
import gc

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'

problem_cols = [
    'hoba_purchase_current_util', 'hoba_chip_current_util',
    'hoba_abnormal_hour_current_util', 'hoba_swipe_current_util',
    'hoba_online_current_util', 'hoba_foreign_merchant_current_util',
    'hoba_error_current_util', 'hoba_high_risk_mcc_current_util'
]

for split_name in ['train', 'test']:
    path = f'{DRIVE_PATH}/{split_name}_hoba.parquet'
    print(f"Fixing {split_name}...")

    # Load only the problem columns first to confirm
    chunk = pd.read_parquet(path, columns=problem_cols)
    print(f"  Before fix — inf counts: {np.isinf(chunk.values).sum()}")

    chunk_fixed = chunk.clip(lower=-1, upper=5)
    print(f"  After fix — inf counts: {np.isinf(chunk_fixed.values).sum()}")

    # Load full file, replace these columns, save back
    full = pd.read_parquet(path)
    full[problem_cols] = chunk_fixed
    full.to_parquet(path, index=False)
    print(f"  ✓ {split_name}_hoba.parquet updated and saved.")

    del chunk, chunk_fixed, full
    gc.collect()

print("\n✅ Both train and test HOBA files fixed and re-saved.")

Fixing train...
  Before fix — inf counts: 13324
  After fix — inf counts: 0
  ✓ train_hoba.parquet updated and saved.
Fixing test...
  Before fix — inf counts: 4138
  After fix — inf counts: 0
  ✓ test_hoba.parquet updated and saved.

✅ Both train and test HOBA files fixed and re-saved.


In [ ]:
import os
import pyarrow.parquet as pq

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
BLOCK_DIR = f'{DRIVE_PATH}/hoba_blocks'

char_names = ['purchase', 'online', 'chip', 'swipe', 'abnormal_hour',
              'foreign_merchant', 'high_risk_mcc', 'error', 'high_amount']

for name in char_names:
    path = f'{BLOCK_DIR}/{name}_block.parquet'
    if os.path.exists(path):
        try:
            pf = pq.ParquetFile(path)
            print(f"✓ {name}_block.parquet: {pf.metadata.num_rows} rows, {pf.metadata.num_columns} cols")
        except Exception as e:
            print(f"✗ {name}_block.parquet CORRUPTED: {e}")
    else:
        print(f"✗ {name}_block.parquet MISSING")

# Also check base_block
base_path = f'{BLOCK_DIR}/base_block.parquet'
if os.path.exists(base_path):
    try:
        pf = pq.ParquetFile(base_path)
        print(f"✓ base_block.parquet: {pf.metadata.num_rows} rows, {pf.metadata.num_columns} cols")
    except Exception as e:
        print(f"✗ base_block.parquet CORRUPTED: {e}")

✓ purchase_block.parquet: 1727004 rows, 99 cols
✓ online_block.parquet: 1727004 rows, 99 cols
✓ chip_block.parquet: 1727004 rows, 99 cols
✓ swipe_block.parquet: 1727004 rows, 99 cols
✓ abnormal_hour_block.parquet: 1727004 rows, 99 cols
✓ foreign_merchant_block.parquet: 1727004 rows, 99 cols
✓ high_risk_mcc_block.parquet: 1727004 rows, 99 cols
✓ error_block.parquet: 1727004 rows, 99 cols
✓ high_amount_block.parquet: 1727004 rows, 99 cols
✓ base_block.parquet: 1727004 rows, 23 cols


In [ ]:
import gc, os
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
BLOCK_DIR = f'{DRIVE_PATH}/hoba_blocks'
HOBA_FINAL_PATH_NEW = f'{DRIVE_PATH}/hoba_final_v2.parquet'

char_names = ['purchase', 'online', 'chip', 'swipe', 'abnormal_hour',
              'foreign_merchant', 'high_risk_mcc', 'error', 'high_amount']

print("Loading and fixing each block, writing to new parquet one at a time...")
tables = []

# Base first
base_path = f'{BLOCK_DIR}/base_block.parquet'
t = pq.read_table(base_path)
tables.append(t)
print(f"  ✓ base loaded ({t.num_columns} cols)")

for name in char_names:
    path = f'{BLOCK_DIR}/{name}_block.parquet'
    df = pd.read_parquet(path)

    # Fix the current_util column for this characteristic if it has inf
    util_col = f'hoba_{name}_current_util'
    if util_col in df.columns:
        n_inf_before = np.isinf(df[util_col].values).sum()
        if n_inf_before > 0:
            df[util_col] = df[util_col].clip(lower=-1, upper=5)
            print(f"  Fixed {util_col}: {n_inf_before} inf values clipped")

    t = pa.Table.from_pandas(df)
    tables.append(t)
    del df
    gc.collect()
    print(f"  ✓ {name} loaded and fixed ({t.num_columns} cols)")

print("\nConcatenating all column blocks...")
combined_arrays = []
combined_names = []
for t in tables:
    for col_name in t.column_names:
        combined_arrays.append(t.column(col_name))
        combined_names.append(col_name)

final_table = pa.Table.from_arrays(combined_arrays, names=combined_names)
del tables, combined_arrays
gc.collect()

print(f"Final table: {final_table.num_rows} rows x {final_table.num_columns} cols")
print("Writing to Drive...")
pq.write_table(final_table, HOBA_FINAL_PATH_NEW)
del final_table
gc.collect()
print("✓ Saved hoba_final_v2.parquet")

Loading and fixing each block, writing to new parquet one at a time...
  ✓ base loaded (23 cols)
  Fixed hoba_purchase_current_util: 7722 inf values clipped
  ✓ purchase loaded and fixed (99 cols)
  Fixed hoba_online_current_util: 681 inf values clipped
  ✓ online loaded and fixed (99 cols)
  Fixed hoba_chip_current_util: 6366 inf values clipped
  ✓ chip loaded and fixed (99 cols)
  Fixed hoba_swipe_current_util: 1018 inf values clipped
  ✓ swipe loaded and fixed (99 cols)
  Fixed hoba_abnormal_hour_current_util: 4905 inf values clipped
  ✓ abnormal_hour loaded and fixed (99 cols)
  Fixed hoba_foreign_merchant_current_util: 629 inf values clipped
  ✓ foreign_merchant loaded and fixed (99 cols)
  Fixed hoba_high_risk_mcc_current_util: 84 inf values clipped
  ✓ high_risk_mcc loaded and fixed (99 cols)
  Fixed hoba_error_current_util: 82 inf values clipped
  ✓ error loaded and fixed (99 cols)
  ✓ high_amount loaded and fixed (99 cols)

Concatenating all column blocks...
Final table: 17270

In [ ]:
import gc, os
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pyarrow.compute as pc
import pandas as pd
import numpy as np

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
HOBA_FINAL_PATH = f'{DRIVE_PATH}/hoba_final_v2.parquet'
TRAIN_HOBA_PATH = f'{DRIVE_PATH}/train_hoba_v3.parquet'
TEST_HOBA_PATH  = f'{DRIVE_PATH}/test_hoba_v3.parquet'

# ── Step 1: compute year as a precomputed boolean array (lightweight) ──
print("Computing year column...")
dt = pd.read_parquet(HOBA_FINAL_PATH, columns=['datetime'])['datetime']
years = pd.to_datetime(dt).dt.year
train_mask_np = years.between(2016, 2018).values
test_mask_np  = (years == 2019).values
del dt, years
gc.collect()
print(f"Train rows: {train_mask_np.sum():,} | Test rows: {test_mask_np.sum():,}")

# We need a row-index-aligned way to filter via the dataset scanner.
# Since the boolean mask is computed in pandas order (file row order),
# we can write it as a temporary index array and use it per-batch.

dataset = ds.dataset(HOBA_FINAL_PATH, format='parquet')

def stream_filtered_write(mask_array, out_path, label):
    print(f"\nStreaming {label}...")
    writer = None
    row_offset = 0
    scanner = dataset.scanner(batch_size=100_000)
    for batch in scanner.to_batches():
        n = batch.num_rows
        batch_mask = mask_array[row_offset: row_offset + n]
        row_offset += n
        if batch_mask.any():
            filtered = batch.filter(pa.array(batch_mask))
            if writer is None:
                writer = pq.ParquetWriter(out_path, filtered.schema)
            writer.write_table(pa.Table.from_batches([filtered]))
        print(f"  processed {row_offset:,} rows so far...", end='\r')
    if writer:
        writer.close()
    print(f"\n  ✓ {label} written to {out_path}")

stream_filtered_write(train_mask_np, TRAIN_HOBA_PATH, "TRAIN")
gc.collect()
stream_filtered_write(test_mask_np, TEST_HOBA_PATH, "TEST")
gc.collect()

print("\n✅ Streaming split complete — verifying...")
pf_train = pq.ParquetFile(TRAIN_HOBA_PATH)
pf_test  = pq.ParquetFile(TEST_HOBA_PATH)
print(f"Train: {pf_train.metadata.num_rows} rows, {pf_train.metadata.num_columns} cols")
print(f"Test:  {pf_test.metadata.num_rows} rows, {pf_test.metadata.num_columns} cols")

Computing year column...
Train rows: 1,035,957 | Test rows: 346,537

Streaming TRAIN...

  ✓ TRAIN written to /content/drive/MyDrive/HOBA_Fraud_Detection_V2/train_hoba_v3.parquet

Streaming TEST...
  processed 1,727,004 rows so far...
  ✓ TEST written to /content/drive/MyDrive/HOBA_Fraud_Detection_V2/test_hoba_v3.parquet

✅ Streaming split complete — verifying...
Train: 1035957 rows, 914 cols
Test:  346537 rows, 914 cols


In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import gc

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
TRAIN_HOBA_PATH = f'{DRIVE_PATH}/train_hoba_v3.parquet'

pf = pq.ParquetFile(TRAIN_HOBA_PATH)
hoba_cols = [c for c in pf.schema_arrow.names if c.startswith('hoba_')]

BATCH = 150
total_inf = 0
for i in range(0, len(hoba_cols), BATCH):
    batch_cols = hoba_cols[i:i+BATCH]
    chunk = pd.read_parquet(TRAIN_HOBA_PATH, columns=batch_cols)
    n_inf = np.isinf(chunk.to_numpy(dtype=np.float32)).sum()
    total_inf += n_inf
    del chunk
    gc.collect()

print(f"Total inf values across all HOBA columns in train_hoba_v3: {total_inf}")

Total inf values across all HOBA columns in train_hoba_v3: 0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import psutil
print(f"RAM after restart: {psutil.virtual_memory().percent}%")
print(f"RAM available: {psutil.virtual_memory().available/1e9:.2f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RAM after restart: 11.6%
RAM available: 12.03 GB


In [ ]:
import gc, os
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
import psutil

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
TRAIN_HOBA_PATH = f'{DRIVE_PATH}/train_hoba_v3.parquet'

# ── Pass 1: compute mean/std per HOBA column using running sums (no full load) ──
pf = pq.ParquetFile(TRAIN_HOBA_PATH)
hoba_cols = [c for c in pf.schema_arrow.names if c.startswith('hoba_')]
print(f"HOBA columns: {len(hoba_cols)}")

n_total = 0
sum_ = np.zeros(len(hoba_cols), dtype=np.float64)
sumsq = np.zeros(len(hoba_cols), dtype=np.float64)

print("Pass 1: computing mean/std via streaming batches...")
for batch in pf.iter_batches(batch_size=100_000, columns=hoba_cols):
    arr = np.column_stack([batch.column(c).to_numpy(zero_copy_only=False) for c in hoba_cols]).astype(np.float64)
    n_total += arr.shape[0]
    sum_ += arr.sum(axis=0)
    sumsq += (arr ** 2).sum(axis=0)
    del arr
    gc.collect()
    print(f"  processed {n_total:,} rows...", end='\r')

mean_ = sum_ / n_total
std_  = np.sqrt(sumsq / n_total - mean_**2)
std_[std_ == 0] = 1.0  # avoid div by zero for constant columns

print(f"\n✓ Mean/std computed for {len(hoba_cols)} columns.")
print(f"RAM: {psutil.virtual_memory().percent}%")

np.save(f'{DRIVE_PATH}/hoba_train_mean.npy', mean_)
np.save(f'{DRIVE_PATH}/hoba_train_std.npy', std_)
print("✓ Saved scaler stats to Drive.")

HOBA columns: 891
Pass 1: computing mean/std via streaming batches...
  processed 1,035,957 rows...
✓ Mean/std computed for 891 columns.
RAM: 23.2%
✓ Saved scaler stats to Drive.


In [ ]:
import gc, os
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
import psutil

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
TRAIN_HOBA_PATH = f'{DRIVE_PATH}/train_hoba_v3.parquet'
TRAIN_SCALED_PATH = f'{DRIVE_PATH}/train_hoba_scaled.parquet'

mean_ = np.load(f'{DRIVE_PATH}/hoba_train_mean.npy')
std_  = np.load(f'{DRIVE_PATH}/hoba_train_std.npy')

pf = pq.ParquetFile(TRAIN_HOBA_PATH)
hoba_cols = [c for c in pf.schema_arrow.names if c.startswith('hoba_')]
other_cols = ['is_fraud']  # keep label alongside scaled features

print("Pass 2: writing scaled train data, streaming...")
writer = None
n_done = 0
for batch in pf.iter_batches(batch_size=100_000, columns=hoba_cols + other_cols):
    arr = np.column_stack([batch.column(c).to_numpy(zero_copy_only=False) for c in hoba_cols]).astype(np.float32)
    scaled = (arr - mean_.astype(np.float32)) / std_.astype(np.float32)
    del arr

    y_chunk = batch.column('is_fraud').to_numpy(zero_copy_only=False)

    out_table = pa.table(
        {**{c: scaled[:, i] for i, c in enumerate(hoba_cols)}, 'is_fraud': y_chunk}
    )
    if writer is None:
        writer = pq.ParquetWriter(TRAIN_SCALED_PATH, out_table.schema)
    writer.write_table(out_table)

    n_done += batch.num_rows
    del scaled, out_table
    gc.collect()
    print(f"  written {n_done:,} rows...", end='\r')

if writer:
    writer.close()
print(f"\n✓ Scaled train saved to {TRAIN_SCALED_PATH}")
print(f"RAM: {psutil.virtual_memory().percent}%")

Pass 2: writing scaled train data, streaming...
  written 1,035,957 rows...
✓ Scaled train saved to /content/drive/MyDrive/HOBA_Fraud_Detection_V2/train_hoba_scaled.parquet
RAM: 32.1%


In [ ]:
import gc, os
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
import psutil

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
TEST_HOBA_PATH = f'{DRIVE_PATH}/test_hoba_v3.parquet'
TEST_SCALED_PATH = f'{DRIVE_PATH}/test_hoba_scaled.parquet'

mean_ = np.load(f'{DRIVE_PATH}/hoba_train_mean.npy')
std_  = np.load(f'{DRIVE_PATH}/hoba_train_std.npy')

pf = pq.ParquetFile(TEST_HOBA_PATH)
hoba_cols = [c for c in pf.schema_arrow.names if c.startswith('hoba_')]
other_cols = ['is_fraud']

print("Streaming test scaling...")
writer = None
n_done = 0
for batch in pf.iter_batches(batch_size=100_000, columns=hoba_cols + other_cols):
    arr = np.column_stack([batch.column(c).to_numpy(zero_copy_only=False) for c in hoba_cols]).astype(np.float32)
    scaled = (arr - mean_.astype(np.float32)) / std_.astype(np.float32)
    del arr

    y_chunk = batch.column('is_fraud').to_numpy(zero_copy_only=False)

    out_table = pa.table(
        {**{c: scaled[:, i] for i, c in enumerate(hoba_cols)}, 'is_fraud': y_chunk}
    )
    if writer is None:
        writer = pq.ParquetWriter(TEST_SCALED_PATH, out_table.schema)
    writer.write_table(out_table)

    n_done += batch.num_rows
    del scaled, out_table
    gc.collect()
    print(f"  written {n_done:,} rows...", end='\r')

if writer:
    writer.close()
print(f"\n✓ Scaled test saved to {TEST_SCALED_PATH}")
print(f"RAM: {psutil.virtual_memory().percent}%")

Streaming test scaling...
  written 346,537 rows...
✓ Scaled test saved to /content/drive/MyDrive/HOBA_Fraud_Detection_V2/test_hoba_scaled.parquet
RAM: 31.6%


In [ ]:
import gc, os
import numpy as np
import pyarrow.parquet as pq
import psutil

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
TRAIN_SCALED_PATH = f'{DRIVE_PATH}/train_hoba_scaled.parquet'

pf = pq.ParquetFile(TRAIN_SCALED_PATH)
hoba_cols = [c for c in pf.schema_arrow.names if c.startswith('hoba_')]
n_rows = pf.metadata.num_rows
n_cols = len(hoba_cols)
print(f"Target array: {n_rows:,} x {n_cols} = {n_rows*n_cols*4/1e9:.2f} GB (float32)")

# Pre-allocate the full array ONCE — no repeated copies
X_train_hoba = np.empty((n_rows, n_cols), dtype=np.float32)
y_train = np.empty(n_rows, dtype=np.int8)

row_offset = 0
for batch in pf.iter_batches(batch_size=100_000, columns=hoba_cols + ['is_fraud']):
    n = batch.num_rows
    for i, c in enumerate(hoba_cols):
        X_train_hoba[row_offset:row_offset+n, i] = batch.column(c).to_numpy(zero_copy_only=False)
    y_train[row_offset:row_offset+n] = batch.column('is_fraud').to_numpy(zero_copy_only=False)
    row_offset += n
    del batch
    gc.collect()
    print(f"  filled {row_offset:,}/{n_rows:,} rows... RAM: {psutil.virtual_memory().percent}%", end='\r')

print(f"\n✓ X_train_hoba shape: {X_train_hoba.shape}, dtype: {X_train_hoba.dtype}")
print(f"Final RAM: {psutil.virtual_memory().percent}%")

Target array: 1,035,957 x 891 = 3.69 GB (float32)
  filled 1,035,957/1,035,957 rows... RAM: 56.7%
✓ X_train_hoba shape: (1035957, 891), dtype: float32
Final RAM: 56.7%


In [ ]:
import gc
import numpy as np
import pyarrow.parquet as pq
import psutil

TEST_SCALED_PATH = f'{DRIVE_PATH}/test_hoba_scaled.parquet'

pf = pq.ParquetFile(TEST_SCALED_PATH)
n_rows_test = pf.metadata.num_rows
print(f"Target test array: {n_rows_test:,} x {n_cols} = {n_rows_test*n_cols*4/1e9:.2f} GB")

X_test_hoba = np.empty((n_rows_test, n_cols), dtype=np.float32)
y_test = np.empty(n_rows_test, dtype=np.int8)

row_offset = 0
for batch in pf.iter_batches(batch_size=100_000, columns=hoba_cols + ['is_fraud']):
    n = batch.num_rows
    for i, c in enumerate(hoba_cols):
        X_test_hoba[row_offset:row_offset+n, i] = batch.column(c).to_numpy(zero_copy_only=False)
    y_test[row_offset:row_offset+n] = batch.column('is_fraud').to_numpy(zero_copy_only=False)
    row_offset += n
    del batch
    gc.collect()
    print(f"  filled {row_offset:,}/{n_rows_test:,} rows... RAM: {psutil.virtual_memory().percent}%", end='\r')

print(f"\n✓ X_test_hoba shape: {X_test_hoba.shape}, dtype: {X_test_hoba.dtype}")
print(f"Final RAM: {psutil.virtual_memory().percent}%")

Target test array: 346,537 x 891 = 1.24 GB
  filled 346,537/346,537 rows... RAM: 67.3%
✓ X_test_hoba shape: (346537, 891), dtype: float32
Final RAM: 67.3%


In [ ]:
import numpy as np
import pandas as pd

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
TRAIN_RFM_PATH = f'{DRIVE_PATH}/train_rfm.parquet'

train_rfm_df = pd.read_parquet(TRAIN_RFM_PATH)
rfm_cols = [c for c in train_rfm_df.columns if c.startswith('rfm_')]

arr = train_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
inf_per_col = np.isinf(arr).sum(axis=0)
nan_per_col = np.isnan(arr).sum(axis=0)

problem_cols = [(rfm_cols[i], int(inf_per_col[i]), int(nan_per_col[i]))
                for i in range(len(rfm_cols))
                if inf_per_col[i] > 0 or nan_per_col[i] > 0]

print(f"Columns with inf/nan: {len(problem_cols)}")
for col, n_inf, n_nan in problem_cols:
    print(f"  {col}: inf={n_inf}, nan={n_nan}")

del arr

Columns with inf/nan: 1
  rfm_current_util: inf=4997, nan=0


In [ ]:
import numpy as np
import pandas as pd

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'

for split in ['train', 'test']:
    path = f'{DRIVE_PATH}/{split}_rfm.parquet'
    df = pd.read_parquet(path)
    n_inf_before = np.isinf(df['rfm_current_util'].to_numpy(dtype=np.float32)).sum()
    df['rfm_current_util'] = df['rfm_current_util'].clip(lower=-1, upper=5)
    n_inf_after = np.isinf(df['rfm_current_util'].to_numpy(dtype=np.float32)).sum()
    print(f"{split}: before={n_inf_before}, after={n_inf_after}")
    df.to_parquet(path, index=False)
    del df

print("✓ Both RFM files fixed and saved.")

train: before=4997, after=0
test: before=1553, after=0
✓ Both RFM files fixed and saved.


# MODEL requirements


In [ ]:
import gc
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import psutil

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'

print("Loading RFM...")
train_rfm_df = pd.read_parquet(f'{DRIVE_PATH}/train_rfm.parquet')
test_rfm_df  = pd.read_parquet(f'{DRIVE_PATH}/test_rfm.parquet')

rfm_cols = [c for c in train_rfm_df.columns if c.startswith('rfm_')]
print(f"RFM columns: {len(rfm_cols)}")

X_train_rfm_raw = train_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
X_test_rfm_raw  = test_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
del train_rfm_df, test_rfm_df
gc.collect()

scaler_rfm = StandardScaler()
X_train_rfm = scaler_rfm.fit_transform(X_train_rfm_raw)
X_test_rfm  = scaler_rfm.transform(X_test_rfm_raw)
del X_train_rfm_raw, X_test_rfm_raw
gc.collect()

print(f"X_train_rfm: {X_train_rfm.shape} | X_test_rfm: {X_test_rfm.shape}")
print(f"inf check: train={np.isinf(X_train_rfm).sum()}, test={np.isinf(X_test_rfm).sum()}")
print(f"RAM: {psutil.virtual_memory().percent}%")

Loading RFM...
RFM columns: 65
X_train_rfm: (1035957, 65) | X_test_rfm: (346537, 65)
inf check: train=0, test=0
RAM: 22.4%


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import psutil

print("Training Random Forest on RFM features...")
rf_rfm = RandomForestClassifier(
    n_estimators=100, max_depth=10,
    class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf_rfm.fit(X_train_rfm, y_train)
print(f"RAM after fit: {psutil.virtual_memory().percent}%")

y_pred = rf_rfm.predict(X_test_rfm)
y_prob = rf_rfm.predict_proba(X_test_rfm)[:, 1]
evaluate_model("Random Forest — RFM", y_test, y_pred, y_prob)

print(f"\nRAM after predict: {psutil.virtual_memory().percent}%")

Training Random Forest on RFM features...
RAM after fit: 73.3%

=== Random Forest — RFM ===
AUC: 0.9513 | F1: 0.2881 | Precision: 0.1805 | Recall: 0.7139
Recall @ 1% FPR: 0.6349 | Recall @ 3% FPR: 0.7604
Result saved to Drive.

RAM after predict: 73.4%


In [ ]:
import gc

# We don't need the trained RF-RFM model object anymore — result is saved
if 'rf_rfm' in dir():
    del rf_rfm
if 'y_pred' in dir():
    del y_pred
if 'y_prob' in dir():
    del y_prob

gc.collect()
gc.collect()

import psutil
print(f"RAM used: {psutil.virtual_memory().percent}%")
print(f"RAM available: {psutil.virtual_memory().available/1e9:.2f} GB")

RAM used: 18.2%
RAM available: 11.13 GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc, os, json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, roc_curve)
from sklearn.utils.class_weight import compute_class_weight
import psutil

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
print(f"RAM: {psutil.virtual_memory().percent}%")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RAM: 18.1%


In [ ]:
TRAIN_SCALED_PATH = f'{DRIVE_PATH}/train_hoba_scaled.parquet'

pf = pq.ParquetFile(TRAIN_SCALED_PATH)
hoba_cols = [c for c in pf.schema_arrow.names if c.startswith('hoba_')]
n_rows = pf.metadata.num_rows
n_cols = len(hoba_cols)
print(f"Target array: {n_rows:,} x {n_cols} = {n_rows*n_cols*4/1e9:.2f} GB")

X_train_hoba = np.empty((n_rows, n_cols), dtype=np.float32)
y_train = np.empty(n_rows, dtype=np.int8)

row_offset = 0
for batch in pf.iter_batches(batch_size=100_000, columns=hoba_cols + ['is_fraud']):
    n = batch.num_rows
    for i, c in enumerate(hoba_cols):
        X_train_hoba[row_offset:row_offset+n, i] = batch.column(c).to_numpy(zero_copy_only=False)
    y_train[row_offset:row_offset+n] = batch.column('is_fraud').to_numpy(zero_copy_only=False)
    row_offset += n
    del batch
    gc.collect()

print(f"✓ X_train_hoba: {X_train_hoba.shape}")
print(f"RAM: {psutil.virtual_memory().percent}%")

Target array: 1,035,957 x 891 = 3.69 GB
✓ X_train_hoba: (1035957, 891)
RAM: 54.6%


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve
import psutil

# Re-define eval helpers (lost on restart)
def fpr_constrained_recall(y_true, y_prob, fpr_threshold):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    valid_idx = np.where(fpr <= fpr_threshold)[0]
    if len(valid_idx) == 0:
        return 0.0
    return tpr[valid_idx[np.argmax(tpr[valid_idx])]]

def evaluate_model(name, y_true, y_pred, y_prob):
    auc = roc_auc_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    r1fpr = fpr_constrained_recall(y_true, y_prob, 0.01)
    r3fpr = fpr_constrained_recall(y_true, y_prob, 0.03)
    print(f"\n=== {name} ===")
    print(f"AUC: {auc:.4f} | F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
    print(f"Recall @ 1% FPR: {r1fpr:.4f} | Recall @ 3% FPR: {r3fpr:.4f}")
    result = {'name': name, 'auc': auc, 'f1': f1, 'precision': precision,
              'recall': recall, 'recall_1fpr': r1fpr, 'recall_3fpr': r3fpr}
    results_path = f'{DRIVE_PATH}/results.json'
    all_results = json.load(open(results_path)) if os.path.exists(results_path) else []
    all_results = [r for r in all_results if r['name'] != name]
    all_results.append(result)
    json.dump(all_results, open(results_path, 'w'), indent=2)
    print("Result saved to Drive.")
    return result

print("Training Random Forest on HOBA features...")
rf_hoba = RandomForestClassifier(
    n_estimators=100, max_depth=10,
    class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf_hoba.fit(X_train_hoba, y_train)
print(f"RAM after fit: {psutil.virtual_memory().percent}%")

Training Random Forest on HOBA features...
RAM after fit: 56.8%


In [ ]:
TEST_SCALED_PATH = f'{DRIVE_PATH}/test_hoba_scaled.parquet'

pf_test = pq.ParquetFile(TEST_SCALED_PATH)
n_rows_test = pf_test.metadata.num_rows
print(f"Loading test: {n_rows_test:,} x {n_cols}")

X_test_hoba = np.empty((n_rows_test, n_cols), dtype=np.float32)
y_test = np.empty(n_rows_test, dtype=np.int8)

row_offset = 0
for batch in pf_test.iter_batches(batch_size=100_000, columns=hoba_cols + ['is_fraud']):
    n = batch.num_rows
    for i, c in enumerate(hoba_cols):
        X_test_hoba[row_offset:row_offset+n, i] = batch.column(c).to_numpy(zero_copy_only=False)
    y_test[row_offset:row_offset+n] = batch.column('is_fraud').to_numpy(zero_copy_only=False)
    row_offset += n
    del batch
    gc.collect()

print(f"✓ X_test_hoba: {X_test_hoba.shape}")
print(f"RAM: {psutil.virtual_memory().percent}%")

y_pred = rf_hoba.predict(X_test_hoba)
y_prob = rf_hoba.predict_proba(X_test_hoba)[:, 1]
evaluate_model("Random Forest — HOBA", y_test, y_pred, y_prob)

print(f"\nRAM after predict: {psutil.virtual_memory().percent}%")

Loading test: 346,537 x 891
✓ X_test_hoba: (346537, 891)
RAM: 68.0%

=== Random Forest — HOBA ===
AUC: 0.9933 | F1: 0.3355 | Precision: 0.2047 | Recall: 0.9291
Recall @ 1% FPR: 0.8773 | Recall @ 3% FPR: 0.9459
Result saved to Drive.

RAM after predict: 68.5%


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc, os, json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve
import psutil

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
print(f"RAM after restart: {psutil.virtual_memory().percent}%")

Mounted at /content/drive
RAM after restart: 13.8%


In [ ]:
from sklearn.preprocessing import StandardScaler

# ── RFM (small, safe single load) ──────────────────────────────
print("Loading RFM...")
train_rfm_df = pd.read_parquet(f'{DRIVE_PATH}/train_rfm.parquet')
test_rfm_df  = pd.read_parquet(f'{DRIVE_PATH}/test_rfm.parquet')
rfm_cols = [c for c in train_rfm_df.columns if c.startswith('rfm_')]

X_train_rfm_raw = train_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
X_test_rfm_raw  = test_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
del train_rfm_df, test_rfm_df
gc.collect()

scaler_rfm = StandardScaler()
X_train_rfm = scaler_rfm.fit_transform(X_train_rfm_raw)
X_test_rfm  = scaler_rfm.transform(X_test_rfm_raw)
del X_train_rfm_raw, X_test_rfm_raw
gc.collect()
print(f"RFM ready: {X_train_rfm.shape}, RAM: {psutil.virtual_memory().percent}%")

# ── HOBA train (already scaled parquet, streaming load) ────────
print("\nLoading HOBA train...")
pf = pq.ParquetFile(f'{DRIVE_PATH}/train_hoba_scaled.parquet')
hoba_cols = [c for c in pf.schema_arrow.names if c.startswith('hoba_')]
n_rows = pf.metadata.num_rows
n_cols = len(hoba_cols)

X_train_hoba = np.empty((n_rows, n_cols), dtype=np.float32)
y_train = np.empty(n_rows, dtype=np.int8)
row_offset = 0
for batch in pf.iter_batches(batch_size=100_000, columns=hoba_cols + ['is_fraud']):
    n = batch.num_rows
    for i, c in enumerate(hoba_cols):
        X_train_hoba[row_offset:row_offset+n, i] = batch.column(c).to_numpy(zero_copy_only=False)
    y_train[row_offset:row_offset+n] = batch.column('is_fraud').to_numpy(zero_copy_only=False)
    row_offset += n
    del batch
    gc.collect()
print(f"HOBA train ready: {X_train_hoba.shape}, RAM: {psutil.virtual_memory().percent}%")

# ── HOBA test (streaming load) ──────────────────────────────────
print("\nLoading HOBA test...")
pf_test = pq.ParquetFile(f'{DRIVE_PATH}/test_hoba_scaled.parquet')
n_rows_test = pf_test.metadata.num_rows

X_test_hoba = np.empty((n_rows_test, n_cols), dtype=np.float32)
y_test = np.empty(n_rows_test, dtype=np.int8)
row_offset = 0
for batch in pf_test.iter_batches(batch_size=100_000, columns=hoba_cols + ['is_fraud']):
    n = batch.num_rows
    for i, c in enumerate(hoba_cols):
        X_test_hoba[row_offset:row_offset+n, i] = batch.column(c).to_numpy(zero_copy_only=False)
    y_test[row_offset:row_offset+n] = batch.column('is_fraud').to_numpy(zero_copy_only=False)
    row_offset += n
    del batch
    gc.collect()
print(f"HOBA test ready: {X_test_hoba.shape}, RAM: {psutil.virtual_memory().percent}%")

# ── Eval helpers ─────────────────────────────────────────────────
def fpr_constrained_recall(y_true, y_prob, fpr_threshold):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    valid_idx = np.where(fpr <= fpr_threshold)[0]
    if len(valid_idx) == 0:
        return 0.0
    return tpr[valid_idx[np.argmax(tpr[valid_idx])]]

def evaluate_model(name, y_true, y_pred, y_prob):
    auc = roc_auc_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    r1fpr = fpr_constrained_recall(y_true, y_prob, 0.01)
    r3fpr = fpr_constrained_recall(y_true, y_prob, 0.03)
    print(f"\n=== {name} ===")
    print(f"AUC: {auc:.4f} | F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
    print(f"Recall @ 1% FPR: {r1fpr:.4f} | Recall @ 3% FPR: {r3fpr:.4f}")
    result = {'name': name, 'auc': auc, 'f1': f1, 'precision': precision,
              'recall': recall, 'recall_1fpr': r1fpr, 'recall_3fpr': r3fpr}
    results_path = f'{DRIVE_PATH}/results.json'
    all_results = json.load(open(results_path)) if os.path.exists(results_path) else []
    all_results = [r for r in all_results if r['name'] != name]
    all_results.append(result)
    json.dump(all_results, open(results_path, 'w'), indent=2)
    print("Result saved to Drive.")
    return result

def print_results_table():
    results_path = f'{DRIVE_PATH}/results.json'
    if not os.path.exists(results_path):
        print("No results saved yet.")
        return
    all_results = json.load(open(results_path))
    print(f"\n{'Model':<35} {'AUC':>6} {'F1':>6} {'Recall':>8} {'R@1%FPR':>9} {'R@3%FPR':>9}")
    print("-" * 75)
    for r in all_results:
        print(f"{r['name']:<35} {r['auc']:>6.4f} {r['f1']:>6.4f} "
              f"{r['recall']:>8.4f} {r['recall_1fpr']:>9.4f} {r['recall_3fpr']:>9.4f}")

print_results_table()

Loading RFM...
RFM ready: (1035957, 65), RAM: 21.5%

Loading HOBA train...
HOBA train ready: (1035957, 891), RAM: 57.7%

Loading HOBA test...
HOBA test ready: (346537, 891), RAM: 67.9%

Model                                  AUC     F1   Recall   R@1%FPR   R@3%FPR
---------------------------------------------------------------------------
Random Forest — RFM                 0.9513 0.2881   0.7139    0.6349    0.7604
Random Forest — HOBA                0.9933 0.3355   0.9291    0.8773    0.9459


## Section 10: SVM (SGDClassifier) on RFM and HOBA
Fully self-contained — defines paths, loads RFM (single read) and HOBA
(streaming, since 891 cols won't safely fit via direct pandas read),
defines evaluation helpers, trains SVM on both feature sets, saves results
to Drive. Run this single cell; nothing else needed beforehand except a
fresh runtime restart with Drive mounted.

In [ ]:
# ============================================================
# SECTION 10 — SVM (SGDClassifier) — FULLY SELF-CONTAINED
# ============================================================
import gc, os, json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, roc_curve)

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'

# ── Evaluation helpers ───────────────────────────────────────
def fpr_constrained_recall(y_true, y_prob, fpr_threshold):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    valid_idx = np.where(fpr <= fpr_threshold)[0]
    if len(valid_idx) == 0:
        return 0.0
    return tpr[valid_idx[np.argmax(tpr[valid_idx])]]

def evaluate_model(name, y_true, y_pred, y_prob):
    auc = roc_auc_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    r1fpr = fpr_constrained_recall(y_true, y_prob, 0.01)
    r3fpr = fpr_constrained_recall(y_true, y_prob, 0.03)
    print(f"\n=== {name} ===")
    print(f"AUC: {auc:.4f} | F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
    print(f"Recall @ 1% FPR: {r1fpr:.4f} | Recall @ 3% FPR: {r3fpr:.4f}")
    result = {'name': name, 'auc': auc, 'f1': f1, 'precision': precision,
              'recall': recall, 'recall_1fpr': r1fpr, 'recall_3fpr': r3fpr}
    results_path = f'{DRIVE_PATH}/results.json'
    all_results = json.load(open(results_path)) if os.path.exists(results_path) else []
    all_results = [r for r in all_results if r['name'] != name]
    all_results.append(result)
    json.dump(all_results, open(results_path, 'w'), indent=2)
    print("Result saved to Drive.")
    return result

def print_results_table():
    results_path = f'{DRIVE_PATH}/results.json'
    if not os.path.exists(results_path):
        print("No results saved yet.")
        return
    all_results = json.load(open(results_path))
    print(f"\n{'Model':<35} {'AUC':>6} {'F1':>6} {'Recall':>8} {'R@1%FPR':>9} {'R@3%FPR':>9}")
    print("-" * 75)
    for r in all_results:
        print(f"{r['name']:<35} {r['auc']:>6.4f} {r['f1']:>6.4f} "
              f"{r['recall']:>8.4f} {r['recall_1fpr']:>9.4f} {r['recall_3fpr']:>9.4f}")

# ── Load RFM (small, single safe read) ──────────────────────
print("Loading RFM...")
train_rfm_df = pd.read_parquet(f'{DRIVE_PATH}/train_rfm.parquet')
test_rfm_df  = pd.read_parquet(f'{DRIVE_PATH}/test_rfm.parquet')
rfm_cols = [c for c in train_rfm_df.columns if c.startswith('rfm_')]

X_train_rfm_raw = train_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
X_test_rfm_raw  = test_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
del train_rfm_df, test_rfm_df
gc.collect()

scaler_rfm = StandardScaler()
X_train_rfm = scaler_rfm.fit_transform(X_train_rfm_raw)
X_test_rfm  = scaler_rfm.transform(X_test_rfm_raw)
del X_train_rfm_raw, X_test_rfm_raw
gc.collect()
print(f"RFM ready: {X_train_rfm.shape} | RAM: {psutil.virtual_memory().percent}%")

# ── Load HOBA train (streaming, pre-scaled file) ────────────
print("\nLoading HOBA train...")
pf = pq.ParquetFile(f'{DRIVE_PATH}/train_hoba_scaled.parquet')
hoba_cols = [c for c in pf.schema_arrow.names if c.startswith('hoba_')]
n_rows = pf.metadata.num_rows
n_cols = len(hoba_cols)

X_train_hoba = np.empty((n_rows, n_cols), dtype=np.float32)
y_train = np.empty(n_rows, dtype=np.int8)
row_offset = 0
for batch in pf.iter_batches(batch_size=100_000, columns=hoba_cols + ['is_fraud']):
    n = batch.num_rows
    for i, c in enumerate(hoba_cols):
        X_train_hoba[row_offset:row_offset+n, i] = batch.column(c).to_numpy(zero_copy_only=False)
    y_train[row_offset:row_offset+n] = batch.column('is_fraud').to_numpy(zero_copy_only=False)
    row_offset += n
    del batch
    gc.collect()
print(f"HOBA train ready: {X_train_hoba.shape} | RAM: {psutil.virtual_memory().percent}%")

# ── Load HOBA test (streaming, pre-scaled file) ─────────────
print("\nLoading HOBA test...")
pf_test = pq.ParquetFile(f'{DRIVE_PATH}/test_hoba_scaled.parquet')
n_rows_test = pf_test.metadata.num_rows

X_test_hoba = np.empty((n_rows_test, n_cols), dtype=np.float32)
y_test = np.empty(n_rows_test, dtype=np.int8)
row_offset = 0
for batch in pf_test.iter_batches(batch_size=100_000, columns=hoba_cols + ['is_fraud']):
    n = batch.num_rows
    for i, c in enumerate(hoba_cols):
        X_test_hoba[row_offset:row_offset+n, i] = batch.column(c).to_numpy(zero_copy_only=False)
    y_test[row_offset:row_offset+n] = batch.column('is_fraud').to_numpy(zero_copy_only=False)
    row_offset += n
    del batch
    gc.collect()
print(f"HOBA test ready: {X_test_hoba.shape} | RAM: {psutil.virtual_memory().percent}%")

# ── Train SVM on RFM ─────────────────────────────────────────
print("\nTraining SVM on RFM features...")
svm_rfm = SGDClassifier(loss='hinge', class_weight='balanced',
                         random_state=42, max_iter=1000, tol=1e-3, n_jobs=-1)
svm_rfm.fit(X_train_rfm, y_train)
y_pred = svm_rfm.predict(X_test_rfm)
y_prob = svm_rfm.decision_function(X_test_rfm)
evaluate_model("SVM — RFM", y_test, y_pred, y_prob)
del svm_rfm, y_pred, y_prob
gc.collect()
print(f"RAM after RFM SVM: {psutil.virtual_memory().percent}%")

# ── Train SVM on HOBA ────────────────────────────────────────
print("\nTraining SVM on HOBA features...")
svm_hoba = SGDClassifier(loss='hinge', class_weight='balanced',
                          random_state=42, max_iter=1000, tol=1e-3, n_jobs=-1)
svm_hoba.fit(X_train_hoba, y_train)
y_pred = svm_hoba.predict(X_test_hoba)
y_prob = svm_hoba.decision_function(X_test_hoba)
evaluate_model("SVM — HOBA", y_test, y_pred, y_prob)
print(f"RAM after HOBA SVM: {psutil.virtual_memory().percent}%")

print("\n")
print_results_table()

Loading RFM...
RFM ready: (1035957, 65) | RAM: 25.7%

Loading HOBA train...
HOBA train ready: (1035957, 891) | RAM: 61.7%

Loading HOBA test...
HOBA test ready: (346537, 891) | RAM: 73.2%

Training SVM on RFM features...

=== SVM — RFM ===
AUC: 0.8849 | F1: 0.0688 | Precision: 0.0361 | Recall: 0.7398
Recall @ 1% FPR: 0.3426 | Recall @ 3% FPR: 0.4969
Result saved to Drive.
RAM after RFM SVM: 73.2%

Training SVM on HOBA features...

=== SVM — HOBA ===
AUC: 0.9597 | F1: 0.1548 | Precision: 0.0848 | Recall: 0.8860
Recall @ 1% FPR: 0.6004 | Recall @ 3% FPR: 0.8150
Result saved to Drive.
RAM after HOBA SVM: 71.5%



Model                                  AUC     F1   Recall   R@1%FPR   R@3%FPR
---------------------------------------------------------------------------
Random Forest — RFM                 0.9513 0.2881   0.7139    0.6349    0.7604
Random Forest — HOBA                0.9933 0.3355   0.9291    0.8773    0.9459
SVM — RFM                           0.8849 0.0688   0.7398    0.3426 

## Section 11: BPNN (Basic Backpropagation Neural Network)
Fully self-contained — loads RFM/HOBA, defines a simple feedforward network
(Dense 64 → Dense 32 → Sigmoid output), trains on both feature sets via
TensorFlow/Keras. This is the first model to actually use GPU — RAM checkpoints
track System RAM throughout; GPU RAM is separate and tracked via nvidia-smi.
Class imbalance handled via Keras class_weight parameter.

## Section 11b: BPNN on HOBA — Using tf.data Pipeline (Memory-Safe)
Same architecture as before, but feeding data via tf.data.Dataset instead
of raw numpy arrays directly into .fit(). This avoids Keras' internal full-array
copying overhead that likely caused the previous crash at training start.

In [ ]:
# ============================================================
# SECTION 11b — BPNN on HOBA via streaming generator — SELF-CONTAINED
# No GPU available, so this is pure CPU-side streaming, RAM-safe by design.
# ============================================================
import gc, os, json
import numpy as np
import pyarrow.parquet as pq
import psutil
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, roc_curve)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
RANDOM_SEED = 42
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def fpr_constrained_recall(y_true, y_prob, fpr_threshold):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    valid_idx = np.where(fpr <= fpr_threshold)[0]
    if len(valid_idx) == 0:
        return 0.0
    return tpr[valid_idx[np.argmax(tpr[valid_idx])]]

def evaluate_model(name, y_true, y_pred, y_prob):
    auc = roc_auc_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    r1fpr = fpr_constrained_recall(y_true, y_prob, 0.01)
    r3fpr = fpr_constrained_recall(y_true, y_prob, 0.03)
    print(f"\n=== {name} ===")
    print(f"AUC: {auc:.4f} | F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
    print(f"Recall @ 1% FPR: {r1fpr:.4f} | Recall @ 3% FPR: {r3fpr:.4f}")
    result = {'name': name, 'auc': auc, 'f1': f1, 'precision': precision,
              'recall': recall, 'recall_1fpr': r1fpr, 'recall_3fpr': r3fpr}
    results_path = f'{DRIVE_PATH}/results.json'
    all_results = json.load(open(results_path)) if os.path.exists(results_path) else []
    all_results = [r for r in all_results if r['name'] != name]
    all_results.append(result)
    json.dump(all_results, open(results_path, 'w'), indent=2)
    print("Result saved to Drive.")
    return result

def print_results_table():
    results_path = f'{DRIVE_PATH}/results.json'
    all_results = json.load(open(results_path)) if os.path.exists(results_path) else []
    print(f"\n{'Model':<35} {'AUC':>6} {'F1':>6} {'Recall':>8} {'R@1%FPR':>9} {'R@3%FPR':>9}")
    print("-" * 75)
    for r in all_results:
        print(f"{r['name']:<35} {r['auc']:>6.4f} {r['f1']:>6.4f} "
              f"{r['recall']:>8.4f} {r['recall_1fpr']:>9.4f} {r['recall_3fpr']:>9.4f}")

# ── Schema / row counts only — no full array load ────────────
pf_train = pq.ParquetFile(f'{DRIVE_PATH}/train_hoba_scaled.parquet')
hoba_cols = [c for c in pf_train.schema_arrow.names if c.startswith('hoba_')]
n_cols = len(hoba_cols)
n_rows_train_full = pf_train.metadata.num_rows
print(f"HOBA train rows: {n_rows_train_full} | cols: {n_cols}")

pf_test = pq.ParquetFile(f'{DRIVE_PATH}/test_hoba_scaled.parquet')
n_rows_test = pf_test.metadata.num_rows
print(f"HOBA test rows: {n_rows_test}")

# Contiguous holdout split (no shuffling needed since parquet row order
# is already randomized/i.i.d. from earlier preprocessing — adjust if not)
VAL_FRACTION = 0.1
n_val = int(n_rows_train_full * VAL_FRACTION)
n_train = n_rows_train_full - n_val
print(f"Train rows: {n_train} | Val rows: {n_val}")

# ── Class weights — single cheap pass over just the label column ─
print("\nComputing class weights (label-only streaming pass)...")
labels = []
for batch in pf_train.iter_batches(batch_size=200_000, columns=['is_fraud']):
    labels.append(batch.column('is_fraud').to_numpy(zero_copy_only=False))
labels = np.concatenate(labels)
weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=labels)
class_weight_dict = {0: float(weights[0]), 1: float(weights[1])}
print(f"Class weights: {class_weight_dict}")
del labels
gc.collect()
print(f"RAM after class weights: {psutil.virtual_memory().percent}%")

# ── Streaming batch generators (constant RAM regardless of file size) ─
BATCH_SIZE = 512

def make_generator(parquet_path, row_start, row_end, batch_size):
    def gen():
        pf = pq.ParquetFile(parquet_path)
        row_cursor = 0
        for batch in pf.iter_batches(batch_size=batch_size, columns=hoba_cols + ['is_fraud']):
            n = batch.num_rows
            batch_start, batch_end = row_cursor, row_cursor + n
            row_cursor = batch_end
            # Skip rows outside this generator's assigned range
            lo = max(row_start, batch_start)
            hi = min(row_end, batch_end)
            if lo >= hi:
                continue
            offset_lo, offset_hi = lo - batch_start, hi - batch_start
            X = np.column_stack(
                [batch.column(c).to_numpy(zero_copy_only=False)[offset_lo:offset_hi] for c in hoba_cols]
            ).astype(np.float32)
            y = batch.column('is_fraud').to_numpy(zero_copy_only=False)[offset_lo:offset_hi].astype(np.int8)
            yield X, y
    return gen

train_ds = tf.data.Dataset.from_generator(
    make_generator(f'{DRIVE_PATH}/train_hoba_scaled.parquet', 0, n_train, BATCH_SIZE),
    output_signature=(
        tf.TensorSpec(shape=(None, n_cols), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.int8),
    )
).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_generator(
    make_generator(f'{DRIVE_PATH}/train_hoba_scaled.parquet', n_train, n_rows_train_full, BATCH_SIZE),
    output_signature=(
        tf.TensorSpec(shape=(None, n_cols), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.int8),
    )
).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_generator(
    make_generator(f'{DRIVE_PATH}/test_hoba_scaled.parquet', 0, n_rows_test, BATCH_SIZE),
    output_signature=(
        tf.TensorSpec(shape=(None, n_cols), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.int8),
    )
).prefetch(tf.data.AUTOTUNE)

print(f"RAM after dataset construction (should be near-baseline): {psutil.virtual_memory().percent}%")

early_stop = EarlyStopping(monitor='val_loss', patience=3,
                            restore_best_weights=True, verbose=1)

def build_bpnn(input_dim):
    model = Sequential([
        Dense(64, activation='relu', input_shape=(input_dim,)),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

print("\nTraining BPNN on HOBA features via streaming generator...")
bpnn_hoba = build_bpnn(input_dim=n_cols)
bpnn_hoba.fit(train_ds, validation_data=val_ds, epochs=20,
              class_weight=class_weight_dict, callbacks=[early_stop], verbose=1)
print(f"RAM after fit: {psutil.virtual_memory().percent}%")

# ── Predict via streaming too — avoid materializing X_test_hoba ──
print("\nPredicting on test set via streaming...")
y_prob_batches = []
y_test_batches = []
for X_batch, y_batch in test_ds:
    y_prob_batches.append(bpnn_hoba.predict(X_batch, verbose=0).flatten())
    y_test_batches.append(y_batch.numpy())
y_prob = np.concatenate(y_prob_batches)
y_test = np.concatenate(y_test_batches)
y_pred = (y_prob >= 0.5).astype(int)

evaluate_model("BPNN — HOBA", y_test, y_pred, y_prob)
print(f"RAM after eval: {psutil.virtual_memory().percent}%")

print("\n")
print_results_table()

HOBA train rows: 1035957 | cols: 891
HOBA test rows: 346537
Train rows: 932362 | Val rows: 103595

Computing class weights (label-only streaming pass)...
Class weights: {0: 0.5030714857347091, 1: 81.89383399209486}
RAM after class weights: 16.8%
RAM after dataset construction (should be near-baseline): 16.9%

Training BPNN on HOBA features via streaming generator...
Epoch 1/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 122s 62ms/step - AUC: 0.9526 - loss: 0.2980 - val_AUC: 0.9752 - val_loss: 0.1147
Epoch 2/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 142s 78ms/step - AUC: 0.9729 - loss: 0.2179 - val_AUC: 0.9800 - val_loss: 0.1190
Epoch 3/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 105s 58ms/step - AUC: 0.9771 - loss: 0.1998 - val_AUC: 0.9810 - val_loss: 0.1088
Epoch 4/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 104s 57ms/step - AUC: 0.9800 - loss: 0.1842 - val_AUC: 0.9822 - val_loss: 0.0926
Epoch 5/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 106s 58ms/step - AUC: 0.9815 - loss: 0.1753 - val_AUC: 0.9824 - val_loss: 0.0899
Epoch 6/20
1822/

## Section 12: DBN (Deep Belief Network) on RFM and HOBA
Implemented as Stacked Autoencoder with supervised fine-tuning — the modern
equivalent of RBM-based DBN pretraining. Fully self-contained, streaming
parquet generator for HOBA to stay within RAM limits.

Stage 1: Autoencoder 1 trained unsupervised (input → 64 dims)
Stage 2: Autoencoder 2 trained unsupervised (64 → 32 dims)  
Stage 3: Stacked encoders + classification head, fine-tuned end-to-end

Updated evaluate_model reports full metric set matching paper's Table 3
and Table 5a: Overall (AUC, F1, Precision, Recall, Accuracy) + constrained
FPR metrics at both 1% and 3% thresholds.

In [ ]:
# ============================================================
# SECTION 12 — DBN — FULLY SELF-CONTAINED
# ============================================================
import gc, os, json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, roc_curve)
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
RANDOM_SEED = 42
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

# ── Updated evaluation helpers ────────────────────────────────
def get_metrics_at_fpr(y_true, y_prob, fpr_threshold):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    valid_idx = np.where(fpr <= fpr_threshold)[0]
    if len(valid_idx) == 0:
        return {'f1': 0, 'precision': 0, 'recall': 0, 'accuracy': 0}
    best_idx = valid_idx[np.argmax(tpr[valid_idx])]
    thresh = thresholds[best_idx]
    y_pred_thresh = (y_prob >= thresh).astype(int)
    TP = ((y_pred_thresh==1)&(y_true==1)).sum()
    TN = ((y_pred_thresh==0)&(y_true==0)).sum()
    FP = ((y_pred_thresh==1)&(y_true==0)).sum()
    FN = ((y_pred_thresh==0)&(y_true==1)).sum()
    precision = TP/(TP+FP) if (TP+FP)>0 else 0
    recall    = TP/(TP+FN) if (TP+FN)>0 else 0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0
    accuracy  = (TP+TN)/len(y_true)
    return {'f1': f1, 'precision': precision, 'recall': recall, 'accuracy': accuracy}

def evaluate_model(name, y_true, y_pred, y_prob):
    auc      = roc_auc_score(y_true, y_prob)
    f1       = f1_score(y_true, y_pred)
    prec     = precision_score(y_true, y_pred)
    rec      = recall_score(y_true, y_pred)
    acc      = (y_pred==y_true).sum()/len(y_true)
    m1       = get_metrics_at_fpr(y_true, y_prob, 0.01)
    m3       = get_metrics_at_fpr(y_true, y_prob, 0.03)
    print(f"\n=== {name} ===")
    print(f"Overall  — AUC:{auc:.4f} F1:{f1:.4f} Pre:{prec:.4f} Rec:{rec:.4f} Acc:{acc:.4f}")
    print(f"@ 1% FPR — F1:{m1['f1']:.4f} Pre:{m1['precision']:.4f} Rec:{m1['recall']:.4f} Acc:{m1['accuracy']:.4f}")
    print(f"@ 3% FPR — F1:{m3['f1']:.4f} Pre:{m3['precision']:.4f} Rec:{m3['recall']:.4f} Acc:{m3['accuracy']:.4f}")
    result = {
        'name': name, 'auc': auc, 'f1': f1, 'precision': prec,
        'recall': rec, 'accuracy': acc,
        'f1_1fpr': m1['f1'], 'precision_1fpr': m1['precision'],
        'recall_1fpr': m1['recall'], 'accuracy_1fpr': m1['accuracy'],
        'f1_3fpr': m3['f1'], 'precision_3fpr': m3['precision'],
        'recall_3fpr': m3['recall'], 'accuracy_3fpr': m3['accuracy']
    }
    results_path = f'{DRIVE_PATH}/results.json'
    all_results = json.load(open(results_path)) if os.path.exists(results_path) else []
    all_results = [r for r in all_results if r['name'] != name]
    all_results.append(result)
    json.dump(all_results, open(results_path, 'w'), indent=2)
    print("Result saved to Drive.")
    return result

def print_results_table():
    results_path = f'{DRIVE_PATH}/results.json'
    if not os.path.exists(results_path):
        print("No results saved yet.")
        return
    all_results = json.load(open(results_path))
    print(f"\n{'Model':<25} {'AUC':>6} {'F1':>6} {'Rec':>6} {'Acc':>6} | {'F1@1%':>6} {'Pre@1%':>7} {'Rec@1%':>7} | {'F1@3%':>6} {'Rec@3%':>7}")
    print("-"*105)
    for r in all_results:
        print(f"{r['name']:<25} {r['auc']:>6.4f} {r['f1']:>6.4f} {r['recall']:>6.4f} {r.get('accuracy',0):>6.4f} | "
              f"{r.get('f1_1fpr',0):>6.4f} {r.get('precision_1fpr',0):>7.4f} {r.get('recall_1fpr',0):>7.4f} | "
              f"{r.get('f1_3fpr',0):>6.4f} {r.get('recall_3fpr',0):>7.4f}")

# ── Load RFM ─────────────────────────────────────────────────
print("\nLoading RFM...")
train_rfm_df = pd.read_parquet(f'{DRIVE_PATH}/train_rfm.parquet')
test_rfm_df  = pd.read_parquet(f'{DRIVE_PATH}/test_rfm.parquet')
rfm_cols = [c for c in train_rfm_df.columns if c.startswith('rfm_')]
y_train_rfm = train_rfm_df['is_fraud'].astype(int).to_numpy()
y_test_rfm  = test_rfm_df['is_fraud'].astype(int).to_numpy()

X_train_rfm_raw = train_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
X_test_rfm_raw  = test_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
del train_rfm_df, test_rfm_df
gc.collect()

scaler_rfm = StandardScaler()
X_train_rfm = scaler_rfm.fit_transform(X_train_rfm_raw)
X_test_rfm  = scaler_rfm.transform(X_test_rfm_raw)
del X_train_rfm_raw, X_test_rfm_raw
gc.collect()
print(f"RFM ready: {X_train_rfm.shape} | RAM: {psutil.virtual_memory().percent}%")

# ── Class weights ─────────────────────────────────────────────
weights = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train_rfm)
class_weight_dict = {0: float(weights[0]), 1: float(weights[1])}
print(f"Class weights: {class_weight_dict}")

early_stop = EarlyStopping(monitor='val_loss', patience=3,
                            restore_best_weights=True, verbose=1)

# ── DBN on RFM ────────────────────────────────────────────────
print("\n── Stage 1: Autoencoder 1 on RFM (in→64) ──")
inp1 = Input(shape=(X_train_rfm.shape[1],))
enc1 = Dense(64, activation='relu')(inp1)
dec1 = Dense(X_train_rfm.shape[1], activation='linear')(enc1)
ae1  = Model(inp1, dec1)
ae1.compile(optimizer='adam', loss='mse')
ae1.fit(X_train_rfm, X_train_rfm, epochs=10, batch_size=512,
        validation_split=0.1,
        callbacks=[EarlyStopping(monitor='val_loss', patience=3,
                                  restore_best_weights=True)], verbose=1)
encoder1_rfm = Model(inp1, enc1)
X_train_enc1 = encoder1_rfm.predict(X_train_rfm, verbose=0)
X_test_enc1  = encoder1_rfm.predict(X_test_rfm, verbose=0)

print("\n── Stage 2: Autoencoder 2 on RFM (64→32) ──")
inp2 = Input(shape=(64,))
enc2 = Dense(32, activation='relu')(inp2)
dec2 = Dense(64, activation='linear')(enc2)
ae2  = Model(inp2, dec2)
ae2.compile(optimizer='adam', loss='mse')
ae2.fit(X_train_enc1, X_train_enc1, epochs=10, batch_size=512,
        validation_split=0.1,
        callbacks=[EarlyStopping(monitor='val_loss', patience=3,
                                  restore_best_weights=True)], verbose=1)
encoder2_rfm = Model(inp2, enc2)

print("\n── Stage 3: Fine-tuning DBN on RFM ──")
inp_dbn = Input(shape=(X_train_rfm.shape[1],))
x = encoder1_rfm(inp_dbn)
x = encoder2_rfm(x)
x = Dropout(0.3)(x)
x = Dense(16, activation='relu')(x)
x = Dropout(0.2)(x)
out = Dense(1, activation='sigmoid')(x)
dbn_rfm = Model(inp_dbn, out)
dbn_rfm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
dbn_rfm.fit(X_train_rfm, y_train_rfm, epochs=20, batch_size=512,
            validation_split=0.1, class_weight=class_weight_dict,
            callbacks=[early_stop], verbose=1)
y_prob = dbn_rfm.predict(X_test_rfm, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)
evaluate_model("DBN — RFM", y_test_rfm, y_pred, y_prob)
print(f"RAM after DBN-RFM: {psutil.virtual_memory().percent}%")

del ae1, ae2, dbn_rfm, encoder1_rfm, encoder2_rfm
del X_train_enc1, X_test_enc1
del X_train_rfm, X_test_rfm
gc.collect()

# ── HOBA streaming setup ──────────────────────────────────────
print("\nSetting up HOBA streaming...")
pf_train = pq.ParquetFile(f'{DRIVE_PATH}/train_hoba_scaled.parquet')
hoba_cols = [c for c in pf_train.schema_arrow.names if c.startswith('hoba_')]
n_cols = len(hoba_cols)
n_rows_train = pf_train.metadata.num_rows

pf_test_hoba = pq.ParquetFile(f'{DRIVE_PATH}/test_hoba_scaled.parquet')
n_rows_test_hoba = pf_test_hoba.metadata.num_rows

VAL_FRACTION = 0.1
n_val   = int(n_rows_train * VAL_FRACTION)
n_train = n_rows_train - n_val

# Class weights for HOBA (stream labels only)
labels = np.concatenate([
    batch.column('is_fraud').to_numpy(zero_copy_only=False)
    for batch in pf_train.iter_batches(batch_size=200_000, columns=['is_fraud'])
])
weights_h = compute_class_weight('balanced', classes=np.array([0,1]), y=labels)
cw_hoba = {0: float(weights_h[0]), 1: float(weights_h[1])}
del labels; gc.collect()
print(f"HOBA class weights: {cw_hoba}")
print(f"RAM: {psutil.virtual_memory().percent}%")

BATCH_SIZE = 512

def make_generator(parquet_path, row_start, row_end, batch_size, cols, label_col='is_fraud'):
    def gen():
        pf = pq.ParquetFile(parquet_path)
        row_cursor = 0
        for batch in pf.iter_batches(batch_size=batch_size, columns=cols+[label_col]):
            n = batch.num_rows
            lo = max(row_start, row_cursor)
            hi = min(row_end, row_cursor+n)
            row_cursor += n
            if lo >= hi:
                continue
            olo, ohi = lo-(row_cursor-n), hi-(row_cursor-n)
            X = np.column_stack([batch.column(c).to_numpy(zero_copy_only=False)[olo:ohi]
                                  for c in cols]).astype(np.float32)
            y = batch.column(label_col).to_numpy(zero_copy_only=False)[olo:ohi].astype(np.int8)
            yield X, y
    return gen

train_path = f'{DRIVE_PATH}/train_hoba_scaled.parquet'
test_path  = f'{DRIVE_PATH}/test_hoba_scaled.parquet'
sig = (tf.TensorSpec(shape=(None,n_cols), dtype=tf.float32),
       tf.TensorSpec(shape=(None,),       dtype=tf.int8))

train_ds = tf.data.Dataset.from_generator(
    make_generator(train_path, 0, n_train, BATCH_SIZE, hoba_cols),
    output_signature=sig).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_generator(
    make_generator(train_path, n_train, n_rows_train, BATCH_SIZE, hoba_cols),
    output_signature=sig).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_generator(
    make_generator(test_path, 0, n_rows_test_hoba, BATCH_SIZE, hoba_cols),
    output_signature=sig).prefetch(tf.data.AUTOTUNE)

# ── DBN on HOBA — Stage 1: AE1 via streaming ─────────────────
print("\n── Stage 1: Autoencoder 1 on HOBA (891→64) ──")
ae1_ds = tf.data.Dataset.from_generator(
    make_generator(train_path, 0, n_train, BATCH_SIZE, hoba_cols),
    output_signature=sig).map(lambda x,y: (x,x)).prefetch(tf.data.AUTOTUNE)
ae1_val_ds = tf.data.Dataset.from_generator(
    make_generator(train_path, n_train, n_rows_train, BATCH_SIZE, hoba_cols),
    output_signature=sig).map(lambda x,y: (x,x)).prefetch(tf.data.AUTOTUNE)

inp_h = Input(shape=(n_cols,))
enc1_h = Dense(64, activation='relu')(inp_h)
dec1_h = Dense(n_cols, activation='linear')(enc1_h)
ae1_hoba = Model(inp_h, dec1_h)
ae1_hoba.compile(optimizer='adam', loss='mse')
ae1_hoba.fit(ae1_ds, validation_data=ae1_val_ds, epochs=5,
             callbacks=[EarlyStopping(monitor='val_loss', patience=2,
                                       restore_best_weights=True)], verbose=1)
encoder1_hoba = Model(inp_h, enc1_h)

# ── Stage 2: AE2 via streaming encoded representations ───────
print("\n── Stage 2: Autoencoder 2 on HOBA (64→32) ──")
enc1_train_ds = tf.data.Dataset.from_generator(
    make_generator(train_path, 0, n_train, BATCH_SIZE, hoba_cols),
    output_signature=sig).map(lambda x,y: (encoder1_hoba(x, training=False),
                                             encoder1_hoba(x, training=False))).prefetch(tf.data.AUTOTUNE)
enc1_val_ds = tf.data.Dataset.from_generator(
    make_generator(train_path, n_train, n_rows_train, BATCH_SIZE, hoba_cols),
    output_signature=sig).map(lambda x,y: (encoder1_hoba(x, training=False),
                                             encoder1_hoba(x, training=False))).prefetch(tf.data.AUTOTUNE)

inp2_h = Input(shape=(64,))
enc2_h = Dense(32, activation='relu')(inp2_h)
dec2_h = Dense(64, activation='linear')(enc2_h)
ae2_hoba = Model(inp2_h, dec2_h)
ae2_hoba.compile(optimizer='adam', loss='mse')
ae2_hoba.fit(enc1_train_ds, validation_data=enc1_val_ds, epochs=5,
             callbacks=[EarlyStopping(monitor='val_loss', patience=2,
                                       restore_best_weights=True)], verbose=1)
encoder2_hoba = Model(inp2_h, enc2_h)

# ── Stage 3: Fine-tuning DBN on HOBA ─────────────────────────
print("\n── Stage 3: Fine-tuning DBN on HOBA ──")
inp_dbn_h = Input(shape=(n_cols,))
x = encoder1_hoba(inp_dbn_h)
x = encoder2_hoba(x)
x = Dropout(0.3)(x)
x = Dense(16, activation='relu')(x)
x = Dropout(0.2)(x)
out_h = Dense(1, activation='sigmoid')(x)
dbn_hoba = Model(inp_dbn_h, out_h)
dbn_hoba.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
dbn_hoba.fit(train_ds, validation_data=val_ds, epochs=20,
             class_weight=cw_hoba, callbacks=[early_stop], verbose=1)
print(f"RAM after DBN-HOBA fit: {psutil.virtual_memory().percent}%")

# ── Predict via streaming ─────────────────────────────────────
print("\nPredicting on test set...")
y_prob_list, y_test_list = [], []
for X_batch, y_batch in test_ds:
    y_prob_list.append(dbn_hoba.predict(X_batch, verbose=0).flatten())
    y_test_list.append(y_batch.numpy())
y_prob = np.concatenate(y_prob_list)
y_test = np.concatenate(y_test_list)
y_pred = (y_prob >= 0.5).astype(int)

evaluate_model("DBN — HOBA", y_test, y_pred, y_prob)
print(f"RAM after eval: {psutil.virtual_memory().percent}%")

print("\n")
print_results_table()

GPU: []

Loading RFM...
RFM ready: (1035957, 65) | RAM: 24.3%
Class weights: {0: 0.5030714857347091, 1: 81.89383399209486}

── Stage 1: Autoencoder 1 on RFM (in→64) ──
Epoch 1/10
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.2867 - val_loss: 0.0287
Epoch 2/10
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0842 - val_loss: 0.0154
Epoch 3/10
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.0515 - val_loss: 0.0113
Epoch 4/10
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.0409 - val_loss: 0.0088
Epoch 5/10
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.0336 - val_loss: 0.0072
Epoch 6/10
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.0330 - val_loss: 0.0057
Epoch 7/10
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0249 - val_loss: 0.0046
Epoch 8/10
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0238 - val_loss: 0.0039
Epoch 9/10
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0279 - val_loss: 0.0041
Epoch 10/10
1822/1822 ━━━━━━━━━━━━━━━━━━

## Section 13: CNN on RFM and HOBA (True Feature Matrix Implementation)
Fully self-contained. Implements the paper's exact CNN input structure:
rows = feature types (891), columns = recent transactions (15) in
chronological order. For each transaction T, the last 15 prior transactions
on the same card form the matrix columns, zero-padded on left if fewer than
15 prior transactions exist. Card→row index lookup built from sorted parquet.
RFM CNN uses standard 1D-CNN on the 65-feature vector.
Falls back to streaming 1D-CNN if full HOBA array load fails.
Class imbalance via class_weight. Results saved to Drive.

In [2]:
# ============================================================
# SECTION 13 — CNN (TRUE FEATURE MATRIX) — FIXED, SELF-CONTAINED
# ============================================================
import gc, os, json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, roc_curve)
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Dense, Dropout, Input, Conv1D,
                                     MaxPooling1D, Flatten, Reshape,
                                     Permute)
from tensorflow.keras.callbacks import EarlyStopping

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
RANDOM_SEED = 42
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
SEQ_LEN = 15
print(f"GPU: {tf.config.list_physical_devices('GPU')}")
if not tf.config.list_physical_devices('GPU'):
    print("WARNING: running on CPU. The HOBA matrix generator does a Python "
          "loop per transaction — this will be slow. Switch runtime to GPU "
          "(Runtime > Change runtime type) if this cell is taking forever.")

# ── Evaluation helpers ────────────────────────────────────────
def get_metrics_at_fpr(y_true, y_prob, fpr_threshold):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    valid_idx = np.where(fpr <= fpr_threshold)[0]
    if len(valid_idx) == 0:
        return {'f1': 0, 'precision': 0, 'recall': 0, 'accuracy': 0}
    best_idx = valid_idx[np.argmax(tpr[valid_idx])]
    thresh = thresholds[best_idx]
    y_pred_thresh = (y_prob >= thresh).astype(int)
    TP = ((y_pred_thresh==1)&(y_true==1)).sum()
    TN = ((y_pred_thresh==0)&(y_true==0)).sum()
    FP = ((y_pred_thresh==1)&(y_true==0)).sum()
    FN = ((y_pred_thresh==0)&(y_true==1)).sum()
    precision = TP/(TP+FP) if (TP+FP)>0 else 0
    recall    = TP/(TP+FN) if (TP+FN)>0 else 0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0
    return {'f1': f1, 'precision': precision, 'recall': recall,
            'accuracy': (TP+TN)/len(y_true)}

def evaluate_model(name, y_true, y_pred, y_prob):
    auc  = roc_auc_score(y_true, y_prob)
    f1   = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    acc  = (y_pred==y_true).sum()/len(y_true)
    m1   = get_metrics_at_fpr(y_true, y_prob, 0.01)
    m3   = get_metrics_at_fpr(y_true, y_prob, 0.03)
    print(f"\n=== {name} ===")
    print(f"Overall  — AUC:{auc:.4f} F1:{f1:.4f} Pre:{prec:.4f} Rec:{rec:.4f} Acc:{acc:.4f}")
    print(f"@ 1% FPR — F1:{m1['f1']:.4f} Pre:{m1['precision']:.4f} Rec:{m1['recall']:.4f} Acc:{m1['accuracy']:.4f}")
    print(f"@ 3% FPR — F1:{m3['f1']:.4f} Pre:{m3['precision']:.4f} Rec:{m3['recall']:.4f} Acc:{m3['accuracy']:.4f}")
    result = {
        'name': name, 'auc': auc, 'f1': f1, 'precision': prec,
        'recall': rec, 'accuracy': acc,
        'f1_1fpr': m1['f1'], 'precision_1fpr': m1['precision'],
        'recall_1fpr': m1['recall'], 'accuracy_1fpr': m1['accuracy'],
        'f1_3fpr': m3['f1'], 'precision_3fpr': m3['precision'],
        'recall_3fpr': m3['recall'], 'accuracy_3fpr': m3['accuracy']
    }
    results_path = f'{DRIVE_PATH}/results.json'
    all_results = json.load(open(results_path)) if os.path.exists(results_path) else []
    all_results = [r for r in all_results if r['name'] != name]
    all_results.append(result)
    json.dump(all_results, open(results_path, 'w'), indent=2)
    print("Result saved to Drive.")
    return result

def print_results_table():
    results_path = f'{DRIVE_PATH}/results.json'
    if not os.path.exists(results_path):
        print("No results saved yet.")
        return
    all_results = json.load(open(results_path))
    print(f"\n{'Model':<25} {'AUC':>6} {'F1':>6} {'Rec':>6} {'Acc':>6} | {'F1@1%':>6} {'Pre@1%':>7} {'Rec@1%':>7} | {'F1@3%':>6} {'Rec@3%':>7}")
    print("-"*105)
    for r in all_results:
        print(f"{r['name']:<25} {r['auc']:>6.4f} {r['f1']:>6.4f} {r['recall']:>6.4f} {r.get('accuracy',0):>6.4f} | "
              f"{r.get('f1_1fpr',0):>6.4f} {r.get('precision_1fpr',0):>7.4f} {r.get('recall_1fpr',0):>7.4f} | "
              f"{r.get('f1_3fpr',0):>6.4f} {r.get('recall_3fpr',0):>7.4f}")

# ── Load RFM ─────────────────────────────────────────────────
print("\nLoading RFM...")
train_rfm_df = pd.read_parquet(f'{DRIVE_PATH}/train_rfm.parquet')
test_rfm_df  = pd.read_parquet(f'{DRIVE_PATH}/test_rfm.parquet')
rfm_cols     = [c for c in train_rfm_df.columns if c.startswith('rfm_')]
y_train_rfm  = train_rfm_df['is_fraud'].astype(int).to_numpy()
y_test_rfm   = test_rfm_df['is_fraud'].astype(int).to_numpy()
X_train_rfm_raw = train_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
X_test_rfm_raw  = test_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
del train_rfm_df, test_rfm_df; gc.collect()

scaler_rfm   = StandardScaler()
X_train_rfm  = scaler_rfm.fit_transform(X_train_rfm_raw)
X_test_rfm   = scaler_rfm.transform(X_test_rfm_raw)
del X_train_rfm_raw, X_test_rfm_raw; gc.collect()
print(f"RFM ready: {X_train_rfm.shape} | RAM: {psutil.virtual_memory().percent}%")

weights = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train_rfm)
class_weight_dict = {0: float(weights[0]), 1: float(weights[1])}
early_stop = EarlyStopping(monitor='val_loss', patience=3,
                            restore_best_weights=True, verbose=1)

# ── CNN architectures ──────────────────────────────────────────
def build_cnn_1d(input_dim):
    model = Sequential([
        Reshape((input_dim, 1), input_shape=(input_dim,)),
        Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        Conv1D(32, kernel_size=3, activation='relu', padding='same'),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

def build_cnn_matrix(n_features, seq_len):
    """Paper's Fig. 4 architecture: rows=feature types, cols=recent transactions."""
    inp = Input(shape=(seq_len, n_features))
    x = Permute((2, 1))(inp)  # -> (batch, n_features, seq_len)
    x = Conv1D(64, kernel_size=3, activation='relu', padding='same')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)
    x = Conv1D(32, kernel_size=3, activation='relu', padding='same')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x)
    out = Dense(1, activation='sigmoid')(x)
    model = Model(inp, out)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

# ── CNN on RFM (1D) ───────────────────────────────────────────
print("\nTraining CNN on RFM features...")
cnn_rfm = build_cnn_1d(input_dim=X_train_rfm.shape[1])
cnn_rfm.fit(X_train_rfm, y_train_rfm, epochs=20, batch_size=512,
            validation_split=0.1, class_weight=class_weight_dict,
            callbacks=[early_stop], verbose=1)
y_prob = cnn_rfm.predict(X_test_rfm, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)
evaluate_model("CNN — RFM", y_test_rfm, y_pred, y_prob)
print(f"RAM after CNN-RFM: {psutil.virtual_memory().percent}%")
del cnn_rfm, y_pred, y_prob, X_train_rfm, X_test_rfm; gc.collect()

# ── Load full HOBA arrays for true matrix construction ────────
print("\nAttempting to load full HOBA arrays...")

pf = pq.ParquetFile(f'{DRIVE_PATH}/train_hoba_scaled.parquet')

all_cols = pf.schema_arrow.names
hoba_cols = [c for c in all_cols if c.startswith('hoba_')]
n_cols = len(hoba_cols)
n_rows_train = pf.metadata.num_rows

try:
    pf = pq.ParquetFile(f'{DRIVE_PATH}/train_hoba_scaled.parquet')
    all_cols = pf.schema_arrow.names
    hoba_cols = [c for c in all_cols if c.startswith('hoba_')]
    n_cols = len(hoba_cols)
    n_rows_train = pf.metadata.num_rows

    key_spec = detect_card_key_columns(all_cols)
    if isinstance(key_spec, tuple):
        _, user_col, card_col = key_spec
        read_cols = [user_col, card_col]
        print(f"  No single card_id column found. Building composite key from "
              f"'{user_col}' + '{card_col}'.")
    else:
        read_cols = [key_spec]
        print(f"  Using card identifier column: '{key_spec}'")

    print(f"  Allocating train array: {n_rows_train} x {n_cols} = {n_rows_train*n_cols*4/1e9:.2f}GB")
    X_train_hoba = np.empty((n_rows_train, n_cols), dtype=np.float32)
    y_train_hoba = np.empty(n_rows_train, dtype=np.int8)
    card_ids_train = np.empty(n_rows_train, dtype=np.int64)

    row_offset = 0
    for batch in pf.iter_batches(batch_size=100_000,
                                   columns=hoba_cols + ['is_fraud'] + read_cols):
        n = batch.num_rows
        for i, c in enumerate(hoba_cols):
            X_train_hoba[row_offset:row_offset+n, i] = \
                batch.column(c).to_numpy(zero_copy_only=False)
        y_train_hoba[row_offset:row_offset+n] = \
            batch.column('is_fraud').to_numpy(zero_copy_only=False)
        if isinstance(key_spec, tuple):
            u = batch.column(user_col).to_numpy(zero_copy_only=False).astype(np.int64)
            c_ = batch.column(card_col).to_numpy(zero_copy_only=False).astype(np.int64)
            card_ids_train[row_offset:row_offset+n] = u * 100 + c_
        else:
            vals = batch.column(key_spec).to_pylist()
            card_ids_train[row_offset:row_offset+n] = np.array(
                [hash(v) & 0x7FFFFFFF for v in vals], dtype=np.int64
            ) if not isinstance(vals[0], (int, np.integer)) else np.array(vals, dtype=np.int64)
        row_offset += n
        del batch; gc.collect()

    print(f"  Train loaded. RAM: {psutil.virtual_memory().percent}%")

    pf_test = pq.ParquetFile(f'{DRIVE_PATH}/test_hoba_scaled.parquet')
    n_rows_test = pf_test.metadata.num_rows
    X_test_hoba = np.empty((n_rows_test, n_cols), dtype=np.float32)
    y_test_hoba = np.empty(n_rows_test, dtype=np.int8)
    card_ids_test = np.empty(n_rows_test, dtype=np.int64)

    row_offset = 0
    for batch in pf_test.iter_batches(batch_size=100_000,
                                       columns=hoba_cols + ['is_fraud'] + read_cols):
        n = batch.num_rows
        for i, c in enumerate(hoba_cols):
            X_test_hoba[row_offset:row_offset+n, i] = \
                batch.column(c).to_numpy(zero_copy_only=False)
        y_test_hoba[row_offset:row_offset+n] = \
            batch.column('is_fraud').to_numpy(zero_copy_only=False)
        if isinstance(key_spec, tuple):
            u = batch.column(user_col).to_numpy(zero_copy_only=False).astype(np.int64)
            c_ = batch.column(card_col).to_numpy(zero_copy_only=False).astype(np.int64)
            card_ids_test[row_offset:row_offset+n] = u * 100 + c_
        else:
            vals = batch.column(key_spec).to_pylist()
            card_ids_test[row_offset:row_offset+n] = np.array(
                [hash(v) & 0x7FFFFFFF for v in vals], dtype=np.int64
            ) if not isinstance(vals[0], (int, np.integer)) else np.array(vals, dtype=np.int64)
        row_offset += n
        del batch; gc.collect()

    print(f"  Test loaded. RAM: {psutil.virtual_memory().percent}%")
    USE_MATRIX_CNN = True

except MemoryError:
    print("  MemoryError — falling back to streaming 1D-CNN")
    USE_MATRIX_CNN = False

# ── Build card → sorted row indices lookup (fast, groupby-based) ──
if USE_MATRIX_CNN:
    print("\nBuilding card→row index lookup (train)...")
    card_to_rows = (
        pd.Series(np.arange(len(card_ids_train)))
          .groupby(card_ids_train)
          .apply(lambda s: s.to_numpy())
          .to_dict()
    )
    print(f"  Lookup built for {len(card_to_rows)} cards. RAM: {psutil.virtual_memory().percent}%")

    # ── Batch builders ─────────────────────────────────────────
    def build_matrix_batch(X_src, row_indices, y_src, card_ids_arr, card_to_rows,
                            seq_len, cutoff_by_position):
        """cutoff_by_position=True: only use same-array history strictly before
        the current row (leakage prevention within train/val). Use False only
        when the history array is guaranteed chronologically earlier than the
        rows being scored (train history -> test predictions)."""
        batch_X = np.zeros((len(row_indices), seq_len, X_src.shape[1]), dtype=np.float32)
        batch_y = y_src[row_indices]
        for bi, idx in enumerate(row_indices):
            card = card_ids_arr[idx]
            card_rows = card_to_rows.get(card)
            if card_rows is None or len(card_rows) == 0:
                continue
            prior = card_rows[card_rows < idx] if cutoff_by_position else card_rows
            if len(prior) == 0:
                continue
            seq = prior[-seq_len:]
            offset = seq_len - len(seq)
            batch_X[bi, offset:, :] = X_src[seq, :]
        return batch_X, batch_y

    def matrix_batch_generator(X_src, y_src, card_ids_arr, card_to_rows,
                                seq_len, batch_size, indices, shuffle=False):
        idx_order = indices.copy()
        if shuffle:
            np.random.shuffle(idx_order)
        for start in range(0, len(idx_order), batch_size):
            batch_idx = idx_order[start:start+batch_size]
            yield build_matrix_batch(X_src, batch_idx, y_src, card_ids_arr,
                                      card_to_rows, seq_len, cutoff_by_position=True)

    def test_batch_generator(card_ids_test_arr, y_test_arr, card_to_rows_train,
                              X_train_src, seq_len, batch_size):
        n = len(y_test_arr)
        indices = np.arange(n)
        for start in range(0, n, batch_size):
            batch_idx = indices[start:start+batch_size]
            yield build_matrix_batch(X_train_src, batch_idx, y_test_arr,
                                      card_ids_test_arr, card_to_rows_train,
                                      seq_len, cutoff_by_position=False)

    # ── Train/val split (chronological — last 10% held out) ────
    n_train_hoba = len(y_train_hoba)
    val_size   = int(0.1 * n_train_hoba)
    train_size = n_train_hoba - val_size
    train_idx = np.arange(train_size)
    val_idx   = np.arange(train_size, n_train_hoba)

    BATCH_SIZE = 256
    cnn_hoba = build_cnn_matrix(n_features=n_cols, seq_len=SEQ_LEN)
    cnn_hoba.summary()

    best_val_loss = np.inf
    patience_count = 0
    PATIENCE = 3
    best_weights = None

    print("\nTraining CNN on HOBA (true feature matrix)...")
    for epoch in range(20):
        train_losses, train_aucs = [], []
        for X_batch, y_batch in matrix_batch_generator(
                X_train_hoba, y_train_hoba, card_ids_train,
                card_to_rows, SEQ_LEN, BATCH_SIZE, train_idx, shuffle=True):
            sample_w = np.where(y_batch==1, class_weight_dict[1], class_weight_dict[0])
            metrics = cnn_hoba.train_on_batch(X_batch, y_batch, sample_weight=sample_w)
            train_losses.append(metrics[0]); train_aucs.append(metrics[1])

        val_losses, val_aucs = [], []
        for X_batch, y_batch in matrix_batch_generator(
                X_train_hoba, y_train_hoba, card_ids_train,
                card_to_rows, SEQ_LEN, BATCH_SIZE, val_idx, shuffle=False):
            metrics = cnn_hoba.test_on_batch(X_batch, y_batch)
            val_losses.append(metrics[0]); val_aucs.append(metrics[1])

        avg_train_loss, avg_val_loss = np.mean(train_losses), np.mean(val_losses)
        avg_train_auc, avg_val_auc   = np.mean(train_aucs), np.mean(val_aucs)
        print(f"Epoch {epoch+1}/20 — loss:{avg_train_loss:.4f} AUC:{avg_train_auc:.4f} "
              f"val_loss:{avg_val_loss:.4f} val_AUC:{avg_val_auc:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_weights = cnn_hoba.get_weights()
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"Early stopping at epoch {epoch+1}")
                cnn_hoba.set_weights(best_weights)
                break

    print(f"RAM after CNN-HOBA fit: {psutil.virtual_memory().percent}%")

    # ── Predict on test (history always from train array) ──────
    print("\nPredicting on test set...")
    y_prob_list, y_test_list = [], []
    for X_batch, y_batch in test_batch_generator(
            card_ids_test, y_test_hoba, card_to_rows, X_train_hoba, SEQ_LEN, BATCH_SIZE):
        y_prob_list.append(cnn_hoba.predict(X_batch, verbose=0).flatten())
        y_test_list.append(y_batch)

    y_prob = np.concatenate(y_prob_list)
    y_test = np.concatenate(y_test_list)
    y_pred = (y_prob >= 0.5).astype(int)
    evaluate_model("CNN — HOBA", y_test, y_pred, y_prob)

else:
    # Fallback: streaming 1D-CNN (unchanged — doesn't need card_id at all)
    print("\nFallback: Training streaming 1D-CNN on HOBA...")
    pf_train = pq.ParquetFile(f'{DRIVE_PATH}/train_hoba_scaled.parquet')
    hoba_cols = [c for c in pf_train.schema_arrow.names if c.startswith('hoba_')]
    n_cols = len(hoba_cols)
    n_rows_train = pf_train.metadata.num_rows
    pf_test_f = pq.ParquetFile(f'{DRIVE_PATH}/test_hoba_scaled.parquet')
    n_rows_test = pf_test_f.metadata.num_rows
    n_val   = int(n_rows_train * 0.1)
    n_train = n_rows_train - n_val

    labels = np.concatenate([
        b.column('is_fraud').to_numpy(zero_copy_only=False)
        for b in pf_train.iter_batches(batch_size=200_000, columns=['is_fraud'])
    ])
    wh = compute_class_weight('balanced', classes=np.array([0,1]), y=labels)
    cw_hoba = {0: float(wh[0]), 1: float(wh[1])}
    del labels; gc.collect()

    train_path = f'{DRIVE_PATH}/train_hoba_scaled.parquet'
    test_path  = f'{DRIVE_PATH}/test_hoba_scaled.parquet'

    def make_generator(path, row_start, row_end, bsz, cols):
        def gen():
            pf = pq.ParquetFile(path)
            rc = 0
            for batch in pf.iter_batches(batch_size=bsz, columns=cols+['is_fraud']):
                n = batch.num_rows
                lo = max(row_start, rc); hi = min(row_end, rc+n); rc += n
                if lo >= hi: continue
                olo, ohi = lo-(rc-n), hi-(rc-n)
                X = np.column_stack([batch.column(c).to_numpy(
                    zero_copy_only=False)[olo:ohi] for c in cols]).astype(np.float32)
                y = batch.column('is_fraud').to_numpy(
                    zero_copy_only=False)[olo:ohi].astype(np.int8)
                yield X, y
        return gen

    sig = (tf.TensorSpec(shape=(None, n_cols), dtype=tf.float32),
           tf.TensorSpec(shape=(None,),        dtype=tf.int8))
    train_ds = tf.data.Dataset.from_generator(
        make_generator(train_path, 0, n_train, 512, hoba_cols),
        output_signature=sig).prefetch(tf.data.AUTOTUNE)
    val_ds = tf.data.Dataset.from_generator(
        make_generator(train_path, n_train, n_rows_train, 512, hoba_cols),
        output_signature=sig).prefetch(tf.data.AUTOTUNE)
    test_ds = tf.data.Dataset.from_generator(
        make_generator(test_path, 0, n_rows_test, 512, hoba_cols),
        output_signature=sig).prefetch(tf.data.AUTOTUNE)

    cnn_hoba = build_cnn_1d(input_dim=n_cols)
    cnn_hoba.fit(train_ds, validation_data=val_ds, epochs=20,
                 class_weight=cw_hoba, callbacks=[early_stop], verbose=1)

    y_prob_list, y_test_list = [], []
    for X_batch, y_batch in test_ds:
        y_prob_list.append(cnn_hoba.predict(X_batch, verbose=0).flatten())
        y_test_list.append(y_batch.numpy())
    y_prob = np.concatenate(y_prob_list)
    y_test = np.concatenate(y_test_list)
    y_pred = (y_prob >= 0.5).astype(int)
    evaluate_model("CNN — HOBA", y_test, y_pred, y_prob)

print(f"\nFinal RAM: {psutil.virtual_memory().percent}%")
print("\n")
print_results_table()

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Loading RFM...
RFM ready: (1035957, 65) | RAM: 27.5%

Training CNN on RFM features...
Epoch 1/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - AUC: 0.9159 - loss: 0.3712 - val_AUC: 0.9621 - val_loss: 0.2692
Epoch 2/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - AUC: 0.9470 - loss: 0.2921 - val_AUC: 0.9699 - val_loss: 0.2473
Epoch 3/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - AUC: 0.9549 - loss: 0.2677 - val_AUC: 0.9738 - val_loss: 0.2280
Epoch 4/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - AUC: 0.9595 - loss: 0.2546 - val_AUC: 0.9745 - val_loss: 0.1898
Epoch 5/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - AUC: 0.9613 - loss: 0.2479 - val_AUC: 0.9764 - val_loss: 0.2070
Epoch 6/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - AUC: 0.9645 - loss: 0.2377 - val_AUC: 0.9765 - val_loss: 0.2065
Epoch 7/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - AUC: 0.9645 - loss: 0.2352 - val_AUC: 0.9783 - val_loss: 0.

KeyError: "Could not find a card identifier column or a User/Card pair to build one. Available columns: ['hoba_purchase_1d_amt_mean', 'hoba_purchase_1d_amt_std', 'hoba_purchase_1d_amt_max', 'hoba_purchase_1d_amt_sum', 'hoba_purchase_1d_timeint_mean', 'hoba_purchase_1d_timeint_std', 'hoba_purchase_1d_timeint_max', 'hoba_purchase_1d_timeint_sum', 'hoba_purchase_1d_util_mean', 'hoba_purchase_1d_util_std', 'hoba_purchase_1d_util_max', 'hoba_purchase_1d_util_sum', 'hoba_purchase_3d_amt_mean', 'hoba_purchase_3d_amt_std', 'hoba_purchase_3d_amt_max', 'hoba_purchase_3d_amt_sum', 'hoba_purchase_3d_timeint_mean', 'hoba_purchase_3d_timeint_std', 'hoba_purchase_3d_timeint_max', 'hoba_purchase_3d_timeint_sum', 'hoba_purchase_3d_util_mean', 'hoba_purchase_3d_util_std', 'hoba_purchase_3d_util_max', 'hoba_purchase_3d_util_sum', 'hoba_purchase_5d_amt_mean', 'hoba_purchase_5d_amt_std', 'hoba_purchase_5d_amt_max', 'hoba_purchase_5d_amt_sum', 'hoba_purchase_5d_timeint_mean', 'hoba_purchase_5d_timeint_std',

# CNN Model (Paper Feature Matrix)

This notebook implements the CNN model described in the HOBA paper.

Workflow:
1. Import libraries and define constants.
2. Define evaluation and CNN helper functions.
3. Load and preprocess RFM features.
4. Train and evaluate CNN on RFM.
5. Load HOBA features together with card IDs.
6. Construct transaction feature matrices and train CNN on HOBA.

In [5]:
# ============================================================
# CELL 1 — Imports & Configuration
# ============================================================

import gc
import os
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    roc_curve
)

import tensorflow as tf

from tensorflow.keras.models import (
    Sequential,
    Model
)

from tensorflow.keras.layers import (
    Dense,
    Dropout,
    Input,
    Conv1D,
    MaxPooling1D,
    Flatten,
    Reshape,
    Permute
)

from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# Configuration
# ============================================================

DRIVE_PATH = "/content/drive/MyDrive/HOBA_Fraud_Detection_V2"

RANDOM_SEED = 42
SEQ_LEN = 15

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# ============================================================
# Hardware Check
# ============================================================

print(f"GPU: {tf.config.list_physical_devices('GPU')}")

if not tf.config.list_physical_devices("GPU"):
    print(
        "WARNING: running on CPU.\n"
        "The HOBA matrix generator performs a Python loop per "
        "transaction and will be very slow.\n"
        "Switch Runtime → Change Runtime Type → GPU."
    )

print(f"Current RAM Usage: {psutil.virtual_memory().percent}%")

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Current RAM Usage: 26.0%


In [6]:
# ============================================================
# CELL 2A — Evaluation Helpers
# ============================================================

def get_metrics_at_fpr(y_true, y_prob, fpr_threshold):

    fpr, tpr, thresholds = roc_curve(y_true, y_prob)

    valid_idx = np.where(fpr <= fpr_threshold)[0]

    if len(valid_idx) == 0:
        return {
            "f1": 0,
            "precision": 0,
            "recall": 0,
            "accuracy": 0
        }

    best_idx = valid_idx[np.argmax(tpr[valid_idx])]
    thresh = thresholds[best_idx]

    y_pred_thresh = (y_prob >= thresh).astype(int)

    TP = ((y_pred_thresh == 1) & (y_true == 1)).sum()
    TN = ((y_pred_thresh == 0) & (y_true == 0)).sum()
    FP = ((y_pred_thresh == 1) & (y_true == 0)).sum()
    FN = ((y_pred_thresh == 0) & (y_true == 1)).sum()

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0
    )

    return {
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "accuracy": (TP + TN) / len(y_true)
    }


def evaluate_model(name, y_true, y_pred, y_prob):

    auc  = roc_auc_score(y_true, y_prob)
    f1   = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    acc  = (y_pred == y_true).sum() / len(y_true)

    m1 = get_metrics_at_fpr(y_true, y_prob, 0.01)
    m3 = get_metrics_at_fpr(y_true, y_prob, 0.03)

    print(f"\n=== {name} ===")

    print(
        f"Overall  — "
        f"AUC:{auc:.4f} "
        f"F1:{f1:.4f} "
        f"Pre:{prec:.4f} "
        f"Rec:{rec:.4f} "
        f"Acc:{acc:.4f}"
    )

    print(
        f"@ 1% FPR — "
        f"F1:{m1['f1']:.4f} "
        f"Pre:{m1['precision']:.4f} "
        f"Rec:{m1['recall']:.4f} "
        f"Acc:{m1['accuracy']:.4f}"
    )

    print(
        f"@ 3% FPR — "
        f"F1:{m3['f1']:.4f} "
        f"Pre:{m3['precision']:.4f} "
        f"Rec:{m3['recall']:.4f} "
        f"Acc:{m3['accuracy']:.4f}"
    )

    result = {

        "name": name,

        "auc": auc,
        "f1": f1,
        "precision": prec,
        "recall": rec,
        "accuracy": acc,

        "f1_1fpr": m1["f1"],
        "precision_1fpr": m1["precision"],
        "recall_1fpr": m1["recall"],
        "accuracy_1fpr": m1["accuracy"],

        "f1_3fpr": m3["f1"],
        "precision_3fpr": m3["precision"],
        "recall_3fpr": m3["recall"],
        "accuracy_3fpr": m3["accuracy"]
    }

    results_path = f"{DRIVE_PATH}/results.json"

    all_results = (
        json.load(open(results_path))
        if os.path.exists(results_path)
        else []
    )

    all_results = [
        r for r in all_results
        if r["name"] != name
    ]

    all_results.append(result)

    json.dump(
        all_results,
        open(results_path, "w"),
        indent=2
    )

    print("Result saved to Drive.")

    return result


def print_results_table():

    results_path = f"{DRIVE_PATH}/results.json"

    if not os.path.exists(results_path):
        print("No results saved yet.")
        return

    all_results = json.load(open(results_path))

    print(
        f"\n{'Model':<25}"
        f"{'AUC':>6}"
        f"{'F1':>6}"
        f"{'Rec':>6}"
        f"{'Acc':>6}"
        f" | "
        f"{'F1@1%':>6}"
        f"{'Pre@1%':>7}"
        f"{'Rec@1%':>7}"
        f" | "
        f"{'F1@3%':>6}"
        f"{'Rec@3%':>7}"
    )

    print("-" * 105)

    for r in all_results:

        print(
            f"{r['name']:<25}"
            f"{r['auc']:>6.4f}"
            f"{r['f1']:>6.4f}"
            f"{r['recall']:>6.4f}"
            f"{r.get('accuracy',0):>6.4f}"
            f" | "
            f"{r.get('f1_1fpr',0):>6.4f}"
            f"{r.get('precision_1fpr',0):>7.4f}"
            f"{r.get('recall_1fpr',0):>7.4f}"
            f" | "
            f"{r.get('f1_3fpr',0):>6.4f}"
            f"{r.get('recall_3fpr',0):>7.4f}"
        )

print("✓ Evaluation helper functions loaded.")

✓ Evaluation helper functions loaded.


In [7]:
# ============================================================
# CELL 2B — CNN Architectures
# ============================================================

# ------------------------------------------------------------
# 1D CNN (RFM)
# ------------------------------------------------------------

def build_cnn_1d(input_dim):

    model = Sequential([

        Reshape((input_dim, 1), input_shape=(input_dim,)),

        Conv1D(
            64,
            kernel_size=3,
            activation='relu',
            padding='same'
        ),

        MaxPooling1D(pool_size=2),

        Dropout(0.3),

        Conv1D(
            32,
            kernel_size=3,
            activation='relu',
            padding='same'
        ),

        MaxPooling1D(pool_size=2),

        Dropout(0.3),

        Flatten(),

        Dense(64, activation='relu'),

        Dropout(0.2),

        Dense(1, activation='sigmoid')

    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['AUC']
    )

    return model


# ------------------------------------------------------------
# Feature Matrix CNN (Paper Figure 4)
# ------------------------------------------------------------

def build_cnn_matrix(n_features, seq_len):
    """
    Paper Figure 4

    Input shape:
        (batch,
         recent_transactions,
         feature_types)

    Permute ->
        (batch,
         feature_types,
         recent_transactions)
    """

    inp = Input(shape=(seq_len, n_features))

    x = Permute((2, 1))(inp)

    x = Conv1D(
        64,
        kernel_size=3,
        activation='relu',
        padding='same'
    )(x)

    x = MaxPooling1D(pool_size=2)(x)

    x = Dropout(0.3)(x)

    x = Conv1D(
        32,
        kernel_size=3,
        activation='relu',
        padding='same'
    )(x)

    x = MaxPooling1D(pool_size=2)(x)

    x = Dropout(0.3)(x)

    x = Flatten()(x)

    x = Dense(
        64,
        activation='relu'
    )(x)

    x = Dropout(0.2)(x)

    out = Dense(
        1,
        activation='sigmoid'
    )(x)

    model = Model(inp, out)

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['AUC']
    )

    return model


# ------------------------------------------------------------
# Early Stopping
# ------------------------------------------------------------

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

print("✓ CNN architectures loaded.")

✓ CNN architectures loaded.


In [8]:
# ============================================================
# CELL 3 — Load RFM Features
# ============================================================

print("\nLoading RFM...")

# ------------------------------------------------------------
# Load train/test
# ------------------------------------------------------------

train_rfm_df = pd.read_parquet(
    f"{DRIVE_PATH}/train_rfm.parquet"
)

test_rfm_df = pd.read_parquet(
    f"{DRIVE_PATH}/test_rfm.parquet"
)

# ------------------------------------------------------------
# Feature columns
# ------------------------------------------------------------

rfm_cols = [
    c for c in train_rfm_df.columns
    if c.startswith("rfm_")
]

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

y_train_rfm = (
    train_rfm_df["is_fraud"]
    .astype(int)
    .to_numpy()
)

y_test_rfm = (
    test_rfm_df["is_fraud"]
    .astype(int)
    .to_numpy()
)

# ------------------------------------------------------------
# Feature matrices
# ------------------------------------------------------------

X_train_rfm_raw = train_rfm_df[
    rfm_cols
].to_numpy(dtype=np.float32)

X_test_rfm_raw = test_rfm_df[
    rfm_cols
].to_numpy(dtype=np.float32)

del train_rfm_df
del test_rfm_df
gc.collect()

# ------------------------------------------------------------
# Standardize
# ------------------------------------------------------------

scaler_rfm = StandardScaler()

X_train_rfm = scaler_rfm.fit_transform(
    X_train_rfm_raw
)

X_test_rfm = scaler_rfm.transform(
    X_test_rfm_raw
)

del X_train_rfm_raw
del X_test_rfm_raw
gc.collect()

print(
    f"RFM ready: {X_train_rfm.shape} | "
    f"RAM: {psutil.virtual_memory().percent}%"
)

# ------------------------------------------------------------
# Class weights
# ------------------------------------------------------------

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train_rfm
)

class_weight_dict = {
    0: float(weights[0]),
    1: float(weights[1])
}

print("\nClass weights:")
print(class_weight_dict)


Loading RFM...
RFM ready: (1035957, 65) | RAM: 27.0%

Class weights:
{0: 0.5030714857347091, 1: 81.89383399209486}


In [9]:
# ============================================================
# CELL 4 — Train CNN on RFM
# ============================================================

print("\nTraining CNN on RFM features...")

cnn_rfm = build_cnn_1d(X_train_rfm.shape[1])

history = cnn_rfm.fit(

    X_train_rfm,
    y_train_rfm,

    validation_split=0.20,

    epochs=20,

    batch_size=512,

    class_weight=class_weight_dict,

    callbacks=[early_stop],

    verbose=1
)

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

y_prob = cnn_rfm.predict(
    X_test_rfm,
    batch_size=4096,
    verbose=1
).ravel()

y_pred = (y_prob >= 0.5).astype(int)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

evaluate_model(
    "CNN — RFM",
    y_test_rfm,
    y_pred,
    y_prob
)

# ------------------------------------------------------------
# Free RAM
# ------------------------------------------------------------

del X_train_rfm
del X_test_rfm
del y_train_rfm
del y_test_rfm
del history

gc.collect()

print(
    f"\nRAM after CNN-RFM: "
    f"{psutil.virtual_memory().percent}%"
)


Training CNN on RFM features...
Epoch 1/20
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 27s 10ms/step - AUC: 0.9118 - loss: 0.3786 - val_AUC: 0.9609 - val_loss: 0.1505
Epoch 2/20
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - AUC: 0.9451 - loss: 0.2985 - val_AUC: 0.9681 - val_loss: 0.1210
Epoch 3/20
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - AUC: 0.9545 - loss: 0.2701 - val_AUC: 0.9719 - val_loss: 0.0902
Epoch 4/20
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - AUC: 0.9589 - loss: 0.2569 - val_AUC: 0.9739 - val_loss: 0.1070
Epoch 5/20
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - AUC: 0.9610 - loss: 0.2498 - val_AUC: 0.9750 - val_loss: 0.1050
Epoch 6/20
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - AUC: 0.9634 - loss: 0.2407 - val_AUC: 0.9751 - val_loss: 0.1099
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 3.
85/85 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step

=== CNN — RFM ===
Overall  — AUC:0.9390 F1:0.2315 Pre:0.1398 Rec:0.6732 Acc:0.9731
@ 1% FPR — F1:0.3551 Pre:0.2579 Re

In [2]:
# ============================================================
# SECTION 13 — CNN (TRUE FEATURE MATRIX) — FIXED, SELF-CONTAINED
# ============================================================
import gc, os, json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, roc_curve)
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Dense, Dropout, Input, Conv1D,
                                     MaxPooling1D, Flatten, Reshape,
                                     Permute)
from tensorflow.keras.callbacks import EarlyStopping

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
RANDOM_SEED = 42
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
SEQ_LEN = 15
print(f"GPU: {tf.config.list_physical_devices('GPU')}")
if not tf.config.list_physical_devices('GPU'):
    print("WARNING: running on CPU. The HOBA matrix generator does a Python "
          "loop per transaction — this will be slow. Switch runtime to GPU "
          "(Runtime > Change runtime type) if this cell is taking forever.")

# ── Evaluation helpers ────────────────────────────────────────
def get_metrics_at_fpr(y_true, y_prob, fpr_threshold):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    valid_idx = np.where(fpr <= fpr_threshold)[0]
    if len(valid_idx) == 0:
        return {'f1': 0, 'precision': 0, 'recall': 0, 'accuracy': 0}
    best_idx = valid_idx[np.argmax(tpr[valid_idx])]
    thresh = thresholds[best_idx]
    y_pred_thresh = (y_prob >= thresh).astype(int)
    TP = ((y_pred_thresh==1)&(y_true==1)).sum()
    TN = ((y_pred_thresh==0)&(y_true==0)).sum()
    FP = ((y_pred_thresh==1)&(y_true==0)).sum()
    FN = ((y_pred_thresh==0)&(y_true==1)).sum()
    precision = TP/(TP+FP) if (TP+FP)>0 else 0
    recall    = TP/(TP+FN) if (TP+FN)>0 else 0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0
    return {'f1': f1, 'precision': precision, 'recall': recall,
            'accuracy': (TP+TN)/len(y_true)}

def evaluate_model(name, y_true, y_pred, y_prob):
    auc  = roc_auc_score(y_true, y_prob)
    f1   = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    acc  = (y_pred==y_true).sum()/len(y_true)
    m1   = get_metrics_at_fpr(y_true, y_prob, 0.01)
    m3   = get_metrics_at_fpr(y_true, y_prob, 0.03)
    print(f"\n=== {name} ===")
    print(f"Overall  — AUC:{auc:.4f} F1:{f1:.4f} Pre:{prec:.4f} Rec:{rec:.4f} Acc:{acc:.4f}")
    print(f"@ 1% FPR — F1:{m1['f1']:.4f} Pre:{m1['precision']:.4f} Rec:{m1['recall']:.4f} Acc:{m1['accuracy']:.4f}")
    print(f"@ 3% FPR — F1:{m3['f1']:.4f} Pre:{m3['precision']:.4f} Rec:{m3['recall']:.4f} Acc:{m3['accuracy']:.4f}")
    result = {
        'name': name, 'auc': auc, 'f1': f1, 'precision': prec,
        'recall': rec, 'accuracy': acc,
        'f1_1fpr': m1['f1'], 'precision_1fpr': m1['precision'],
        'recall_1fpr': m1['recall'], 'accuracy_1fpr': m1['accuracy'],
        'f1_3fpr': m3['f1'], 'precision_3fpr': m3['precision'],
        'recall_3fpr': m3['recall'], 'accuracy_3fpr': m3['accuracy']
    }
    results_path = f'{DRIVE_PATH}/results.json'
    all_results = json.load(open(results_path)) if os.path.exists(results_path) else []
    all_results = [r for r in all_results if r['name'] != name]
    all_results.append(result)
    json.dump(all_results, open(results_path, 'w'), indent=2)
    print("Result saved to Drive.")
    return result

def print_results_table():
    results_path = f'{DRIVE_PATH}/results.json'
    if not os.path.exists(results_path):
        print("No results saved yet.")
        return
    all_results = json.load(open(results_path))
    print(f"\n{'Model':<25} {'AUC':>6} {'F1':>6} {'Rec':>6} {'Acc':>6} | {'F1@1%':>6} {'Pre@1%':>7} {'Rec@1%':>7} | {'F1@3%':>6} {'Rec@3%':>7}")
    print("-"*105)
    for r in all_results:
        print(f"{r['name']:<25} {r['auc']:>6.4f} {r['f1']:>6.4f} {r['recall']:>6.4f} {r.get('accuracy',0):>6.4f} | "
              f"{r.get('f1_1fpr',0):>6.4f} {r.get('precision_1fpr',0):>7.4f} {r.get('recall_1fpr',0):>7.4f} | "
              f"{r.get('f1_3fpr',0):>6.4f} {r.get('recall_3fpr',0):>7.4f}")

# ── Load RFM ─────────────────────────────────────────────────
print("\nLoading RFM...")
train_rfm_df = pd.read_parquet(f'{DRIVE_PATH}/train_rfm.parquet')
test_rfm_df  = pd.read_parquet(f'{DRIVE_PATH}/test_rfm.parquet')
rfm_cols     = [c for c in train_rfm_df.columns if c.startswith('rfm_')]
y_train_rfm  = train_rfm_df['is_fraud'].astype(int).to_numpy()
y_test_rfm   = test_rfm_df['is_fraud'].astype(int).to_numpy()
X_train_rfm_raw = train_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
X_test_rfm_raw  = test_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
del train_rfm_df, test_rfm_df; gc.collect()

scaler_rfm   = StandardScaler()
X_train_rfm  = scaler_rfm.fit_transform(X_train_rfm_raw)
X_test_rfm   = scaler_rfm.transform(X_test_rfm_raw)
del X_train_rfm_raw, X_test_rfm_raw; gc.collect()
print(f"RFM ready: {X_train_rfm.shape} | RAM: {psutil.virtual_memory().percent}%")

weights = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train_rfm)
class_weight_dict = {0: float(weights[0]), 1: float(weights[1])}
early_stop = EarlyStopping(monitor='val_loss', patience=3,
                            restore_best_weights=True, verbose=1)

# ── CNN architectures ──────────────────────────────────────────
def build_cnn_1d(input_dim):
    model = Sequential([
        Reshape((input_dim, 1), input_shape=(input_dim,)),
        Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        Conv1D(32, kernel_size=3, activation='relu', padding='same'),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

def build_cnn_matrix(n_features, seq_len):
    """Paper's Fig. 4 architecture: rows=feature types, cols=recent transactions."""
    inp = Input(shape=(seq_len, n_features))
    x = Permute((2, 1))(inp)  # -> (batch, n_features, seq_len)
    x = Conv1D(64, kernel_size=3, activation='relu', padding='same')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)
    x = Conv1D(32, kernel_size=3, activation='relu', padding='same')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x)
    out = Dense(1, activation='sigmoid')(x)
    model = Model(inp, out)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

# ── CNN on RFM (1D) ───────────────────────────────────────────
print("\nTraining CNN on RFM features...")
cnn_rfm = build_cnn_1d(input_dim=X_train_rfm.shape[1])
cnn_rfm.fit(X_train_rfm, y_train_rfm, epochs=20, batch_size=512,
            validation_split=0.1, class_weight=class_weight_dict,
            callbacks=[early_stop], verbose=1)
y_prob = cnn_rfm.predict(X_test_rfm, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)
evaluate_model("CNN — RFM", y_test_rfm, y_pred, y_prob)
print(f"RAM after CNN-RFM: {psutil.virtual_memory().percent}%")
del cnn_rfm, y_pred, y_prob, X_train_rfm, X_test_rfm; gc.collect()

# ── Load full HOBA arrays for true matrix construction ────────
print("\nAttempting to load full HOBA arrays...")


try:
    pf = pq.ParquetFile(f'{DRIVE_PATH}/train_hoba_scaled.parquet')
    all_cols = pf.schema_arrow.names
    hoba_cols = [c for c in all_cols if c.startswith('hoba_')]
    n_cols = len(hoba_cols)
    n_rows_train = pf.metadata.num_rows

    # Load card IDs from the original HOBA parquet
    pf_card = pq.ParquetFile(f"{DRIVE_PATH}/train_hoba.parquet")

    card_ids_train = np.empty(n_rows_train, dtype=np.int64)

    offset = 0
    for batch in pf_card.iter_batches(
            batch_size=100000,
            columns=["card_id"]):

        n = batch.num_rows
        card_ids_train[offset:offset+n] = batch.column("card_id").to_numpy(
            zero_copy_only=False
        )
        offset += n

    del pf_card
    gc.collect()

    print(f"  Allocating train array: {n_rows_train} x {n_cols} = {n_rows_train*n_cols*4/1e9:.2f}GB")
    X_train_hoba = np.empty((n_rows_train, n_cols), dtype=np.float32)
    y_train_hoba = np.empty(n_rows_train, dtype=np.int8)


    row_offset = 0
    for batch in pf.iter_batches(batch_size=100_000,
                                   columns=hoba_cols + ['is_fraud']):
        n = batch.num_rows
        for i, c in enumerate(hoba_cols):
            X_train_hoba[row_offset:row_offset+n, i] = \
                batch.column(c).to_numpy(zero_copy_only=False)
        y_train_hoba[row_offset:row_offset+n] = \
            batch.column('is_fraud').to_numpy(zero_copy_only=False)

        row_offset += n
        del batch; gc.collect()

    print(f"  Train loaded. RAM: {psutil.virtual_memory().percent}%")

    pf_test = pq.ParquetFile(f'{DRIVE_PATH}/test_hoba_scaled.parquet')

    n_rows_test = pf_test.metadata.num_rows

    pf_card_test = pq.ParquetFile(f"{DRIVE_PATH}/test_hoba.parquet")

    card_ids_test = np.empty(n_rows_test, dtype=np.int64)

    offset = 0
    for batch in pf_card_test.iter_batches(
            batch_size=100000,
            columns=["card_id"]):

        n = batch.num_rows
        card_ids_test[offset:offset+n] = batch.column("card_id").to_numpy(
            zero_copy_only=False
        )
        offset += n

    del pf_card_test
    gc.collect()
    n_rows_test = pf_test.metadata.num_rows
    X_test_hoba = np.empty((n_rows_test, n_cols), dtype=np.float32)
    y_test_hoba = np.empty(n_rows_test, dtype=np.int8)


    row_offset = 0
    for batch in pf_test.iter_batches(batch_size=100_000,
                                       columns=hoba_cols + ['is_fraud']):
        n = batch.num_rows
        for i, c in enumerate(hoba_cols):
            X_test_hoba[row_offset:row_offset+n, i] = \
                batch.column(c).to_numpy(zero_copy_only=False)
        y_test_hoba[row_offset:row_offset+n] = \
            batch.column('is_fraud').to_numpy(zero_copy_only=False)

        row_offset += n
        del batch; gc.collect()

    print(f"  Test loaded. RAM: {psutil.virtual_memory().percent}%")
    USE_MATRIX_CNN = True

except MemoryError:
    print("  MemoryError — falling back to streaming 1D-CNN")
    USE_MATRIX_CNN = False

# ── Build card → sorted row indices lookup (fast, groupby-based) ──
if USE_MATRIX_CNN:
    print("\nBuilding card→row index lookup (train)...")
    card_to_rows = (
        pd.Series(np.arange(len(card_ids_train)))
          .groupby(card_ids_train)
          .apply(lambda s: s.to_numpy())
          .to_dict()
    )
    print(f"  Lookup built for {len(card_to_rows)} cards. RAM: {psutil.virtual_memory().percent}%")

    # ── Batch builders ─────────────────────────────────────────
    def build_matrix_batch(X_src, row_indices, y_src, card_ids_arr, card_to_rows,
                            seq_len, cutoff_by_position):
        """cutoff_by_position=True: only use same-array history strictly before
        the current row (leakage prevention within train/val). Use False only
        when the history array is guaranteed chronologically earlier than the
        rows being scored (train history -> test predictions)."""
        batch_X = np.zeros((len(row_indices), seq_len, X_src.shape[1]), dtype=np.float32)
        batch_y = y_src[row_indices]
        for bi, idx in enumerate(row_indices):
            card = card_ids_arr[idx]
            card_rows = card_to_rows.get(card)
            if card_rows is None or len(card_rows) == 0:
                continue
            prior = card_rows[card_rows < idx] if cutoff_by_position else card_rows
            if len(prior) == 0:
                continue
            seq = prior[-seq_len:]
            offset = seq_len - len(seq)
            batch_X[bi, offset:, :] = X_src[seq, :]
        return batch_X, batch_y

    def matrix_batch_generator(X_src, y_src, card_ids_arr, card_to_rows,
                                seq_len, batch_size, indices, shuffle=False):
        idx_order = indices.copy()
        if shuffle:
            np.random.shuffle(idx_order)
        for start in range(0, len(idx_order), batch_size):
            batch_idx = idx_order[start:start+batch_size]
            yield build_matrix_batch(X_src, batch_idx, y_src, card_ids_arr,
                                      card_to_rows, seq_len, cutoff_by_position=True)

    def test_batch_generator(card_ids_test_arr, y_test_arr, card_to_rows_train,
                              X_train_src, seq_len, batch_size):
        n = len(y_test_arr)
        indices = np.arange(n)
        for start in range(0, n, batch_size):
            batch_idx = indices[start:start+batch_size]
            yield build_matrix_batch(X_train_src, batch_idx, y_test_arr,
                                      card_ids_test_arr, card_to_rows_train,
                                      seq_len, cutoff_by_position=False)

    # ── Train/val split (chronological — last 10% held out) ────
    n_train_hoba = len(y_train_hoba)
    val_size   = int(0.1 * n_train_hoba)
    train_size = n_train_hoba - val_size
    train_idx = np.arange(train_size)
    val_idx   = np.arange(train_size, n_train_hoba)

    BATCH_SIZE = 256
    cnn_hoba = build_cnn_matrix(n_features=n_cols, seq_len=SEQ_LEN)
    cnn_hoba.summary()

    best_val_loss = np.inf
    patience_count = 0
    PATIENCE = 3
    best_weights = None

    print("\nTraining CNN on HOBA (true feature matrix)...")
    for epoch in range(20):
        train_losses, train_aucs = [], []
        for X_batch, y_batch in matrix_batch_generator(
                X_train_hoba, y_train_hoba, card_ids_train,
                card_to_rows, SEQ_LEN, BATCH_SIZE, train_idx, shuffle=True):
            sample_w = np.where(y_batch==1, class_weight_dict[1], class_weight_dict[0])
            metrics = cnn_hoba.train_on_batch(X_batch, y_batch, sample_weight=sample_w)
            train_losses.append(metrics[0]); train_aucs.append(metrics[1])

        val_losses, val_aucs = [], []
        for X_batch, y_batch in matrix_batch_generator(
                X_train_hoba, y_train_hoba, card_ids_train,
                card_to_rows, SEQ_LEN, BATCH_SIZE, val_idx, shuffle=False):
            metrics = cnn_hoba.test_on_batch(X_batch, y_batch)
            val_losses.append(metrics[0]); val_aucs.append(metrics[1])

        avg_train_loss, avg_val_loss = np.mean(train_losses), np.mean(val_losses)
        avg_train_auc, avg_val_auc   = np.mean(train_aucs), np.mean(val_aucs)
        print(f"Epoch {epoch+1}/20 — loss:{avg_train_loss:.4f} AUC:{avg_train_auc:.4f} "
              f"val_loss:{avg_val_loss:.4f} val_AUC:{avg_val_auc:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_weights = cnn_hoba.get_weights()
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"Early stopping at epoch {epoch+1}")
                cnn_hoba.set_weights(best_weights)
                break

    print(f"RAM after CNN-HOBA fit: {psutil.virtual_memory().percent}%")

    # ── Predict on test (history always from train array) ──────
    print("\nPredicting on test set...")
    y_prob_list, y_test_list = [], []
    for X_batch, y_batch in test_batch_generator(
            card_ids_test, y_test_hoba, card_to_rows, X_train_hoba, SEQ_LEN, BATCH_SIZE):
        y_prob_list.append(cnn_hoba.predict(X_batch, verbose=0).flatten())
        y_test_list.append(y_batch)

    y_prob = np.concatenate(y_prob_list)
    y_test = np.concatenate(y_test_list)
    y_pred = (y_prob >= 0.5).astype(int)
    evaluate_model("CNN — HOBA", y_test, y_pred, y_prob)

else:
    # Fallback: streaming 1D-CNN (unchanged — doesn't need card_id at all)
    print("\nFallback: Training streaming 1D-CNN on HOBA...")
    pf_train = pq.ParquetFile(f'{DRIVE_PATH}/train_hoba_scaled.parquet')
    hoba_cols = [c for c in pf_train.schema_arrow.names if c.startswith('hoba_')]
    n_cols = len(hoba_cols)
    n_rows_train = pf_train.metadata.num_rows
    pf_test_f = pq.ParquetFile(f'{DRIVE_PATH}/test_hoba_scaled.parquet')
    n_rows_test = pf_test_f.metadata.num_rows
    n_val   = int(n_rows_train * 0.1)
    n_train = n_rows_train - n_val

    labels = np.concatenate([
        b.column('is_fraud').to_numpy(zero_copy_only=False)
        for b in pf_train.iter_batches(batch_size=200_000, columns=['is_fraud'])
    ])
    wh = compute_class_weight('balanced', classes=np.array([0,1]), y=labels)
    cw_hoba = {0: float(wh[0]), 1: float(wh[1])}
    del labels; gc.collect()

    train_path = f'{DRIVE_PATH}/train_hoba_scaled.parquet'
    test_path  = f'{DRIVE_PATH}/test_hoba_scaled.parquet'

    def make_generator(path, row_start, row_end, bsz, cols):
        def gen():
            pf = pq.ParquetFile(path)
            rc = 0
            for batch in pf.iter_batches(batch_size=bsz, columns=cols+['is_fraud']):
                n = batch.num_rows
                lo = max(row_start, rc); hi = min(row_end, rc+n); rc += n
                if lo >= hi: continue
                olo, ohi = lo-(rc-n), hi-(rc-n)
                X = np.column_stack([batch.column(c).to_numpy(
                    zero_copy_only=False)[olo:ohi] for c in cols]).astype(np.float32)
                y = batch.column('is_fraud').to_numpy(
                    zero_copy_only=False)[olo:ohi].astype(np.int8)
                yield X, y
        return gen

    sig = (tf.TensorSpec(shape=(None, n_cols), dtype=tf.float32),
           tf.TensorSpec(shape=(None,),        dtype=tf.int8))
    train_ds = tf.data.Dataset.from_generator(
        make_generator(train_path, 0, n_train, 512, hoba_cols),
        output_signature=sig).prefetch(tf.data.AUTOTUNE)
    val_ds = tf.data.Dataset.from_generator(
        make_generator(train_path, n_train, n_rows_train, 512, hoba_cols),
        output_signature=sig).prefetch(tf.data.AUTOTUNE)
    test_ds = tf.data.Dataset.from_generator(
        make_generator(test_path, 0, n_rows_test, 512, hoba_cols),
        output_signature=sig).prefetch(tf.data.AUTOTUNE)

    cnn_hoba = build_cnn_1d(input_dim=n_cols)
    cnn_hoba.fit(train_ds, validation_data=val_ds, epochs=20,
                 class_weight=cw_hoba, callbacks=[early_stop], verbose=1)

    y_prob_list, y_test_list = [], []
    for X_batch, y_batch in test_ds:
        y_prob_list.append(cnn_hoba.predict(X_batch, verbose=0).flatten())
        y_test_list.append(y_batch.numpy())
    y_prob = np.concatenate(y_prob_list)
    y_test = np.concatenate(y_test_list)
    y_pred = (y_prob >= 0.5).astype(int)
    evaluate_model("CNN — HOBA", y_test, y_pred, y_prob)

print(f"\nFinal RAM: {psutil.virtual_memory().percent}%")
print("\n")
print_results_table()

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Loading RFM...
RFM ready: (1035957, 65) | RAM: 26.0%

Training CNN on RFM features...
Epoch 1/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - AUC: 0.9159 - loss: 0.3712 - val_AUC: 0.9621 - val_loss: 0.2692
Epoch 2/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - AUC: 0.9470 - loss: 0.2921 - val_AUC: 0.9699 - val_loss: 0.2473
Epoch 3/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - AUC: 0.9549 - loss: 0.2677 - val_AUC: 0.9738 - val_loss: 0.2280
Epoch 4/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - AUC: 0.9595 - loss: 0.2546 - val_AUC: 0.9745 - val_loss: 0.1898
Epoch 5/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - AUC: 0.9613 - loss: 0.2479 - val_AUC: 0.9764 - val_loss: 0.2070
Epoch 6/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - AUC: 0.9645 - loss: 0.2377 - val_AUC: 0.9765 - val_loss: 0.2065
Epoch 7/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - AUC: 0.9645 - loss: 0.2352 - val_AUC: 0.9783 - val_loss: 0.

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 15, 891)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ permute (Permute)               │ (None, 891, 15)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 891, 64)        │         2,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 445, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 445, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 445, 32)        │         6,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 222, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 222, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 7104)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       454,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 463,905 (1.77 MB)

 Trainable params: 463,905 (1.77 MB)

 Non-trainable params: 0 (0.00 B)


Training CNN on HOBA (true feature matrix)...
Epoch 1/20 — loss:0.6191 AUC:0.6965 val_loss:0.5218 val_AUC:0.8131
Epoch 2/20 — loss:0.4980 AUC:0.8316 val_loss:0.4840 val_AUC:0.8421
Epoch 3/20 — loss:0.4750 AUC:0.8482 val_loss:0.4654 val_AUC:0.8532
Epoch 4/20 — loss:0.4577 AUC:0.8573 val_loss:0.4542 val_AUC:0.8601
Epoch 5/20 — loss:0.4504 AUC:0.8629 val_loss:0.4464 val_AUC:0.8657
Epoch 6/20 — loss:0.4428 AUC:0.8681 val_loss:0.4396 val_AUC:0.8698
Epoch 7/20 — loss:0.4365 AUC:0.8715 val_loss:0.4345 val_AUC:0.8726
Epoch 8/20 — loss:0.4322 AUC:0.8740 val_loss:0.4309 val_AUC:0.8750
Epoch 9/20 — loss:0.4294 AUC:0.8759 val_loss:0.4281 val_AUC:0.8769
Epoch 10/20 — loss:0.4267 AUC:0.8778 val_loss:0.4255 val_AUC:0.8786
Epoch 11/20 — loss:0.4243 AUC:0.8793 val_loss:0.4233 val_AUC:0.8799
Epoch 12/20 — loss:0.4220 AUC:0.8805 val_loss:0.4209 val_AUC:0.8811
Epoch 13/20 — loss:0.4197 AUC:0.8816 val_loss:0.4197 val_AUC:0.8818
Epoch 14/20 — loss:0.4191 AUC:0.8821 val_loss:0.4183 val_AUC:0.8824
Epoch 15/2

In [2]:
# ============================================================
# SECTION 13 — CNN — FULLY SELF-CONTAINED
# ============================================================
import gc, os, json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import psutil
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, roc_curve)
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (Dense, Dropout, Input, Conv1D,
                                     MaxPooling1D, Flatten, Reshape)
from tensorflow.keras.callbacks import EarlyStopping

DRIVE_PATH = '/content/drive/MyDrive/HOBA_Fraud_Detection_V2'
RANDOM_SEED = 42
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

# ── Evaluation helpers ────────────────────────────────────────
def get_metrics_at_fpr(y_true, y_prob, fpr_threshold):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    valid_idx = np.where(fpr <= fpr_threshold)[0]
    if len(valid_idx) == 0:
        return {'f1': 0, 'precision': 0, 'recall': 0, 'accuracy': 0}
    best_idx = valid_idx[np.argmax(tpr[valid_idx])]
    thresh = thresholds[best_idx]
    y_pred_thresh = (y_prob >= thresh).astype(int)
    TP = ((y_pred_thresh==1)&(y_true==1)).sum()
    TN = ((y_pred_thresh==0)&(y_true==0)).sum()
    FP = ((y_pred_thresh==1)&(y_true==0)).sum()
    FN = ((y_pred_thresh==0)&(y_true==1)).sum()
    precision = TP/(TP+FP) if (TP+FP)>0 else 0
    recall    = TP/(TP+FN) if (TP+FN)>0 else 0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0
    accuracy  = (TP+TN)/len(y_true)
    return {'f1': f1, 'precision': precision, 'recall': recall, 'accuracy': accuracy}

def evaluate_model(name, y_true, y_pred, y_prob):
    auc  = roc_auc_score(y_true, y_prob)
    f1   = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    acc  = (y_pred==y_true).sum()/len(y_true)
    m1   = get_metrics_at_fpr(y_true, y_prob, 0.01)
    m3   = get_metrics_at_fpr(y_true, y_prob, 0.03)
    print(f"\n=== {name} ===")
    print(f"Overall  — AUC:{auc:.4f} F1:{f1:.4f} Pre:{prec:.4f} Rec:{rec:.4f} Acc:{acc:.4f}")
    print(f"@ 1% FPR — F1:{m1['f1']:.4f} Pre:{m1['precision']:.4f} Rec:{m1['recall']:.4f} Acc:{m1['accuracy']:.4f}")
    print(f"@ 3% FPR — F1:{m3['f1']:.4f} Pre:{m3['precision']:.4f} Rec:{m3['recall']:.4f} Acc:{m3['accuracy']:.4f}")
    result = {
        'name': name, 'auc': auc, 'f1': f1, 'precision': prec,
        'recall': rec, 'accuracy': acc,
        'f1_1fpr': m1['f1'], 'precision_1fpr': m1['precision'],
        'recall_1fpr': m1['recall'], 'accuracy_1fpr': m1['accuracy'],
        'f1_3fpr': m3['f1'], 'precision_3fpr': m3['precision'],
        'recall_3fpr': m3['recall'], 'accuracy_3fpr': m3['accuracy']
    }
    results_path = f'{DRIVE_PATH}/results.json'
    all_results = json.load(open(results_path)) if os.path.exists(results_path) else []
    all_results = [r for r in all_results if r['name'] != name]
    all_results.append(result)
    json.dump(all_results, open(results_path, 'w'), indent=2)
    print("Result saved to Drive.")
    return result

def print_results_table():
    results_path = f'{DRIVE_PATH}/results.json'
    if not os.path.exists(results_path):
        print("No results saved yet.")
        return
    all_results = json.load(open(results_path))
    print(f"\n{'Model':<25} {'AUC':>6} {'F1':>6} {'Rec':>6} {'Acc':>6} | {'F1@1%':>6} {'Pre@1%':>7} {'Rec@1%':>7} | {'F1@3%':>6} {'Rec@3%':>7}")
    print("-"*105)
    for r in all_results:
        print(f"{r['name']:<25} {r['auc']:>6.4f} {r['f1']:>6.4f} {r['recall']:>6.4f} {r.get('accuracy',0):>6.4f} | "
              f"{r.get('f1_1fpr',0):>6.4f} {r.get('precision_1fpr',0):>7.4f} {r.get('recall_1fpr',0):>7.4f} | "
              f"{r.get('f1_3fpr',0):>6.4f} {r.get('recall_3fpr',0):>7.4f}")

# ── Load RFM ─────────────────────────────────────────────────
print("\nLoading RFM...")
train_rfm_df = pd.read_parquet(f'{DRIVE_PATH}/train_rfm.parquet')
test_rfm_df  = pd.read_parquet(f'{DRIVE_PATH}/test_rfm.parquet')
rfm_cols = [c for c in train_rfm_df.columns if c.startswith('rfm_')]
y_train_rfm = train_rfm_df['is_fraud'].astype(int).to_numpy()
y_test_rfm  = test_rfm_df['is_fraud'].astype(int).to_numpy()
X_train_rfm_raw = train_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
X_test_rfm_raw  = test_rfm_df[rfm_cols].to_numpy(dtype=np.float32)
del train_rfm_df, test_rfm_df; gc.collect()

scaler_rfm = StandardScaler()
X_train_rfm = scaler_rfm.fit_transform(X_train_rfm_raw)
X_test_rfm  = scaler_rfm.transform(X_test_rfm_raw)
del X_train_rfm_raw, X_test_rfm_raw; gc.collect()
print(f"RFM ready: {X_train_rfm.shape} | RAM: {psutil.virtual_memory().percent}%")

weights = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train_rfm)
class_weight_dict = {0: float(weights[0]), 1: float(weights[1])}
early_stop = EarlyStopping(monitor='val_loss', patience=3,
                            restore_best_weights=True, verbose=1)

# ── Build CNN ─────────────────────────────────────────────────
def build_cnn(input_dim):
    model = Sequential([
        Reshape((input_dim, 1), input_shape=(input_dim,)),
        Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        Conv1D(32, kernel_size=3, activation='relu', padding='same'),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

# ── CNN on RFM ────────────────────────────────────────────────
print("\nTraining CNN on RFM features...")
cnn_rfm = build_cnn(input_dim=X_train_rfm.shape[1])
cnn_rfm.fit(X_train_rfm, y_train_rfm, epochs=20, batch_size=512,
            validation_split=0.1, class_weight=class_weight_dict,
            callbacks=[early_stop], verbose=1)
y_prob = cnn_rfm.predict(X_test_rfm, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)
evaluate_model("CNN — RFM", y_test_rfm, y_pred, y_prob)
print(f"RAM after CNN-RFM: {psutil.virtual_memory().percent}%")
del cnn_rfm, y_pred, y_prob, X_train_rfm, X_test_rfm; gc.collect()

# ── HOBA streaming setup ──────────────────────────────────────
print("\nSetting up HOBA streaming...")
pf_train = pq.ParquetFile(f'{DRIVE_PATH}/train_hoba_scaled.parquet')
hoba_cols = [c for c in pf_train.schema_arrow.names if c.startswith('hoba_')]
n_cols = len(hoba_cols)
n_rows_train = pf_train.metadata.num_rows
pf_test_f = pq.ParquetFile(f'{DRIVE_PATH}/test_hoba_scaled.parquet')
n_rows_test = pf_test_f.metadata.num_rows

VAL_FRACTION = 0.1
n_val   = int(n_rows_train * VAL_FRACTION)
n_train = n_rows_train - n_val

labels = np.concatenate([
    batch.column('is_fraud').to_numpy(zero_copy_only=False)
    for batch in pf_train.iter_batches(batch_size=200_000, columns=['is_fraud'])
])
weights_h = compute_class_weight('balanced', classes=np.array([0,1]), y=labels)
cw_hoba = {0: float(weights_h[0]), 1: float(weights_h[1])}
del labels; gc.collect()
print(f"RAM: {psutil.virtual_memory().percent}%")

BATCH_SIZE = 512
train_path = f'{DRIVE_PATH}/train_hoba_scaled.parquet'
test_path  = f'{DRIVE_PATH}/test_hoba_scaled.parquet'

def make_generator(parquet_path, row_start, row_end, batch_size, cols):
    def gen():
        pf = pq.ParquetFile(parquet_path)
        row_cursor = 0
        for batch in pf.iter_batches(batch_size=batch_size, columns=cols+['is_fraud']):
            n = batch.num_rows
            lo = max(row_start, row_cursor)
            hi = min(row_end, row_cursor+n)
            row_cursor += n
            if lo >= hi:
                continue
            olo, ohi = lo-(row_cursor-n), hi-(row_cursor-n)
            X = np.column_stack([batch.column(c).to_numpy(zero_copy_only=False)[olo:ohi]
                                  for c in cols]).astype(np.float32)
            y = batch.column('is_fraud').to_numpy(zero_copy_only=False)[olo:ohi].astype(np.int8)
            yield X, y
    return gen

sig = (tf.TensorSpec(shape=(None, n_cols), dtype=tf.float32),
       tf.TensorSpec(shape=(None,),        dtype=tf.int8))

train_ds = tf.data.Dataset.from_generator(
    make_generator(train_path, 0, n_train, BATCH_SIZE, hoba_cols),
    output_signature=sig).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_generator(
    make_generator(train_path, n_train, n_rows_train, BATCH_SIZE, hoba_cols),
    output_signature=sig).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_generator(
    make_generator(test_path, 0, n_rows_test, BATCH_SIZE, hoba_cols),
    output_signature=sig).prefetch(tf.data.AUTOTUNE)

# ── CNN on HOBA ───────────────────────────────────────────────
print("\nTraining CNN on HOBA features...")
cnn_hoba = build_cnn(input_dim=n_cols)
cnn_hoba.fit(train_ds, validation_data=val_ds, epochs=20,
             class_weight=cw_hoba, callbacks=[early_stop], verbose=1)
print(f"RAM after CNN-HOBA fit: {psutil.virtual_memory().percent}%")

print("\nPredicting on test set...")
y_prob_list, y_test_list = [], []
for X_batch, y_batch in test_ds:
    y_prob_list.append(cnn_hoba.predict(X_batch, verbose=0).flatten())
    y_test_list.append(y_batch.numpy())
y_prob = np.concatenate(y_prob_list)
y_test = np.concatenate(y_test_list)
y_pred = (y_prob >= 0.5).astype(int)
evaluate_model("CNN — HOBA", y_test, y_pred, y_prob)
print(f"RAM after eval: {psutil.virtual_memory().percent}%")

print("\n")
print_results_table()

GPU: []

Loading RFM...
RFM ready: (1035957, 65) | RAM: 23.4%

Training CNN on RFM features...
Epoch 1/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 160s 85ms/step - AUC: 0.9159 - loss: 0.3705 - val_AUC: 0.9627 - val_loss: 0.2635
Epoch 2/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 148s 81ms/step - AUC: 0.9467 - loss: 0.2926 - val_AUC: 0.9703 - val_loss: 0.2021
Epoch 3/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 148s 81ms/step - AUC: 0.9549 - loss: 0.2691 - val_AUC: 0.9737 - val_loss: 0.2132
Epoch 4/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 146s 80ms/step - AUC: 0.9595 - loss: 0.2534 - val_AUC: 0.9757 - val_loss: 0.1942
Epoch 5/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 201s 80ms/step - AUC: 0.9623 - loss: 0.2471 - val_AUC: 0.9756 - val_loss: 0.2335
Epoch 6/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 144s 79ms/step - AUC: 0.9638 - loss: 0.2389 - val_AUC: 0.9753 - val_loss: 0.2024
Epoch 7/20
1822/1822 ━━━━━━━━━━━━━━━━━━━━ 144s 79ms/step - AUC: 0.9648 - loss: 0.2362 - val_AUC: 0.9758 - val_loss: 0.2088
Epoch 7: early stopping
Restoring model weig

KeyboardInterrupt: 